In [ ]:
import os
import h5py
import glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
from scipy.integrate import quad
from astropy import constants as const
from scipy.interpolate import interp1d
from scipy.integrate import quad
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 200
mpl.rcParams['savefig.dpi'] = 300
mpl.rcParams['agg.path.chunksize'] = 10000

# Set universal font properties
plt.rcParams.update({
    "font.family": "STIXGeneral",
    "mathtext.fontset": "stix",
    "font.size": 14,
})


In [ ]:
import numpy as np
import cmasher as cmr
import cmocean
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize, SymLogNorm
from matplotlib.ticker import StrMethodFormatter, NullFormatter
import matplotlib.patheffects as path_effects
import h5py
%matplotlib inline

labels = {
    'HII_Fraction': r"$x_\mathrm{H\,II}$",
    'Tb21':         r"$\delta T^\mathrm{H}_\mathrm{b}$",
    'HeII_Fraction':r"$x_\mathrm{He\,II}$",
    'TbHe3':        r"$\delta T^\mathrm{He}_\mathrm{b}$",
}
cmaps = {
    'HII_Fraction': 'magma',
    'Tb21':         'coolwarm',
    'HeII_Fraction':'viridis',
    'TbHe3':        'plasma',
}

grey = [0.1, 0.1, 0.1]

def log_minorticks(vmin, vmax):
    expA = int(np.floor(np.log10(vmin)))
    expB = int(np.floor(np.log10(vmax)))
    cofA = int(np.ceil(vmin/10**expA))
    cofB = int(np.floor(vmax/10**expB))
    lmt = []
    while cofA*10**expA <= cofB*10**expB:
        if expA < expB:
            lmt = np.hstack( (lmt, np.linspace(cofA, 9, 10-cofA)*10**expA) )
            cofA = 1
            expA += 1
        else:
            lmt = np.hstack( (lmt, np.linspace(cofA, cofB, cofB-cofA+1)*10**expA) )
            expA += 1
    return lmt

def colorbar(image, label, position='right', scale=1., fontsize=15, labelsize=10.5, ticklabels=None, extend='neither', ax=None):
    vmin, vmax = image.get_clim()
    Lvmin, Lvmax = int(np.ceil(np.log10(vmin))), int(np.floor(np.log10(vmax)))
    Lcticks = np.linspace(Lvmin, Lvmax, Lvmax-Lvmin+1)
    minorticks = log_minorticks(vmin, vmax)
    if position == 'right':
        if ax is None:
            cax = image.figure.add_axes([1, 0, .03*scale, 1])
        else:
            pos = ax.get_position()
            cax = image.figure.add_axes([pos.x1 + 0.02*pos.height, pos.y0 + 0.225*pos.height, 0.02*scale, pos.height*0.55])
        cbar = image.figure.colorbar(image, cax=cax, ticks=10**Lcticks, orientation='vertical', extend=extend)
        cbar.ax.yaxis.set_ticks(minorticks, minor=True)
        cax.yaxis.set_minor_formatter(NullFormatter())
    elif position == 'left':
        cax = image.figure.add_axes([-.03*scale, 0, .03*scale, 1])
        cbar = image.figure.colorbar(image, cax=cax, ticks=10**Lcticks, orientation='vertical', extend=extend)
        cbar.ax.yaxis.set_ticks(minorticks, minor=True)
        cax.yaxis.set_minor_formatter(NullFormatter())
        cbar.ax.yaxis.set_ticks_position('left')
        cbar.ax.yaxis.set_label_position('left')
        if ticklabels is not None:
            cbar.ax.set_yticklabels(ticklabels)
    elif position == 'top':
        cax = image.figure.add_axes([0, 1, 1, .04*scale])
        cbar = image.figure.colorbar(image, cax=cax, ticks=10**Lcticks, orientation='horizontal', extend=extend)
        cbar.ax.xaxis.set_ticks(minorticks, minor=True)
        cax.xaxis.set_minor_formatter(NullFormatter())
        cbar.ax.xaxis.set_ticks_position('top')
        cbar.ax.xaxis.set_label_position('top')
    elif position == 'bottom':
        cax = image.figure.add_axes([0, -.04*scale, 1, .04*scale])
        cbar = image.figure.colorbar(image, cax=cax, ticks=10**Lcticks, orientation='horizontal', extend=extend)
        cbar.ax.xaxis.set_ticks(minorticks, minor=True)
        cax.xaxis.set_minor_formatter(NullFormatter())
    else:
        raise ValueError('position not recognized!')
    cax.tick_params(which='both', direction='in', labelsize=labelsize)
    if fontsize is not None:
        cbar.set_label(label, fontsize=fontsize, labelpad=7)
    else:
        cbar.set_label(label)

def plot_rlc(SliceFraction=0.2, RedshiftAxis=True, AgeAxis=True):
    outer_rlc = "/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_640"
    inner_rlc = "/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_below_z_4p75_new/postprocessing/rlc_640"

    fields = ['Density', 'Temperature', 'HII_Fraction', 'HeII_Fraction',
              'HeIII_Fraction', 'LyaLuminosityRec', 'LyaLuminosityCol', 'VelocitySquared']

    with h5py.File(f'{outer_rlc}/All.hdf5', 'r') as f:
        header        = f['Header']
        H0            = 100. * header.attrs['HubbleParam']
        Omega0        = header.attrs['Omega0']
        OpeningAngle  = header.attrs['OpeningAngle']
        HalfAngle     = OpeningAngle / 2.
        NumPixels     = header.attrs['NumPixels']
        SliceIndex    = max(0, min(NumPixels - 1, int(SliceFraction * NumPixels)))
        Redshifts_out = f['Redshifts'][:]
        Distances_out = f['Distances'][:]
        ds_out        = {field: f[field][SliceIndex,:,:] for field in fields}

    ds_out['LyaLuminosityRec'] = ds_out['LyaLuminosityRec'] + ds_out['LyaLuminosityCol']
    ds_out.pop('LyaLuminosityCol')
    ds_out['VelocitySquared']  = np.sqrt(ds_out['VelocitySquared'])

    with h5py.File(f'{inner_rlc}/All.hdf5', 'r') as f:
        Redshifts_in = f['Redshifts'][:]
        Distances_in = f['Distances'][:]
        ds_in        = {field: f[field][SliceIndex,:,:] for field in fields}

    ds_in['LyaLuminosityRec'] = ds_in['LyaLuminosityRec'] + ds_in['LyaLuminosityCol']
    ds_in.pop('LyaLuminosityCol')
    ds_in['VelocitySquared']  = np.sqrt(ds_in['VelocitySquared'])

    fields.remove('LyaLuminosityCol')

    # Restrict outer to z <= 20
    zmax = 20.
    imax = np.argmin(np.abs(Redshifts_out - zmax))
    Redshifts_out = Redshifts_out[imax:]
    Distances_out = Distances_out[imax:]
    for field in ds_out:
        ds_out[field] = ds_out[field][:, imax:]

    # Concatenate outer (high-z) + inner (low-z)
    Redshifts = np.concatenate([Redshifts_out, Redshifts_in])
    Distances = np.concatenate([Distances_out, Distances_in])
    ds = {field: np.concatenate([ds_out[field], ds_in[field]], axis=1) for field in ds_out}

    # Trim Distances to match data shape + 1
    nz = ds['HII_Fraction'].shape[1]
    Distances = Distances[:nz + 1]
    Redshifts = Redshifts[:nz + 1]

    import pandas as pd
    from scipy.interpolate import interp1d, CubicSpline
    import astropy.constants as const

    PLANCK       = 6.62607015e-27
    k_B          = 1.381e-16
    C_LIGHT      = const.c.cgs.value
    LAMBDA_21cm  = 21.106
    Tcmb0        = 2.7255
    OMEGA_b      = 0.0486
    OMEGA_m      = 0.3089
    h            = 0.6774

    _T_HH = np.log10([1, 2, 4, 6, 8, 10, 15, 20, 25, 30, 40, 50, 60, 70, 80, 90, 100, 200, 300, 500, 700, 1000, 2000, 3000, 5000, 7000, 10000])
    _k_HH = np.log10([1.38e-13, 1.43e-13, 2.71e-13, 6.60e-13, 1.47e-12, 2.88e-12, 9.10e-12, 1.78e-11, 2.73e-11, 3.67e-11, 5.38e-11, 6.86e-11, 8.14e-11, 9.25e-11, 1.02e-10, 1.11e-10, 1.19e-10, 1.75e-10, 2.09e-10, 2.56e-10, 2.91e-10, 3.31e-10, 4.27e-10, 4.97e-10, 6.03e-10, 6.87e-10, 7.87e-10])
    _T_eH = np.log10([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000, 20000])
    _k_eH = np.log10([2.39e-10, 3.37e-10, 5.30e-10, 7.46e-10, 1.05e-9, 1.63e-9, 2.26e-9, 3.11e-9, 4.59e-9, 5.92e-9, 7.15e-9, 7.71e-9, 8.17e-9, 8.32e-9, 8.37e-9, 8.29e-9, 8.11e-9])
    _T_pH = _T_eH.copy()
    _k_pH = np.log10([0.40e-9, 0.45e-9, 0.430e-9, 0.369e-9, 0.317e-9, 0.3047e-9, 0.3379e-9, 0.4043e-9, 0.5471e-9, 0.7051e-9, 0.9167e-9, 1.070e-9, 1.301e-9, 1.480e-9, 1.695e-9, 1.975e-9, 2.201e-9])
    _spline_HH = CubicSpline(_T_HH, _k_HH, extrapolate=True)
    _spline_eH = CubicSpline(_T_eH, _k_eH, extrapolate=True)
    _spline_pH = CubicSpline(_T_pH, _k_pH, extrapolate=True)

    def get_kappa_2D(T_K):
        logT = np.log10(np.clip(T_K, 1.0, 20000.0))
        k_HH = 10.0 ** _spline_HH(logT)
        k_eH = 10.0 ** _spline_eH(logT)
        k_pH = 10.0 ** _spline_pH(logT)
        return k_HH, k_eH, k_pH

    T_all    = ds['Temperature']
    xHII_all = ds['HII_Fraction']
    xHeII_all= ds['HeII_Fraction'] * 12.666667
    dens_all = ds['Density']

    zc_all = 0.5 * (Redshifts[:-1] + Redshifts[1:])
    zc_all = zc_all[:T_all.shape[1]]
    z2d    = np.tile(zc_all, (NumPixels, 1))

    xHI_all    = 1.0 - xHII_all
    xHeIII_all = 1.0 - xHeII_all

    HYDROGEN_MASSFRAC = 0.76
    m_p               = 1.6738e-24
    rho_H_comoving = HYDROGEN_MASSFRAC * dens_all * 3.0991811197281837e-22
    scale_z3       = (1.0 + z2d) ** 3.0
    rho_H          = rho_H_comoving * scale_z3
    num_H     = rho_H / m_p
    num_HII   = xHII_all  * num_H
    num_HI    = xHI_all   * num_H
    num_HeII  = xHeII_all * num_H
    num_HeIII = xHeIII_all* num_H * 12.666667
    num_p     = num_HII
    num_e     = num_HII + num_HeII + 2.0 * num_HeIII
    mean_density = np.mean(dens_all) if np.mean(dens_all) > 0 else 1.0
    delta_all    = dens_all / mean_density

    Tcmb     = Tcmb0 * (1.0 + z2d)
    e_cgs    = 4.803204712570263e-10
    me_cgs   = 9.1093837e-28
    f_a      = 0.4162
    T_star    = PLANCK * C_LIGHT / (k_B * LAMBDA_21cm)
    A_10      = 2.85e-15
    A_10_He   = 1.95e-12
    LAMBDA_He = 3.46
    T_star_He = PLANCK * C_LIGHT / (k_B * LAMBDA_He)

    df_global    = pd.read_csv("/orcd/data/mvogelsb/004/chcho/LUMINA_ALL.csv")
    df_global    = df_global.sort_values("redshift")
    interp_Ja_H  = interp1d(df_global["redshift"], df_global["J_alpha"],    kind='linear', bounds_error=False, fill_value="extrapolate")
    interp_Ja_He = interp1d(df_global["redshift"], df_global["J_alpha_He"], kind='linear', bounds_error=False, fill_value="extrapolate")
    J_alpha_all    = interp_Ja_H(z2d)  * delta_all
    J_alpha_He_all = interp_Ja_He(z2d) * delta_all

    const_H    = (16.0 * np.pi**2 * T_star    * e_cgs**2 * f_a) / (27.0 * A_10    * Tcmb * me_cgs * C_LIGHT)
    x_a_H_all  = const_H * J_alpha_all
    const_He   = (16.0 * np.pi**2 * T_star_He * e_cgs**2 * f_a) / (9.0  * A_10_He * Tcmb * me_cgs * C_LIGHT)
    x_a_He_all = const_He * J_alpha_He_all

    k_HH, k_eH, k_pH = get_kappa_2D(T_all)
    x_c_H_all  = T_star * (k_HH * num_HI + k_eH * num_e + k_pH * num_p) / (A_10 * Tcmb)
    a_0           = 5.29177e-9
    bar_sigma_all = (2.291112e-11 / (k_B * T_all)) * (a_0 ** 2)
    SQRT_all      = np.sqrt((8.0 * k_B * T_all) / (np.pi * me_cgs * C_LIGHT**2))
    rate_He_all   = num_e * SQRT_all * C_LIGHT * bar_sigma_all
    x_c_He_all    = (T_star_He * rate_He_all) / (A_10_He * Tcmb)

    inv_Tk   = 1.0 / np.maximum(T_all, 1e-3)
    inv_Tcmb = 1.0 / Tcmb
    Ts_H_all  = (1.0 + x_a_H_all  + x_c_H_all)  / (inv_Tcmb + x_a_H_all  * inv_Tk + x_c_H_all  * inv_Tk)
    Ts_He_all = (1.0 + x_a_He_all + x_c_He_all) / (inv_Tcmb + x_a_He_all * inv_Tk + x_c_He_all * inv_Tk)

    cosmo_prefactor = ((OMEGA_b * h**2) / 0.023) * np.sqrt((0.15 * (1.0 + z2d)) / (OMEGA_m * h**2 * 10.0))
    coupling_21 = 1.0 - Tcmb / Ts_H_all
    coupling_He = 1.0 - Tcmb / Ts_He_all
    Tb21_all  = 27.0 * xHI_all   * delta_all * cosmo_prefactor * coupling_21
    TbHe3_all = 1.67 * xHeII_all * delta_all * cosmo_prefactor * coupling_He

    ds['Tb21']  = Tb21_all
    ds['TbHe3'] = TbHe3_all
    ds['HeII_Fraction'] *= 12.666667

    vmin_21,  vmax_21  = np.nanpercentile(Tb21_all,  [1, 99])
    vmin_He3, vmax_He3 = np.nanpercentile(TbHe3_all, [1, 99])
    norms = {
        'HII_Fraction': LogNorm(vmin=1e-4, vmax=1.0, clip=True),
        'Tb21':         SymLogNorm(linthresh=1.0,  linscale=0.5, vmin=vmin_21,  vmax=vmax_21,  base=10),
        'HeII_Fraction':LogNorm(vmin=1e-4, vmax=0.93, clip=True),
        'TbHe3':        SymLogNorm(linthresh=0.01, linscale=0.5, vmin=vmin_He3, vmax=vmax_He3, base=10),
    }

    #fields = ['HII_Fraction', 'Tb21', 'HeII_Fraction', 'TbHe3']
    fields = ['TbHe3', 'HeII_Fraction', 'Tb21', 'HII_Fraction']

    # rotation offset
    #rot_offset = np.pi / 4.

    n_fields = len(fields)

    ## TRYING THICKER
    MaxAngle, RefAngle = float(n_fields) * HalfAngle, np.pi/2.
        
    Theta_E = np.linspace(RefAngle-MaxAngle, RefAngle+MaxAngle, n_fields + 1)
    Theta_C = (Theta_E[1:] + Theta_E[:-1]) / 2.
    Dmin, Dmax, Davg = Distances[-1], Distances[0], 0.5 * (Distances[-1] + Distances[0])
    Xcorners = [Dmin*np.cos(Theta_E[0]), Dmax*np.cos(Theta_E[0]), Dmax*np.cos(Theta_E[-1]), Dmin*np.cos(Theta_E[-1])]
    Ycorners = [Dmin*np.sin(Theta_E[0]), Dmax*np.sin(Theta_E[0]), Dmax*np.sin(Theta_E[-1]), Dmin*np.sin(Theta_E[-1])]
    if Theta_E[0] < 0. < Theta_E[-1]: Xcorners.append(Dmax)
    if Theta_E[0] < np.pi/2. < Theta_E[-1]: Ycorners.append(Dmax)
    if Theta_E[0] < np.pi < Theta_E[-1]: Xcorners.append(-Dmax)
    if Theta_E[0] < 3.*np.pi/2. < Theta_E[-1]: Ycorners.append(-Dmax)
    if RedshiftAxis: Xcorners.append(Dmax*np.cos(Theta_E[-1]+0.002))
    if AgeAxis: Xcorners.append(Dmax*np.cos(Theta_E[0]-0.002))
    Xlims = [np.min(Xcorners), np.max(Xcorners)]
    Ylims = [np.min(Ycorners), np.max(Ycorners)]
    dXlims = Xlims[1] - Xlims[0]
    dYlims = Ylims[1] - Ylims[0]
    Buffer = 0.0025
    Xlims = [Xlims[0] - Buffer*dXlims, Xlims[1] + Buffer*dXlims]
    Ylims = [Ylims[0] - Buffer*dYlims, Ylims[1] + Buffer*dYlims]

    #fig = plt.figure(figsize=(12,12))
    fig = plt.figure(figsize=(12,16))
    fig = plt.figure(figsize=(7, 8.5))
    ax  = plt.axes([0,0,1,1])

    for i, field in enumerate(fields):
        Thetas = np.linspace(Theta_E[i], Theta_E[i+1], NumPixels+1)
        Theta2d, R2d = np.meshgrid(Thetas, Distances, indexing='ij')
        X = R2d * np.cos(Theta2d)
        Y = R2d * np.sin(Theta2d)
        print(f'{field}: {np.percentile(ds[field], [1, 99])}')
        image = ax.pcolormesh(X, Y, ds[field], cmap=cmaps[field], norm=norms[field], rasterized=True, shading='auto')

    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(Xlims)
    ax.set_ylim(Ylims)
    ax.axis('off')

    for i in range(n_fields + 1):
        ax.plot([Dmin*np.cos(Theta_E[i]), Dmax*np.cos(Theta_E[i])],
                [Dmin*np.sin(Theta_E[i]), Dmax*np.sin(Theta_E[i])], c=grey, lw=1)

    theta = np.linspace(Theta_E[0], Theta_E[-1], 200)
    ax.plot(Dmin*np.cos(theta), Dmin*np.sin(theta), c=grey, lw=1)
    ax.plot(Dmax*np.cos(theta), Dmax*np.sin(theta), c=grey, lw=1)

    Dlab = Dmin - 0.02 * (Dmax - Dmin)
    for i, field in enumerate(fields):
        ax.text(Dlab*np.cos(Theta_C[i]), Dlab*np.sin(Theta_C[i]),
                labels[field], ha='center', va='center', c=grey,
                fontsize=15, rotation=np.degrees(Theta_C[i])-90)

    if RedshiftAxis:
        z_ticks = [3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 15, 18]   # ← whatever you actually want
        for i in z_ticks:
            iz = np.argmin(np.abs(Redshifts - float(i)))
            df = Distances[-1] / Distances[iz]
            cos_1 = np.cos(Theta_E[-1])
            cos_2 = np.cos(Theta_E[-1]+0.002*df)
            cos_t = np.cos(Theta_E[-1]+0.010*df)
            sin_1 = np.sin(Theta_E[-1])
            sin_2 = np.sin(Theta_E[-1]+0.002*df)
            sin_t = np.sin(Theta_E[-1]+0.010*df)
            ax.plot([Distances[iz]*cos_1, Distances[iz]*cos_2],
                    [Distances[iz]*sin_1, Distances[iz]*sin_2], c=grey, lw=1.)
            #if i not in [12, 14, 16, 17, 19, 20]:
            ax.text(Distances[iz]*cos_t, Distances[iz]*sin_t,
                    r'${ '+str(i)+r'}$', ha='center', va='center',
                    c=grey, fontsize=10,
                    rotation=np.degrees(Theta_E[-1]+0.006*df))
        ax.text(Davg*np.cos(Theta_E[-1]+0.018), Davg*np.sin(Theta_E[-1]+0.018),
                'Redshift', ha='center', va='center',
                c=grey, fontsize=10, rotation=np.degrees(Theta_E[-1]+0.014))

    if AgeAxis:
        from astropy.cosmology import FlatLambdaCDM, z_at_value
        from astropy import units as u
        cosmo = FlatLambdaCDM(H0=H0, Om0=Omega0, Tcmb0=2.725)
        t_min = 1e3 * cosmo.age(Redshifts[0]).value
        t_max = 1e3 * cosmo.age(Redshifts[-1]).value
        print(f'Dmin: {Dmin/1e6:g} cMpc/h, Dmax: {Dmax/1e6:g} cMpc/h')
        print(f't_min: {t_min:g} Myr, t_max: {t_max:g} Myr')
        Ages = [200., 300., 400., 500., 600., 700., 800., 900., 1000., 1200., 1500, 2100] #1100., 
        for age in Ages:
            z  = z_at_value(cosmo.age, age * u.Myr)
            iz = np.argmin(np.abs(Redshifts - z))
            df = Distances[-1] / Distances[iz]
            cos_1 = np.cos(Theta_E[0])
            cos_2 = np.cos(Theta_E[0]-0.002*df)
            cos_t = np.cos(Theta_E[0]-0.011*df)
            sin_1 = np.sin(Theta_E[0])
            sin_2 = np.sin(Theta_E[0]-0.002*df)
            sin_t = np.sin(Theta_E[0]-0.011*df)
            ax.plot([Distances[iz]*cos_1, Distances[iz]*cos_2],
                    [Distances[iz]*sin_1, Distances[iz]*sin_2], c=grey, lw=1.)
            #if age not in [1100.]:
            ax.text(Distances[iz]*cos_t, Distances[iz]*sin_t,
                    r'${'+str(int(age))+r'}$', ha='center', va='center',
                    c=grey, fontsize=10,
                    rotation=np.degrees(Theta_E[0]-0.007*df))
        ax.text(Davg*np.cos(Theta_E[0]-0.020), Davg*np.sin(Theta_E[0]-0.020),
                'Time Since Big Bang [Myr]', ha='center', va='center',
                c=grey, fontsize=10, rotation=np.degrees(Theta_E[0]-0.016))

    from mpl_toolkits.axes_grid1.inset_locator import inset_axes
    from matplotlib.cm import ScalarMappable

    cbar_specs_left = [
        ('coolwarm', norms['Tb21'],          r"$\delta T^\mathrm{H}_\mathrm{b}\ [\mathrm{mK}]$"),
        ('magma',    norms['HII_Fraction'],  r"$x_\mathrm{H\,II}$"),
    ]
    
    cbar_specs_right = [
        ('plasma',   norms['TbHe3'],         r"$\delta T^\mathrm{He}_\mathrm{b}\ [\mu\mathrm{K}]$"),
        ('viridis',  norms['HeII_Fraction'], r"$x_\mathrm{He\,II}$"),
    ]

    for idx, (cmap, norm, label) in enumerate(cbar_specs_left):
        cax = inset_axes(
            ax, width="13%", height="1.3%", loc="lower left",
            bbox_to_anchor=(0.02, 0.02 + idx * 0.08, 1, 1),
            bbox_transform=ax.transAxes, borderpad=0
        )
        sm = ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cb = fig.colorbar(sm, cax=cax, orientation='horizontal')
        cb.set_label(label, fontsize=10, fontweight='bold', color='#222222', labelpad=4)
        cb.ax.tick_params(labelsize=8, colors='#333333')
        if label == r"$x_\mathrm{H\,II}$":
            cb.set_ticks([1e-4, 1e-2, 1e0])
            cb.set_ticklabels([r"$10^{-4}$", r"$10^{-2}$", r"$1$"])
        if label == r"$\delta T^\mathrm{H}_\mathrm{b}\ [\mathrm{mK}]$":
            cb.set_ticks([-10, 0, 10])
            cb.set_ticklabels([r"$-10$", r"$0$", r"$10$"])
        from matplotlib.ticker import NullLocator
        cb.ax.xaxis.set_minor_locator(NullLocator())
        

    for idx, (cmap, norm, label) in enumerate(cbar_specs_right):
        cax = inset_axes(
            ax, width="13%", height="1.3%", loc="lower right",
            bbox_to_anchor=(-0.02, 0.02 + idx * 0.08, 1, 1),
            bbox_transform=ax.transAxes, borderpad=0
        )
        sm = ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cb = fig.colorbar(sm, cax=cax, orientation='horizontal')
        cb.set_label(label, fontsize=10, fontweight='bold', color='#222222', labelpad=4)
        cb.ax.tick_params(labelsize=8, colors='#333333')
        if label == r"$x_\mathrm{He\,II}$":
            cb.set_ticks([1e-4, 1e-2, 0.93])
            cb.set_ticklabels([r"$10^{-4}$", r"$10^{-2}$", r"$1$"])
        from matplotlib.ticker import NullLocator
        cb.ax.xaxis.set_minor_locator(NullLocator())
        

    plt.savefig('rlc_fan.png', bbox_inches='tight', dpi=200, pad_inches=0.025)
    plt.show()

plot_rlc(SliceFraction=0.1)

In [ ]:
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.cm import ScalarMappable
from scipy.interpolate import interp1d, CubicSpline
import astropy.constants as const
import cmasher as cmr  # Assuming 'cmr' refers to cmasher in your script

# ============================================================
# GLOBAL CONSTANTS & PHYSICS PARAMETERS
# ============================================================
PLANCK       = 6.62607015e-27  # erg s
k_B          = 1.381e-16        # erg/K
C_LIGHT      = const.c.cgs.value # cm / s
LAMBDA_21cm  = 21.106           # cm
Tcmb0        = 2.7255           # K (CMB temperature at z=0)

# Cosmological Parameters (e.g., THESAN suite)
OMEGA_b      = 0.0486 
OMEGA_m      = 0.3089 
h            = 0.6774

# ============================================================
# PRECOMPUTED COLLISIONAL RATE TABLES & GLOBAL SPLINES
# ============================================================
_T_HH = np.log10([1, 2, 4, 6, 8, 10, 15, 20, 25, 30, 40, 50, 60, 70, 80, 90, 100, 200, 300, 500, 700, 1000, 2000, 3000, 5000, 7000, 10000])
_k_HH = np.log10([1.38e-13, 1.43e-13, 2.71e-13, 6.60e-13, 1.47e-12, 2.88e-12, 9.10e-12, 1.78e-11, 2.73e-11, 3.67e-11, 5.38e-11, 6.86e-11, 8.14e-11, 9.25e-11, 1.02e-10, 1.11e-10, 1.19e-10, 1.75e-10, 2.09e-10, 2.56e-10, 2.91e-10, 3.31e-10, 4.27e-10, 4.97e-10, 6.03e-10, 6.87e-10, 7.87e-10])

_T_eH = np.log10([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000, 20000])
_k_eH = np.log10([2.39e-10, 3.37e-10, 5.30e-10, 7.46e-10, 1.05e-9, 1.63e-9, 2.26e-9, 3.11e-9, 4.59e-9, 5.92e-9, 7.15e-9, 7.71e-9, 8.17e-9, 8.32e-9, 8.37e-9, 8.29e-9, 8.11e-9])

_T_pH = _T_eH.copy()
_k_pH = np.log10([0.40e-9, 0.45e-9, 0.430e-9, 0.369e-9, 0.317e-9, 0.3047e-9, 0.3379e-9, 0.4043e-9, 0.5471e-9, 0.7051e-9, 0.9167e-9, 1.070e-9, 1.301e-9, 1.480e-9, 1.695e-9, 1.975e-9, 2.201e-9])

_spline_HH = CubicSpline(_T_HH, _k_HH, extrapolate=True)
_spline_eH = CubicSpline(_T_eH, _k_eH, extrapolate=True)
_spline_pH = CubicSpline(_T_pH, _k_pH, extrapolate=True)

def get_kappa_2D(T_K):
    """Evaluates Hydrogen rate coefficients safely across an entire 2D temperature array."""
    logT = np.log10(np.clip(T_K, 1.0, 20000.0))
    k_HH = 10.0 ** _spline_HH(logT)
    k_eH = 10.0 ** _spline_eH(logT)
    k_pH = 10.0 ** _spline_pH(logT)
    return k_HH, k_eH, k_pH


# =====================================================
# 1. Load a raw chunk and standardize orientation
# =====================================================
def load_raw_chunk(rlc_dir, SliceFraction, zmin, zmax):
    with h5py.File(f"{rlc_dir}/All.hdf5", "r") as f:
        N = f["Header"].attrs["NumPixels"]
        i_center = int(SliceFraction * N)
        T_block    = f["Temperature"][i_center-1 : i_center+2]
        xHII_block = f["HII_Fraction"][i_center-1 : i_center+2]
        xHeII_block= f["HeII_Fraction"][i_center-1 : i_center+2] * 12.666667
        z = f["Redshifts"][:]
        
    with h5py.File(f"{rlc_dir}/Density.hdf5", "r") as f_dens:
        density_block = f_dens["Density"][i_center-1 : i_center+2]

    T       = np.mean(T_block,    axis=0)
    xHII    = np.mean(xHII_block, axis=0)
    xHeII   = np.mean(xHeII_block, axis=0)
    density = np.mean(density_block, axis=0)

    mask = (z >= zmin) & (z <= zmax)
    idx  = np.where(mask)[0]
    i0, i1 = idx.min(), idx.max()

    z_chunk       = z[i0:i1+1]
    T_chunk       = T[:,    i0:i1]
    xHII_chunk    = xHII[:, i0:i1]
    xHeII_chunk   = xHeII[:, i0:i1]
    density_chunk = density[:, i0:i1]

    zc_chunk = 0.5 * (z_chunk[:-1] + z_chunk[1:])
    zc_chunk = zc_chunk[:T_chunk.shape[1]]

    if zc_chunk[0] > zc_chunk[-1]:
        zc_chunk      = zc_chunk[::-1]
        T_chunk       = T_chunk[:,    ::-1]
        xHII_chunk    = xHII_chunk[:, ::-1]
        xHeII_chunk   = xHeII_chunk[:, ::-1]
        density_chunk = density_chunk[:, ::-1]

    return T_chunk, xHII_chunk, xHeII_chunk, density_chunk, zc_chunk, N


# =====================================================
# 2. Main Processing Pipeline 
# =====================================================
def generate_combined_dataset(outer_dir, inner_dir, SliceFraction,
                               outer_zmin, outer_zmax, inner_zmin, inner_zmax):
    # Unpack chunks
    T_in,  xHII_in,  xHeII_in,  dens_in,  zc_in,  N = load_raw_chunk(inner_dir, SliceFraction, inner_zmin, inner_zmax)
    T_out, xHII_out, xHeII_out, dens_out, zc_out, _ = load_raw_chunk(outer_dir, SliceFraction, outer_zmin, outer_zmax)

    z_gap          = zc_out[0] - zc_in[-1]
    zc_out_correct = zc_out - z_gap

    T_all    = np.concatenate([T_in,    T_out],    axis=1)
    xHII_all = np.concatenate([xHII_in, xHII_out], axis=1)
    xHeII_all= np.concatenate([xHeII_in, xHeII_out], axis=1)
    dens_all = np.concatenate([dens_in, dens_out], axis=1)
    zc_all   = np.concatenate([zc_in,   zc_out_correct])

    z2d       = np.tile(zc_all, (N, 1))
    xHI_all   = 1.0 - xHII_all
    xHeIII_all= 1.0 - xHeII_all

    # =====================================================
    # PHYSICAL NUMBER DENSITIES (CGS Units)
    # =====================================================
    HYDROGEN_MASSFRAC = 0.76
    y_frac            = (1.0 - HYDROGEN_MASSFRAC) / (4.0 * HYDROGEN_MASSFRAC) 
    m_p               = 1.6738e-24 
    
    rho_H_comoving = HYDROGEN_MASSFRAC * dens_all * 3.0991811197281837e-22
    scale_z3       = (1.0 + z2d) ** 3.0
    rho_H          = rho_H_comoving * scale_z3
    
    num_H     = rho_H / m_p
    num_HII   = xHII_all * num_H
    num_HI    = (1.0 - xHII_all) * num_H
    num_HeII  = xHeII_all * num_H
    num_HeIII = xHeIII_all * num_H 
    
    num_p = num_HII
    num_e = num_HII + num_HeII + 2.0 * num_HeIII

    # Compute true baryonic overdensity (1 + delta)
    mean_density = np.mean(dens_all) if np.mean(dens_all) > 0 else 1.0
    delta_all    = dens_all / mean_density

    # =====================================================
    # SHARED CONSTANTS & BACKGROUND FIELDS
    # =====================================================
    Tcmb       = Tcmb0 * (1.0 + z2d)
    e_cgs      = 4.803204712570263e-10  
    me_cgs     = 9.1093837e-28          
    f_a        = 0.4162                 
    
    # Hydrogen 21cm Line Transition Constants
    T_star     = PLANCK * C_LIGHT / (k_B * LAMBDA_21cm) 
    A_10       = 2.85e-15                               
    
    # Helium 3He+ Hyperfine Constants (3.46 cm line)
    A_10_He    = 1.95e-12              
    LAMBDA_He  = 3.46                  
    T_star_He  = PLANCK * C_LIGHT / (k_B * LAMBDA_He)   

    # Read, sort, and construct 1D interpolators mapped to redshift history
    df_global = pd.read_csv("/orcd/data/mvogelsb/004/chcho/LUMINA_all.csv")
    df_global = df_global.sort_values("redshift")
    
    interp_Ja_H  = interp1d(df_global["redshift"], df_global["J_alpha"],    kind='linear', bounds_error=False, fill_value="extrapolate")
    interp_Ja_He = interp1d(df_global["redshift"], df_global["J_alpha_He"], kind='linear', bounds_error=False, fill_value="extrapolate")
    
    # Evaluate global values across 2D coordinates and modulate by local density
    J_alpha_all    = interp_Ja_H(z2d) * delta_all
    J_alpha_He_all = interp_Ja_He(z2d) * delta_all

    # =====================================================
    # COUPLING COEFFICIENTS (x_a and x_c)
    # =====================================================
    # 1. Wouthuysen-Field Radiation Coupling
    const_H   = (16.0 * np.pi**2 * T_star * e_cgs**2 * f_a) / (27.0 * A_10 * Tcmb * me_cgs * C_LIGHT)
    x_a_H_all = const_H * J_alpha_all

    const_He   = (16.0 * np.pi**2 * T_star_He * e_cgs**2 * f_a) / (9.0 * A_10_He * Tcmb * me_cgs * C_LIGHT)
    x_a_He_all = const_He * J_alpha_He_all

    # 2. Hydrogen Collisional Coupling
    k_HH, k_eH, k_pH = get_kappa_2D(T_all)
    x_c_H_all = T_star * (k_HH * num_HI + k_eH * num_e + k_pH * num_p) / (A_10 * Tcmb)
    
    # 3. Helium Collisional Coupling
    a_0           = 5.29177e-9  
    bar_sigma_all = (2.291112e-11 / (k_B * T_all)) * (a_0 ** 2)
    SQRT_all      = np.sqrt((8.0 * k_B * T_all) / (np.pi * me_cgs * C_LIGHT**2))
    rate_He_all   = num_e * SQRT_all * C_LIGHT * bar_sigma_all
    x_c_He_all    = (T_star_He * rate_He_all) / (A_10_He * Tcmb)

    # =====================================================
    # THERMODYNAMIC SPIN TEMPERATURES (Ts)
    # =====================================================
    inv_Tk   = 1.0 / np.maximum(T_all, 1e-3)
    inv_Tcmb = 1.0 / Tcmb

    Ts_H_all = (1.0 + x_a_H_all + x_c_H_all) / (inv_Tcmb + x_a_H_all * inv_Tk + x_c_H_all * inv_Tk)
    Ts_He_all = (1.0 + x_a_He_all + x_c_He_all) / (inv_Tcmb + x_a_He_all * inv_Tk + x_c_He_all * inv_Tk)

    # =====================================================
    # LINE INTENSITIES (Exact Tb Formulations)
    # =====================================================
    cosmo_prefactor = ((OMEGA_b * h**2) / 0.023) * np.sqrt((0.15 * (1.0 + z2d)) / (OMEGA_m * h**2 * 10.0))

    coupling_21 = 1.0 - Tcmb / Ts_H_all
    coupling_He = 1.0 - Tcmb / Ts_He_all

    Tb21_all  = 27.0 * xHI_all   * delta_all * cosmo_prefactor * coupling_21
    TbHe3_all = 1.67 * xHeII_all * delta_all * cosmo_prefactor * coupling_He

    # FIX: Removed np.abs() so raw physical sign (absorption vs emission) is preserved
    return {
        "N":        N,
        "nz_inner": T_in.shape[1],
        "nz_outer": T_out.shape[1],
        "zc_in":    zc_in,
        "zc_out":   zc_out,
        "zc":       zc_all,
        "T":        T_all,
        "xHII":     xHII_all,
        "xHeII":    xHeII_all,
        "xHeIII":   xHeIII_all,
        "density":  dens_all,
        "Ts_H":     Ts_H_all,
        "Ts_He":    Ts_He_all,
        "Tb21":     Tb21_all, 
        "TbHe3":    TbHe3_all,
    }

# =====================================================
# Analytical Age Auxiliary Function
# =====================================================
def age_Gyr(z):
    """Simple analytical approximation for cosmological age vs redshift."""
    return (2.0 / (3.0 * np.sqrt(1.0 - OMEGA_m))) * (1.0 / (100.0 * h * 1.0227e-3)) * np.arcsinh(np.sqrt((1.0 - OMEGA_m) / OMEGA_m) / (1.0 + z)**1.5)



# =====================================================
# 3. Main plotting function (Updated Layout & Targeted Fonts)
# =====================================================
def plot_double_annulus_hive(dataset, rotation_deg=0, y_offset=0.01):
    from matplotlib.colors import LogNorm, SymLogNorm
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes

    N        = dataset["N"]
    nz_inner = dataset["nz_inner"]
    nz_outer = dataset["nz_outer"]
    zc_in    = dataset["zc_in"]
    zc_out   = dataset["zc_out"]

    # Symmetric logarithmic scaling to handle absorption (-) and emission (+)
    vmin_21 = np.nanpercentile(dataset["Tb21"], 1)
    vmax_21 = np.nanpercentile(dataset["Tb21"], 99)# +100
    norm_21 = SymLogNorm(linthresh=1.0, linscale=0.5, vmin=vmin_21, vmax=vmax_21, base=10)

    vmin_He3 = np.nanpercentile(dataset["TbHe3"], 1)
    vmax_He3 = np.nanpercentile(dataset["TbHe3"], 99)
    norm_He3 = SymLogNorm(linthresh=0.01, linscale=0.5, vmin=vmin_He3, vmax=vmax_He3, base=10)

    norm_HII  = LogNorm(vmin=1.0e-4, vmax=1.0)
    norm_HeII = LogNorm(vmin=1.0e-5, vmax=1.0)

    deg = np.pi / 180
    rot = rotation_deg * deg

    R_core  = 0.08
    R_split = 0.45
    R_outer = 1.0

# In inset_specs, replace the norm for fusion:
#(cmr.fusion, norm_21, r"$\delta T^\mathrm{H}_\mathrm{b}$ (mK)", "upper right"),

    
    sectors = [
        {
            "label":       r"$\delta T^\mathrm{H}_\mathrm{b} \mathrm{[mK]}$",
            "label_angle": 45  * deg + rot,
            "t0": 0   * deg + rot, "t1": 90  * deg + rot,
            "cmap": cmr.fusion,   "norm": norm_21,   "data": dataset["Tb21"], #gnuplot2_r
            "dy":  y_offset,
            # --- CUSTOM POSITION TUNING ---
            "label_radius_offset": 0.40,  
            "label_angle_offset":  0.0,   
        },
        {
            "label":       r"$x_\mathrm{H\,II}$",
            "label_angle": 135 * deg + rot,
            "t0": 90  * deg + rot, "t1": 180 * deg + rot,
            "cmap": "viridis",    "norm": norm_HII,  "data": dataset["xHII"],
            "dy":  y_offset,
            # --- CUSTOM POSITION TUNING --- cividis
            "label_radius_offset": 0.40,
            "label_angle_offset":  0.0,
        },
        {
            "label":       r"$x_\mathrm{He\,II}$",
            "label_angle": 225 * deg + rot,
            "t0": 180 * deg + rot, "t1": 270 * deg + rot,
            "cmap": "magma",     "norm": norm_HeII, "data": dataset["xHeII"],
            "dy": -y_offset,
            # --- CUSTOM POSITION TUNING ---
            "label_radius_offset": 0.40,
            "label_angle_offset":  0.0,
        },
        {
            "label":       r"$\delta T^\mathrm{He}_\mathrm{b} \mathrm{[\mu K]}$ ",
            "label_angle": 315 * deg + rot,
            "t0": 270 * deg + rot, "t1": 360 * deg + rot,
            "cmap": "coolwarm",  "norm": norm_He3,  "data": dataset["TbHe3"],
            "dy": -y_offset,
            # --- CUSTOM POSITION TUNING ---
            "label_radius_offset": 0.40,
            "label_angle_offset":  0.0,
        },
    ]

    fig, ax = plt.subplots(figsize=(10, 11), facecolor='white')
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_xlim(-1.30, 1.30)
    ax.set_ylim(-1.30 - y_offset, 1.30 + y_offset)
    fig.subplots_adjust(left=0.05, right=0.95, top=0.96, bottom=0.04)

    r_inner_edges = np.linspace(R_core,          R_split, nz_inner + 1)
    r_outer_edges = np.linspace(R_split - 0.002, R_outer, nz_outer + 1)

    for sec in sectors:
        dy    = sec["dy"]
        theta = np.linspace(sec["t0"], sec["t1"], N + 1)

        TH_in, RR_in = np.meshgrid(theta, r_inner_edges, indexing='ij')
        ax.pcolormesh(
            RR_in * np.cos(TH_in),
            RR_in * np.sin(TH_in) + dy,
            sec["data"][:, :nz_inner],
            cmap=sec["cmap"], norm=sec["norm"],
            shading='flat', edgecolor='none', rasterized=True
        )

        TH_out, RR_out = np.meshgrid(theta, r_outer_edges, indexing='ij')
        ax.pcolormesh(
            RR_out * np.cos(TH_out),
            RR_out * np.sin(TH_out) + dy,
            sec["data"][:, nz_inner:],
            cmap=sec["cmap"], norm=sec["norm"],
            shading='flat', edgecolor='none', rasterized=True
        )

    # Spokes at 0° and 180°
    for ang in [0 * deg + rot, 180 * deg + rot]:
        ax.plot(
            [R_core * np.cos(ang), R_outer * np.cos(ang)],
            [R_core * np.sin(ang) + y_offset, R_outer * np.sin(ang) + y_offset],
            color='white', lw=7, zorder=10, solid_capstyle='butt'
        )
        ax.plot(
            [R_core * np.cos(ang), R_outer * np.cos(ang)],
            [R_core * np.sin(ang) - y_offset, R_outer * np.sin(ang) - y_offset],
            color='white', lw=7, zorder=10, solid_capstyle='butt'
        )

    # Central circles
    ax.add_artist(plt.Circle((0,  y_offset), R_core, color='white', zorder=30))
    ax.add_artist(plt.Circle((0, -y_offset), R_core, color='white', zorder=30))

    # Z milestones → radial positions
    z_milestones = [3.0, 5.0, 10.0, 20.0]

    r_positions = []
    for zm in z_milestones:
        if zm <= zc_in[-1]:
            idx = np.abs(zc_in - zm).argmin()
            r_positions.append(r_inner_edges[idx])
        else:
            idx = np.abs(zc_out - zm).argmin()
            r_positions.append(r_outer_edges[idx])

    for r, zm in zip(r_positions, z_milestones):
        t_gyr   = age_Gyr(zm)
        age_str = f"${t_gyr:.1f}\\,\\mathrm{{Gyr}}$"
        z_str   = f"$z={zm:.0f}$"

        ax.text( r, 0, z_str,
                fontsize=12, fontweight='bold', color='#222222',
                ha='center', va='center', zorder=51,
                bbox=dict(fc='white', ec='none', pad=1.5))
        ax.text(-r, 0, age_str,
                fontsize=12, fontweight='bold', color='#444444',
                ha='center', va='center', zorder=51,
                bbox=dict(fc='white', ec='none', pad=1.5))

    # Sector labels with targeted font size boosts
    for sec in sectors:
        custom_r = R_outer + sec["label_radius_offset"]
        custom_ang = sec["label_angle"] + (sec["label_angle_offset"] * deg)
        
        # FIX: Check if the label contains x_H or x_He to selectively make it larger
        if "x_\\mathrm{H" in sec["label"] or "x_\\mathrm{He" in sec["label"]:
            current_fontsize = 26  # Visually compensates for the shorter height of variable characters
        else:
            current_fontsize = 20  # Keeps the temperature terms at their normal size
        
        ax.text(
            custom_r * np.cos(custom_ang),
            custom_r * np.sin(custom_ang) + sec["dy"],
            sec["label"],
            ha='center', va='center', fontsize=current_fontsize, fontweight='bold', color='#222222'
        )

    # =====================================================
    # ADJUSTED COLORBARS (Wider canvas footprint, same gap)
    # =====================================================
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes

    cbar_h   = "3.5%"
    cbar_pad = 0.0  

    # TUNING STEP: Increased individual colorbar width footprint to 52%
    cbar_w   = "48%"

    inset_specs = [
        (cmr.fusion,  norm_21,   r"$\delta T^\mathrm{H}_\mathrm{b}$ (mK)", "upper right"),
        ("viridis",   norm_HII,  r"$x_\mathrm{H\,II} \quad \quad \quad$", "upper left"),
        ("magma",     norm_HeII, r"$x_\mathrm{He\,II} \quad \quad \quad$", "lower left"),
        ("coolwarm",  norm_He3,  r"$\delta T^\mathrm{He}_\mathrm{b}$ (mK)","lower right"),
    ]
#cmr.amber
    
    # TUNING STEP: Lowered x-start from 0.13 to 0.05, grew full width parameter from 0.74 to 0.90.
    # This combination pulls the outer bounds of the colorbars way closer to the figure margins, 
    # while the wider `cbar_w` matches the expansion rate, keeping the center gap completely locked in place.
    tight_bbox = (0.05, 0.04, 0.90, 0.92)

    for cmap, norm, label, loc in inset_specs:
        cax = inset_axes(
            ax, width=cbar_w, height=cbar_h, loc=loc, 
            bbox_to_anchor=tight_bbox, bbox_transform=ax.transAxes,
            borderpad=cbar_pad
        )
        
        sm  = ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cb  = fig.colorbar(sm, cax=cax, orientation='horizontal')
        cb.ax.tick_params(labelsize=11, colors='#333333')

    plt.show()


# =====================================================
# EXECUTION
# =====================================================
dataset_combined = generate_combined_dataset(
    outer_dir="/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_320",
    inner_dir="/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_below_z_4p75_new/postprocessing/rlc_320",
    SliceFraction=0.1,
    outer_zmin=5.0, outer_zmax=20.0,
    inner_zmin=3.0, inner_zmax=4.75
)

plot_double_annulus_hive(dataset_combined, rotation_deg=0, y_offset=0.02)

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize, TwoSlopeNorm, PowerNorm
from matplotlib.cm import ScalarMappable
from scipy.integrate import quad
import cmasher as cmr
import h5py
from matplotlib.colors import SymLogNorm

Tcmb0 = 2.725

# =====================================================
# Cosmic age at redshift z (Planck 2018 cosmology)
# =====================================================
def age_Gyr(z, H0=67.74, Om=0.3089):
    Ol = 1.0 - Om
    def integrand(zp):
        return 1.0 / ((1 + zp) * np.sqrt(Om * (1 + zp)**3 + Ol))
    result, _ = quad(integrand, z, np.inf)
    return result / (H0 * 1.022e-3)


# =====================================================
# 1. Load a raw chunk and standardize its orientation
# =====================================================
def load_raw_chunk(rlc_dir, SliceFraction, zmin, zmax):
    with h5py.File(f"{rlc_dir}/All.hdf5", "r") as f:
        N = f["Header"].attrs["NumPixels"]
        i_center = int(SliceFraction * N)
        T_block    = f["Temperature"][i_center-1 : i_center+2]
        xHII_block = f["HII_Fraction"][i_center-1 : i_center+2]
        xHeII_block= f["HeII_Fraction"][i_center-1 : i_center+2] * 12.666667
        z = f["Redshifts"][:]

    T     = np.mean(T_block,    axis=0)
    xHII  = np.mean(xHII_block, axis=0)
    xHeII = np.mean(xHeII_block,axis=0)

    mask = (z >= zmin) & (z <= zmax)
    idx  = np.where(mask)[0]
    i0, i1 = idx.min(), idx.max()

    z_chunk    = z[i0:i1+1]
    T_chunk    = T[:,    i0:i1]
    xHII_chunk = xHII[:, i0:i1]
    xHeII_chunk= xHeII[:,i0:i1]

    zc_chunk = 0.5 * (z_chunk[:-1] + z_chunk[1:])
    zc_chunk = zc_chunk[:T_chunk.shape[1]]

    if zc_chunk[0] > zc_chunk[-1]:
        zc_chunk    = zc_chunk[::-1]
        T_chunk     = T_chunk[:,    ::-1]
        xHII_chunk  = xHII_chunk[:, ::-1]
        xHeII_chunk = xHeII_chunk[:, ::-1]

    return T_chunk, xHII_chunk, xHeII_chunk, zc_chunk, N


# =====================================================
# 2. Pipeline to combine raw data and process physics
# =====================================================
def generate_combined_dataset(outer_dir, inner_dir, SliceFraction,
                               outer_zmin, outer_zmax, inner_zmin, inner_zmax):
    T_in,  xHII_in,  xHeII_in,  zc_in,  N = load_raw_chunk(inner_dir, SliceFraction, inner_zmin, inner_zmax)
    T_out, xHII_out, xHeII_out, zc_out, _ = load_raw_chunk(outer_dir, SliceFraction, outer_zmin, outer_zmax)

    z_gap          = zc_out[0] - zc_in[-1]
    zc_out_correct = zc_out - z_gap

    T_all    = np.concatenate([T_in,    T_out],    axis=1)
    xHII_all = np.concatenate([xHII_in, xHII_out], axis=1)
    xHeII_all= np.concatenate([xHeII_in,xHeII_out],axis=1)
    zc_all   = np.concatenate([zc_in,   zc_out_correct])

    z2d       = np.tile(zc_all, (N, 1))
    xHI_all   = 1.0 - xHII_all
    xHeIII_all= 1.0 - xHeII_all

    valid_xHI = xHI_all[xHI_all > 0]
    mean_xHI  = np.mean(valid_xHI) if valid_xHI.size > 0 else 1.0
    delta_all = xHI_all / mean_xHI

    Tcmb     = Tcmb0 * (1 + z2d)
    Ts       = np.maximum(T_all, Tcmb + 1e-5)
    coupling = 1.0 - Tcmb / Ts

    Tb21_all  = 27.0  * xHI_all   * delta_all * np.sqrt((1 + z2d) / 10) * coupling

    Ts       = np.maximum(T_all, Tcmb + 1e-5)
    coupling = 1.0 - Tcmb / Ts
    TbHe3_all =  0.03 * xHeII_all * delta_all * np.sqrt((1 + z2d) / 10) * coupling

    return {
        "N":        N,
        "nz_inner": T_in.shape[1],
        "nz_outer": T_out.shape[1],
        "zc_in":    zc_in,
        "zc_out":   zc_out,
        "zc":       zc_all,
        "T":        T_all,
        "xHII":     xHII_all,
        "xHeII":    xHeII_all,
        "xHeIII":   xHeIII_all,
        "Tb21":     np.abs(Tb21_all),
        "TbHe3":    np.abs(TbHe3_all),
    }


# =====================================================
# 3. Main plotting function (Symmetric 60-degree Sectors)
# =====================================================
def plot_double_annulus_hive(dataset, rotation_deg=0, y_offset=0.01):
    N        = dataset["N"]
    nz_inner = dataset["nz_inner"]
    nz_outer = dataset["nz_outer"]
    zc_in    = dataset["zc_in"]
    zc_out   = dataset["zc_out"]

    norm_21   = LogNorm(vmin=1.0e-7, vmax=1.0e2)
    norm_He3  = LogNorm(vmin=np.percentile(dataset["TbHe3"], 1), vmax=np.percentile(dataset["TbHe3"], 99))
    norm_HII  = LogNorm(vmin=1.0e-2, vmax=1.0)
    norm_HeII = LogNorm(vmin=1.0e-5, vmax=1.0)

    deg = np.pi / 180
    rot = rotation_deg * deg

    R_core  = 0.08
    R_split = 0.45
    R_outer = 1.0

    # Modified layout to use 60-degree wedges with symmetric padding
    sectors = [
        {
            "label":       r"$\delta T^\mathrm{H}_\mathrm{b} \mathrm{[mK]}$",
            "label_angle": 60  * deg + rot,
            "t0": 30  * deg + rot, "t1": 90  * deg + rot,
            "cmap": cmr.fusion,   "norm": norm_21,   "data": dataset["Tb21"],
            "dy":  y_offset,
            "label_radius_offset": 0.40,  
            "label_angle_offset":  0.0,   
        },
        {
            "label":       r"$x_\mathrm{H\,II}$",
            "label_angle": 120 * deg + rot,
            "t0": 90  * deg + rot, "t1": 150 * deg + rot,
            "cmap": "viridis",    "norm": norm_HII,  "data": dataset["xHII"],
            "dy":  y_offset,
            "label_radius_offset": 0.40,
            "label_angle_offset":  0.0,
        },
        {
            "label":       r"$x_\mathrm{He\,II}$",
            "label_angle": 240 * deg + rot,
            "t0": 210 * deg + rot, "t1": 270 * deg + rot,
            "cmap": "magma",     "norm": norm_HeII, "data": dataset["xHeII"],
            "dy": -y_offset,
            "label_radius_offset": 0.40,
            "label_angle_offset":  0.0,
        },
        {
            "label":       r"$\delta T^\mathrm{He}_\mathrm{b} \mathrm{[\mu K]}$ ",
            "label_angle": 300 * deg + rot,
            "t0": 270 * deg + rot, "t1": 330 * deg + rot,
            "cmap": "coolwarm",  "norm": norm_He3,  "data": dataset["TbHe3"],
            "dy": -y_offset,
            "label_radius_offset": 0.40,
            "label_angle_offset":  0.0,
        },
    ]

    fig, ax = plt.subplots(figsize=(10, 11), facecolor='white')
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_xlim(-1.30, 1.30)
    ax.set_ylim(-1.30 - y_offset, 1.30 + y_offset)
    fig.subplots_adjust(left=0.05, right=0.95, top=0.96, bottom=0.04)

    r_inner_edges = np.linspace(R_core,          R_split, nz_inner + 1)
    r_outer_edges = np.linspace(R_split - 0.002, R_outer, nz_outer + 1)

    for sec in sectors:
        dy    = sec["dy"]
        theta = np.linspace(sec["t0"], sec["t1"], N + 1)

        TH_in, RR_in = np.meshgrid(theta, r_inner_edges, indexing='ij')
        ax.pcolormesh(
            RR_in * np.cos(TH_in),
            RR_in * np.sin(TH_in) + dy,
            sec["data"][:, :nz_inner],
            cmap=sec["cmap"], norm=sec["norm"],
            shading='flat', edgecolor='none', rasterized=True
        )

        TH_out, RR_out = np.meshgrid(theta, r_outer_edges, indexing='ij')
        ax.pcolormesh(
            RR_out * np.cos(TH_out),
            RR_out * np.sin(TH_out) + dy,
            sec["data"][:, nz_inner:],
            cmap=sec["cmap"], norm=sec["norm"],
            shading='flat', edgecolor='none', rasterized=True
        )

    # Central boundary padding line
    for ang in [0 * deg + rot, 180 * deg + rot]:
        ax.plot(
            [R_core * np.cos(ang), R_outer * np.cos(ang)],
            [R_core * np.sin(ang) + y_offset, R_outer * np.sin(ang) + y_offset],
            color='white', lw=7, zorder=10, solid_capstyle='butt'
        )
        ax.plot(
            [R_core * np.cos(ang), R_outer * np.cos(ang)],
            [R_core * np.sin(ang) - y_offset, R_outer * np.sin(ang) - y_offset],
            color='white', lw=7, zorder=10, solid_capstyle='butt'
        )

    # Central core mask
    ax.add_artist(plt.Circle((0,  y_offset), R_core, color='white', zorder=30))
    ax.add_artist(plt.Circle((0, -y_offset), R_core, color='white', zorder=30))

    # Redshift milestones mapped tracking down the horizontal gaps
    z_milestones = [3.0, 5.0, 10.0, 20.0]

    r_positions = []
    for zm in z_milestones:
        if zm <= zc_in[-1]:
            idx = np.abs(zc_in - zm).argmin()
            r_positions.append(r_inner_edges[idx])
        else:
            idx = np.abs(zc_out - zm).argmin()
            r_positions.append(r_outer_edges[idx])

    for r, zm in zip(r_positions, z_milestones):
        t_gyr   = age_Gyr(zm)
        age_str = f"${t_gyr:.1f}\\,\\mathrm{{Gyr}}$"
        z_str   = f"$z={zm:.0f}$"

        ax.text( r, 0, z_str,
                fontsize=12, fontweight='bold', color='#222222',
                ha='center', va='center', zorder=51,
                bbox=dict(fc='white', ec='none', pad=1.5))
        ax.text(-r, 0, age_str,
                fontsize=12, fontweight='bold', color='#444444',
                ha='center', va='center', zorder=51,
                bbox=dict(fc='white', ec='none', pad=1.5))

    # Outer field labels placement
    for sec in sectors:
        custom_r = R_outer + sec["label_radius_offset"]
        custom_ang = sec["label_angle"] + (sec["label_angle_offset"] * deg)
        
        if "x_\\mathrm{H" in sec["label"] or "x_\\mathrm{He" in sec["label"]:
            current_fontsize = 26  
        else:
            current_fontsize = 20  
        
        ax.text(
            custom_r * np.cos(custom_ang),
            custom_r * np.sin(custom_ang) + sec["dy"],
            sec["label"],
            ha='center', va='center', fontsize=current_fontsize, fontweight='bold', color='#222222'
        )

    # Colorbars Configuration
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes

    cbar_h   = "3.5%"
    cbar_pad = 0.0  
    cbar_w   = "48%"

    inset_specs = [
        (cmr.fusion,  norm_21,   r"$\delta T^\mathrm{H}_\mathrm{b}$ (mK)", "upper right"),
        ("viridis",   norm_HII,  r"$x_\mathrm{H\,II} \quad \quad \quad$", "upper left"),
        ("magma",     norm_HeII, r"$x_\mathrm{He\,II} \quad \quad \quad$", "lower left"),
        ("coolwarm",  norm_He3,  r"$\delta T^\mathrm{He}_\mathrm{b}$ (mK)","lower right"),
    ]
    
    tight_bbox = (0.05, 0.04, 0.90, 0.92)

    for cmap, norm, label, loc in inset_specs:
        cax = inset_axes(
            ax, width=cbar_w, height=cbar_h, loc=loc, 
            bbox_to_anchor=tight_bbox, bbox_transform=ax.transAxes,
            borderpad=cbar_pad
        )
        
        sm  = ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cb  = fig.colorbar(sm, cax=cax, orientation='horizontal')
        cb.ax.tick_params(labelsize=11, colors='#333333')

    plt.show()


# =====================================================
# EXECUTION
# =====================================================
dataset_combined = generate_combined_dataset(
    outer_dir="/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_320",
    inner_dir="/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_below_z_4p75_new/postprocessing/rlc_320",
    SliceFraction=0.1,
    outer_zmin=5.0, outer_zmax=20.0,
    inner_zmin=3.0, inner_zmax=4.75
)

plot_double_annulus_hive(dataset_combined, rotation_deg=0, y_offset=0.02)

## More Accurate Calculation

In [ ]:
def f_rec(n):
    """
    Returns the recycling fraction f_rec(n) for a given Lyman series level n.
    Values based on Pritchard & Furlanetto (2006).
    
    Args:
        n (int): Principal quantum number (n >= 2).
        
    Returns:
        float: The recycling fraction f_rec.
    """
    # Array indices 0 to 30
    # Values mapped to index n
    f_rec_values = np.array([
        0.0,      # n=0 (Invalid)
        0.0,      # n=1 (Ground state, Invalid)
        1.0000,   # n=2 (Ly-alpha)
        0.0000,   # n=3 (Ly-beta)
        0.2609,   # n=4
        0.3078,   # n=5
        0.3266,   # n=6
        0.3353,   # n=7
        0.3400,   # n=8
        0.3429,   # n=9
        0.3448,   # n=10
        0.3462,   # n=11
        0.3472,   # n=12
        0.3480,   # n=13
        0.3486,   # n=14
        0.3491,   # n=15
        0.3495,   # n=16
        0.3499,   # n=17
        0.3502,   # n=18
        0.3505,   # n=19
        0.3508,   # n=20
        0.3510,   # n=21
        0.3512,   # n=22
        0.3514,   # n=23
        0.3516,   # n=24
        0.3517,   # n=25
        0.3519,   # n=26
        0.3520,   # n=27
        0.3521,   # n=28
        0.3522,   # n=29
        0.3523    # n=30
    ])
    
    # Return value based on n
    if n < len(f_rec_values):
        return f_rec_values[n]
    else:
        # For n > 30, the value approaches 0.359 asymptotically
        return 0.359

In [ ]:
NU_LYMAN_ALPHA = 2.466e15 # n=2 (Ly_alpha)
NU_LYMAN_BETA  = 2.925e15 # n=3 (Ly_beta)
NU_LYMAN_GAMMA = 3.083e15 # n=4 (Ly_gamma)
NU_LYMAN_DELTA = 3.158e15 # n=5 (Ly_delta)
NU_LYMAN_LIMIT = 3.289e15 # Hz (n=infinity, ionization potential)

# Planck's constant (erg s)
PLANCK = 6.62607015e-27
# Wavelength of 21 cm
LAMBDA_21cm = 21.106 
# Wavelength of Lyman-alpha (1215.67 Angstroms, converted to cm)
LAMBDA_1216 = 1215.67e-8

# Boltzmann constant (ergs per kelvin)
k_B = 1.381e-16 
# Hubble constant (Little h)
h = 0.6774
# Hubble constant at the present day
H_0 = 100 * h
H_0_cgs = 100.0 * h * 3.241e-20 # convert from km/s/Mpc to cm^{-1}
# Spontaneous decay rate of the spin flip transition A_10 (per s)  the spontaneous emission coeﬃcient of the 21 cm transition
A_10 = 2.85e-15 

M_SOL = 1.988e33 # 1 Solar mass in grams
YR = 31556736
G_cgs = 6.6743015e-8 # Gravitational Constant #dyn cm^2 g^{−2}
m_p = 1.6738e-24 # Neutral hydrogen mass
e_cgs = 4.803204712570263e-10   # statC (esu)
me_cgs = const.m_e.cgs.value    # g
C_LIGHT  = const.c.cgs.value      # cm / s
X_H = 0.76 # hydrogen fraction
msol_cgs = 1.98847e33
yr_sec = 3.15576e7 

OMEGA_b = 0.0486 # THESAN (Kannan et al. 2022) / 0.044 (Furlanetto et al. 2006)
OMEGA_m = 0.3089 # THESAN (Kannan et al. 2022)  / 0.26 (Furlanetto et al. 2006)
OMEGA_Lambda = 0.6911 #THESAN (Kannan et al. 2022) / 0.74 (Furlanetto et al. 2006)
HYDROGEN_MASSFRAC = 0.76 # global hydrogen mass fraction
y_frac = (1.0 - HYDROGEN_MASSFRAC) / (4.0 * HYDROGEN_MASSFRAC)

# T_star (kelvin)
T_star = PLANCK * C_LIGHT / (k_B * LAMBDA_21cm)

In [ ]:
# -------------------------------------------------------------
# 1. Load the spectrum table (equivalent to mrt_load_spectrum_table())
# -------------------------------------------------------------
class SpectrumTable:
    def __init__(self, filename):
        self.filename = filename
        self._load()

    def _load(self):
        with h5py.File(self.filename, "r") as f:

            # Dimensions (AREPO: Number_of_freq, N_age, N_metallicity)
            self.Nfreq = int(f["Number_of_freq"][()])
            self.N_age = int(f["N_age"][()]) if "N_age" in f else 1
            self.N_metallicity = int(f["N_metallicity"][()]) if "N_metallicity" in f else 1

            # Energy array
            self.PhotEnergy = f["Energy"][()]  # eV, length Nfreq
            self.freq = f["freq"][()]  # Hz, length Nfreq
            freq = self.freq.reshape(1, 1, self.Nfreq)


            # Luminosity array (erg/s/Hz/Msol)
            Lnu = f["Lnu"][()]  # length Nfreq * N_age * N_metallicity
            Lnu = Lnu.reshape(self.N_metallicity, self.N_age, self.Nfreq)

            # Convert erg/s/Hz/Msol → photons/s/Hz/gram (same as AREPO: divide by PLANCK)
            PLANCK = 6.62607015e-27  # erg*s
            Msol2gram = 1.989e33 # Msol to gram
            m_p = 1.673e-24 # 1 proton mass

            #self.LumPhotons = (Lnu / (PLANCK * self.freq)) * (m_p / Msol2gram)



            # compute epsilon_b
            self.LumPhotons = (Lnu / (PLANCK * freq)) * (m_p / Msol2gram)

            # flatten back to original AREPO-style layout
            self.LumPhotons = self.LumPhotons.reshape(-1)

            #self.LumPhotons = Lnu * m_p / (PLANCK * Msol2gram)

            # Age array (Gyr)
            if self.N_age > 1:
                age_arr = f["age"][()]
                age_arr[age_arr < 1e-6] = 1e-6
                self.AgeGyr = age_arr
                self.LogAge = np.log10(age_arr)
            else:
                self.LogAge = np.array([0.0])
                self.AgeGyr = np.array([0.0])

            # Metallicity array (Z)
            if self.N_metallicity > 1:
                self.Metallicity = f["metallicity"][()]
                self.LogMetallicity = np.log10(self.Metallicity)
            else:
                self.LogMetallicity = np.array([0.0])
                self.Metallicity = np.array([0.0])


        print("Loaded spectrum table:")
        print(f"  Nfreq = {self.Nfreq}")
        print(f"  N_age = {self.N_age}")
        print(f"  N_metallicity = {self.N_metallicity}")

    # -------------------------------------------------------------
    # 2. Return *full spectrum* for given age & metallicity
    # -------------------------------------------------------------
    def full_spectrum(self, i_age, i_metallicity):
        """Returns (Frequency array, LumPhotons array) for specific Z and age indices."""
        # Indexing: Z (outer) -> Age (middle) -> Freq (inner)
        offset = i_metallicity * self.N_age * self.Nfreq + i_age * self.Nfreq
        return self.freq, self.LumPhotons[offset:offset + self.Nfreq]

In [ ]:
# -------------------------------------------------------------
# 3. Main Execution and Plotting Loop
# -------------------------------------------------------------

# Load the table (Update this path if necessary, but it looks correct)
HDF5_FILEPATH = "/orcd/data/mvogelsb/004/ozier/ZOOM_XL/sharedData/bpass_spectra_bin_chab100_v2.2.1.hdf5"
try:
    spec = SpectrumTable(HDF5_FILEPATH)
except FileNotFoundError:
    print(f"ERROR: Could not find the file at {HDF5_FILEPATH}. Please ensure the path is correct.")
    exit()
except Exception as e:
    print(f"ERROR: Failed to load HDF5 data: {e}")
    exit()

In [ ]:
def get_spec(Z_target):
    # find index
    iZ = np.where(spec.Metallicity == Z_target)[0][0]

    #print("Metallicity index:", iZ)

    all_spectra = []
    all_ages = spec.AgeGyr.copy()

    for i_age in range(spec.N_age):
        freq, lum = spec.full_spectrum(i_age, iZ)
        all_spectra.append(lum)

    all_spectra = np.array(all_spectra)  
    # shape = (N_age, Nfreq)

    dt = np.zeros_like(all_ages)

    dt[0] = all_ages[0]
    dt[1:] = all_ages[1:] - all_ages[:-1]

    # conversion factor
    Gyr_to_sec = 3.15576e16  # seconds in 1 Gyr

    dt_sec = dt * Gyr_to_sec

    # reshape dt so it broadcasts along frequency axis
    dt_sec_2d = dt_sec[:, None]   # shape (N_age, 1)

    # time integration
    N_nu = np.sum(all_spectra * dt_sec_2d, axis=0)

    return freq, N_nu


freq, N_nu_5 = get_spec(1.0e-5)
freq, N_nu_4 = get_spec(1.0e-4)
freq, N_nu_3 = get_spec(1.0e-3)
freq, N_nu_2 = get_spec(1.0e-2)




In [ ]:
from scipy.interpolate import CubicSpline
# ============================================================
# Collisional de-excitation rate tables (precomputed once)
# ============================================================

# --- H–H collisions ---
_T_HH = np.log10([
    1, 2, 4, 6, 8, 10, 15, 20, 25, 30, 40, 50, 60, 70, 80, 90, 100,
    200, 300, 500, 700, 1000, 2000, 3000, 5000, 7000, 10000
])

_k_HH = np.log10([
    1.38e-13, 1.43e-13, 2.71e-13, 6.60e-13, 1.47e-12, 2.88e-12,
    9.10e-12, 1.78e-11, 2.73e-11, 3.67e-11, 5.38e-11, 6.86e-11,
    8.14e-11, 9.25e-11, 1.02e-10, 1.11e-10, 1.19e-10, 1.75e-10,
    2.09e-10, 2.56e-10, 2.91e-10, 3.31e-10, 4.27e-10, 4.97e-10,
    6.03e-10, 6.87e-10, 7.87e-10
])


# --- e–H collisions ---
_T_eH = np.log10([
    1, 2, 5, 10, 20, 50, 100, 200, 500, 1000,
    2000, 3000, 5000, 7000, 10000, 15000, 20000
])

_k_eH = np.log10([
    2.39e-10, 3.37e-10, 5.30e-10, 7.46e-10, 1.05e-9, 1.63e-9,
    2.26e-9, 3.11e-9, 4.59e-9, 5.92e-9, 7.15e-9, 7.71e-9,
    8.17e-9, 8.32e-9, 8.37e-9, 8.29e-9, 8.11e-9
])


# --- p–H collisions ---
_T_pH = _T_eH.copy()

_k_pH = np.log10([
    0.40e-9, 0.45e-9, 0.430e-9, 0.369e-9, 0.317e-9, 0.3047e-9,
    0.3379e-9, 0.4043e-9, 0.5471e-9, 0.7051e-9, 0.9167e-9,
    1.070e-9, 1.301e-9, 1.480e-9, 1.695e-9, 1.975e-9, 2.201e-9
])

def kappa_HH(T_K):
    """
    H–H collisional de-excitation rate coefficient (cm^3 s^-1).
    """
    # Build interpolator once
    interp_func = CubicSpline(_T_HH, _k_HH)

    # Evaluate
    k_interp = interp_func(np.log10(T_K))
    return 10.0 ** k_interp #np.interp(np.log10(T_K), _T_HH, _k_HH)


def kappa_eH(T_K):
    """
    e–H collisional de-excitation rate coefficient (cm^3 s^-1).
    """
    # Build interpolator once
    interp_func = CubicSpline(_T_eH, _k_eH)

    # Evaluate
    k_interp = interp_func(np.log10(T_K))
    return 10.0 ** k_interp #np.interp(np.log10(T_K), _T_eH, _k_eH)


def kappa_pH(T_K):
    """
    p–H collisional de-excitation rate coefficient (cm^3 s^-1).
    """

    # Build interpolator once
    interp_func = CubicSpline(_T_pH, _k_pH)

    # Evaluate
    k_interp = interp_func(np.log10(T_K))
    return 10.0 ** k_interp


# --- Data Generation ---
# Create an array of T_K values from 1 (10^0) to 10000 (10^4)
# We use np.logspace to get points evenly spaced in log scale
T_K_plot = np.logspace(0, 5, 400)  # 200 points from 1 to 10000

# Calculate the corresponding interpolated kappa values
kappa_plot = kappa_HH(T_K_plot)
kappa_plot_eH = kappa_eH(T_K_plot)
kappa_plot_pH = kappa_pH(T_K_plot)


# --- Plotting ---
print("Generating plot...")

# Set up the plot
#plt.figure(figsize=(9, 8))

# Plot the interpolated line
#plt.loglog(T_K_plot, kappa_plot, '--', color='#fe218b', label=r'$\kappa^{HH}_{10}$')
#plt.loglog(T_K_plot, kappa_plot_eH, ':', color='#06d6a0', label=r'$\kappa^{eH}_{10}$')
#plt.loglog(T_K_plot, kappa_plot_pH,  color='#21b0fe', label=r'$\kappa^{pH}_{10}$')


# Plot the original data points to verify the fit
#plt.loglog(T_K_values_HH, kappa_values_HH, 'o', color='#fe218b')
#plt.loglog(T_K_values_eH, kappa_values_eH, 'o', color='#06d6a0')
#plt.loglog(T_K_values_pH, kappa_values_pH, 'o', color='#21b0fe')


# --- Customize the Plot ---
# Set axis scales (already loglog, but good to be explicit)
#plt.xscale('log')
#plt.yscale('log')

# Set axis limits
#plt.xlim(1, 10000)
#plt.ylim(8.0e-14, 1.0e-7)

# Add title and labels (using LaTeX for nice formatting)
#plt.xlabel(r'$T_K$ (K)', fontsize=15)
#plt.ylabel(r'$\kappa$ (cm$^3$ s$^{-1}$)', fontsize=15)

# Add a legend
#plt.legend(loc="lower right", fontsize=15)

# Add grid lines for better readability
#plt.grid(True, which="both", linestyle='--', linewidth=0.5, alpha=0.7)

# Save the plot to a file
#plt.savefig('collisional_rate_vs_temp.png', dpi=300)

# Display the plot
#print("Plot saved as 'collisional_rate_vs_temp.png'")
#plt.show()


# T_gamma
# In our case, this temperature is the CMB temperature



In [ ]:
#def num_den_b(z):
#    rho_crit = (3.0 * H_0_cgs * H_0_cgs) / (8.0 * np.pi * G_cgs)
#    return OMEGA_b * rho_crit * (1.0 + z)**(3.0) / m_p
    
# Hubble parameter at z: H(z)
def H_cgm(z):
    """
    Calculates the Hubble parameter H(z) at a given redshift z.
    Assumes a flat Lambda-CDM cosmology.
    """
    return H_0_cgs * (OMEGA_m * (1.0 + z)**3.0 + OMEGA_Lambda)**0.5

# Hubble parameter at z: H(z)
def H(z):
    return H_0 * (OMEGA_m * (1.0 + z)**3.0 + OMEGA_Lambda)**0.5

# CMB brightness temperature at z: T_CMB(z)
def T_CMB(z):
    T_CMB_0 = 2.7255 # Kelvin
    return T_CMB_0 * (1.0 + z)

def x_c(z):
    #print(T_K(z), kappa_HH(T_K(z)), kappa_eH(T_K(z)), kappa_pH(T_K(z)), n_H(z), n_e(z), n_p(z), T_gamma(z))
    return T_star * (kappa_HH(T_K(z)) * n_H(z) + kappa_eH(T_K(z)) * n_e(z) + kappa_pH(T_K(z)) * n_p(z)) / (A_10 * T_gamma(z))

def tau_GP(z):
#Gunn-Peterson optical depth (dimensionless)
    return 3.0e5 * HI_frac(z) * ((1.0+z)/7.0)**1.5

def S_alpha(z):
# Supression factor
    return np.exp(-0.803*(T_K(z)**(-2.0/3.0)) * (1.0e-6*tau_GP(z))**(1.0/3.0))

def cosmic_time(z):
    """
    Cosmic time at redshift z (seconds).
    Uses your H_cgm(z).
    """
    integrand = lambda zp: 1.0 / ((1.0 + zp) * H_cgm(zp))
    
    # First compute age of universe today
    t0, _ = quad(integrand, 0.0, 1e4)   # 1e4 is effectively infinity
    
    # Then subtract integral from 0 to z
    tz_part, _ = quad(integrand, 0.0, z)
    
    return t0 - tz_part


In [ ]:
import numpy as np
from scipy.interpolate import CubicSpline
import astropy.constants as const

# ============================================================
# Physics Constants
# ============================================================
PLANCK       = 6.62607015e-27  # erg s
k_B          = 1.381e-16        # erg/K
C_LIGHT      = const.c.cgs.value # cm / s
LAMBDA_21cm  = 21.106           # cm
A_10         = 2.85e-15         # s^-1
T_star       = PLANCK * C_LIGHT / (k_B * LAMBDA_21cm) # ~0.068 K

# ============================================================
# Precomputed Collisional de-excitation rate tables
# ============================================================
_T_HH = np.log10([1, 2, 4, 6, 8, 10, 15, 20, 25, 30, 40, 50, 60, 70, 80, 90, 100, 200, 300, 500, 700, 1000, 2000, 3000, 5000, 7000, 10000])
_k_HH = np.log10([1.38e-13, 1.43e-13, 2.71e-13, 6.60e-13, 1.47e-12, 2.88e-12, 9.10e-12, 1.78e-11, 2.73e-11, 3.67e-11, 5.38e-11, 6.86e-11, 8.14e-11, 9.25e-11, 1.02e-10, 1.11e-10, 1.19e-10, 1.75e-10, 2.09e-10, 2.56e-10, 2.91e-10, 3.31e-10, 4.27e-10, 4.97e-10, 6.03e-10, 6.87e-10, 7.87e-10])

_T_eH = np.log10([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000, 20000])
_k_eH = np.log10([2.39e-10, 3.37e-10, 5.30e-10, 7.46e-10, 1.05e-9, 1.63e-9, 2.26e-9, 3.11e-9, 4.59e-9, 5.92e-9, 7.15e-9, 7.71e-9, 8.17e-9, 8.32e-9, 8.37e-9, 8.29e-9, 8.11e-9])

_T_pH = _T_eH.copy()
_k_pH = np.log10([0.40e-9, 0.45e-9, 0.430e-9, 0.369e-9, 0.317e-9, 0.3047e-9, 0.3379e-9, 0.4043e-9, 0.5471e-9, 0.7051e-9, 0.9167e-9, 1.070e-9, 1.301e-9, 1.480e-9, 1.695e-9, 1.975e-9, 2.201e-9])

# Instantiated globally ONCE for speed
_spline_HH = CubicSpline(_T_HH, _k_HH, extrapolate=True)
_spline_eH = CubicSpline(_T_eH, _k_eH, extrapolate=True)
_spline_pH = CubicSpline(_T_pH, _k_pH, extrapolate=True)

def get_kappa_2D(T_K):
    """Evaluates rate coefficients safely across an entire 2D temperature array."""
    # Clip to avoid log10(0) or extreme out-of-bounds artifacts
    logT = np.log10(np.clip(T_K, 1.0, 20000.0))
    
    k_HH = 10.0 ** _spline_HH(logT)
    k_eH = 10.0 ** _spline_eH(logT)
    k_pH = 10.0 ** _spline_pH(logT)
    return k_HH, k_eH, k_pH

In [ ]:
import pandas as pd
from scipy.interpolate import interp1d, CubicSpline

In [ ]:
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.cm import ScalarMappable
from scipy.interpolate import interp1d, CubicSpline
import astropy.constants as const
import cmasher as cmr  # Assuming 'cmr' refers to cmasher in your script

# ============================================================
# GLOBAL CONSTANTS & PHYSICS PARAMETERS
# ============================================================
PLANCK       = 6.62607015e-27  # erg s
k_B          = 1.381e-16        # erg/K
C_LIGHT      = const.c.cgs.value # cm / s
LAMBDA_21cm  = 21.106           # cm
Tcmb0        = 2.7255           # K (CMB temperature at z=0)

# Cosmological Parameters (e.g., THESAN suite)
OMEGA_b      = 0.0486 
OMEGA_m      = 0.3089 
h            = 0.6774

# ============================================================
# PRECOMPUTED COLLISIONAL RATE TABLES & GLOBAL SPLINES
# ============================================================
_T_HH = np.log10([1, 2, 4, 6, 8, 10, 15, 20, 25, 30, 40, 50, 60, 70, 80, 90, 100, 200, 300, 500, 700, 1000, 2000, 3000, 5000, 7000, 10000])
_k_HH = np.log10([1.38e-13, 1.43e-13, 2.71e-13, 6.60e-13, 1.47e-12, 2.88e-12, 9.10e-12, 1.78e-11, 2.73e-11, 3.67e-11, 5.38e-11, 6.86e-11, 8.14e-11, 9.25e-11, 1.02e-10, 1.11e-10, 1.19e-10, 1.75e-10, 2.09e-10, 2.56e-10, 2.91e-10, 3.31e-10, 4.27e-10, 4.97e-10, 6.03e-10, 6.87e-10, 7.87e-10])

_T_eH = np.log10([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000, 20000])
_k_eH = np.log10([2.39e-10, 3.37e-10, 5.30e-10, 7.46e-10, 1.05e-9, 1.63e-9, 2.26e-9, 3.11e-9, 4.59e-9, 5.92e-9, 7.15e-9, 7.71e-9, 8.17e-9, 8.32e-9, 8.37e-9, 8.29e-9, 8.11e-9])

_T_pH = _T_eH.copy()
_k_pH = np.log10([0.40e-9, 0.45e-9, 0.430e-9, 0.369e-9, 0.317e-9, 0.3047e-9, 0.3379e-9, 0.4043e-9, 0.5471e-9, 0.7051e-9, 0.9167e-9, 1.070e-9, 1.301e-9, 1.480e-9, 1.695e-9, 1.975e-9, 2.201e-9])

_spline_HH = CubicSpline(_T_HH, _k_HH, extrapolate=True)
_spline_eH = CubicSpline(_T_eH, _k_eH, extrapolate=True)
_spline_pH = CubicSpline(_T_pH, _k_pH, extrapolate=True)

def get_kappa_2D(T_K):
    """Evaluates Hydrogen rate coefficients safely across an entire 2D temperature array."""
    logT = np.log10(np.clip(T_K, 1.0, 20000.0))
    k_HH = 10.0 ** _spline_HH(logT)
    k_eH = 10.0 ** _spline_eH(logT)
    k_pH = 10.0 ** _spline_pH(logT)
    return k_HH, k_eH, k_pH


# =====================================================
# 1. Load a raw chunk and standardize orientation
# =====================================================
def load_raw_chunk(rlc_dir, SliceFraction, zmin, zmax):
    with h5py.File(f"{rlc_dir}/All.hdf5", "r") as f:
        N = f["Header"].attrs["NumPixels"]
        i_center = int(SliceFraction * N)
        T_block    = f["Temperature"][i_center-1 : i_center+2]
        xHII_block = f["HII_Fraction"][i_center-1 : i_center+2]
        xHeII_block= f["HeII_Fraction"][i_center-1 : i_center+2] * 12.666667
        z = f["Redshifts"][:]
        
    with h5py.File(f"{rlc_dir}/Density.hdf5", "r") as f_dens:
        density_block = f_dens["Density"][i_center-1 : i_center+2]

    T       = np.mean(T_block,    axis=0)
    xHII    = np.mean(xHII_block, axis=0)
    xHeII   = np.mean(xHeII_block, axis=0)
    density = np.mean(density_block, axis=0)

    mask = (z >= zmin) & (z <= zmax)
    idx  = np.where(mask)[0]
    i0, i1 = idx.min(), idx.max()

    z_chunk       = z[i0:i1+1]
    T_chunk       = T[:,    i0:i1]
    xHII_chunk    = xHII[:, i0:i1]
    xHeII_chunk   = xHeII[:, i0:i1]
    density_chunk = density[:, i0:i1]

    zc_chunk = 0.5 * (z_chunk[:-1] + z_chunk[1:])
    zc_chunk = zc_chunk[:T_chunk.shape[1]]

    if zc_chunk[0] > zc_chunk[-1]:
        zc_chunk      = zc_chunk[::-1]
        T_chunk       = T_chunk[:,    ::-1]
        xHII_chunk    = xHII_chunk[:, ::-1]
        xHeII_chunk   = xHeII_chunk[:, ::-1]
        density_chunk = density_chunk[:, ::-1]

    return T_chunk, xHII_chunk, xHeII_chunk, density_chunk, zc_chunk, N


# =====================================================
# 2. Main Processing Pipeline 
# =====================================================
def generate_combined_dataset(outer_dir, inner_dir, SliceFraction,
                               outer_zmin, outer_zmax, inner_zmin, inner_zmax):
    # Unpack chunks
    T_in,  xHII_in,  xHeII_in,  dens_in,  zc_in,  N = load_raw_chunk(inner_dir, SliceFraction, inner_zmin, inner_zmax)
    T_out, xHII_out, xHeII_out, dens_out, zc_out, _ = load_raw_chunk(outer_dir, SliceFraction, outer_zmin, outer_zmax)

    z_gap          = zc_out[0] - zc_in[-1]
    zc_out_correct = zc_out - z_gap

    T_all    = np.concatenate([T_in,    T_out],    axis=1)
    xHII_all = np.concatenate([xHII_in, xHII_out], axis=1)
    xHeII_all= np.concatenate([xHeII_in, xHeII_out], axis=1)
    dens_all = np.concatenate([dens_in, dens_out], axis=1)
    zc_all   = np.concatenate([zc_in,   zc_out_correct])

    z2d       = np.tile(zc_all, (N, 1))
    xHI_all   = 1.0 - xHII_all
    xHeIII_all= 1.0 - xHeII_all

    # =====================================================
    # PHYSICAL NUMBER DENSITIES (CGS Units)
    # =====================================================
    HYDROGEN_MASSFRAC = 0.76
    y_frac            = (1.0 - HYDROGEN_MASSFRAC) / (4.0 * HYDROGEN_MASSFRAC) 
    m_p               = 1.6738e-24 
    
    rho_H_comoving = HYDROGEN_MASSFRAC * dens_all * 3.0991811197281837e-22
    scale_z3       = (1.0 + z2d) ** 3.0
    rho_H          = rho_H_comoving * scale_z3
    
    num_H     = rho_H / m_p
    num_HII   = xHII_all * num_H
    num_HI    = (1.0 - xHII_all) * num_H
    num_HeII  = xHeII_all * num_H
    num_HeIII = xHeIII_all * num_H 
    
    num_p = num_HII
    num_e = num_HII + num_HeII + 2.0 * num_HeIII

    # Compute true baryonic overdensity (1 + delta)
    mean_density = np.mean(dens_all) if np.mean(dens_all) > 0 else 1.0
    delta_all    = dens_all / mean_density

    # =====================================================
    # SHARED CONSTANTS & BACKGROUND FIELDS
    # =====================================================
    Tcmb       = Tcmb0 * (1.0 + z2d)
    e_cgs      = 4.803204712570263e-10  
    me_cgs     = 9.1093837e-28          
    f_a        = 0.4162                 
    
    # Hydrogen 21cm Line Transition Constants
    T_star     = PLANCK * C_LIGHT / (k_B * LAMBDA_21cm) 
    A_10       = 2.85e-15                               
    
    # Helium 3He+ Hyperfine Constants (3.46 cm line)
    A_10_He    = 1.95e-12              
    LAMBDA_He  = 3.46                  
    T_star_He  = PLANCK * C_LIGHT / (k_B * LAMBDA_He)   

    # Read, sort, and construct 1D interpolators mapped to redshift history
    df_global = pd.read_csv("/orcd/data/mvogelsb/004/chcho/LUMINA_all.csv")
    df_global = df_global.sort_values("redshift")
    
    interp_Ja_H  = interp1d(df_global["redshift"], df_global["J_alpha"],    kind='linear', bounds_error=False, fill_value="extrapolate")
    interp_Ja_He = interp1d(df_global["redshift"], df_global["J_alpha_He"], kind='linear', bounds_error=False, fill_value="extrapolate")
    
    # Evaluate global values across 2D coordinates and modulate by local density
    J_alpha_all    = interp_Ja_H(z2d) * delta_all
    J_alpha_He_all = interp_Ja_He(z2d) * delta_all

    # =====================================================
    # COUPLING COEFFICIENTS (x_a and x_c)
    # =====================================================
    # 1. Wouthuysen-Field Radiation Coupling
    const_H   = (16.0 * np.pi**2 * T_star * e_cgs**2 * f_a) / (27.0 * A_10 * Tcmb * me_cgs * C_LIGHT)
    x_a_H_all = const_H * J_alpha_all

    const_He   = (16.0 * np.pi**2 * T_star_He * e_cgs**2 * f_a) / (9.0 * A_10_He * Tcmb * me_cgs * C_LIGHT)
    x_a_He_all = const_He * J_alpha_He_all

    # 2. Hydrogen Collisional Coupling
    k_HH, k_eH, k_pH = get_kappa_2D(T_all)
    x_c_H_all = T_star * (k_HH * num_HI + k_eH * num_e + k_pH * num_p) / (A_10 * Tcmb)
    
    # 3. Helium Collisional Coupling
    a_0           = 5.29177e-9  
    bar_sigma_all = (2.291112e-11 / (k_B * T_all)) * (a_0 ** 2)
    SQRT_all      = np.sqrt((8.0 * k_B * T_all) / (np.pi * me_cgs * C_LIGHT**2))
    rate_He_all   = num_e * SQRT_all * C_LIGHT * bar_sigma_all
    x_c_He_all    = (T_star_He * rate_He_all) / (A_10_He * Tcmb)

    # =====================================================
    # THERMODYNAMIC SPIN TEMPERATURES (Ts)
    # =====================================================
    inv_Tk   = 1.0 / np.maximum(T_all, 1e-3)
    inv_Tcmb = 1.0 / Tcmb

    Ts_H_all = (1.0 + x_a_H_all + x_c_H_all) / (inv_Tcmb + x_a_H_all * inv_Tk + x_c_H_all * inv_Tk)
    Ts_He_all = (1.0 + x_a_He_all + x_c_He_all) / (inv_Tcmb + x_a_He_all * inv_Tk + x_c_He_all * inv_Tk)

    # =====================================================
    # LINE INTENSITIES (Exact Tb Formulations)
    # =====================================================
    cosmo_prefactor = ((OMEGA_b * h**2) / 0.023) * np.sqrt((0.15 * (1.0 + z2d)) / (OMEGA_m * h**2 * 10.0))

    coupling_21 = 1.0 - Tcmb / Ts_H_all
    coupling_He = 1.0 - Tcmb / Ts_He_all

    Tb21_all  = 27.0 * xHI_all   * delta_all * cosmo_prefactor * coupling_21
    TbHe3_all = 1.67 * xHeII_all * delta_all * cosmo_prefactor * coupling_He

    # FIX: Removed np.abs() so raw physical sign (absorption vs emission) is preserved
    return {
        "N":        N,
        "nz_inner": T_in.shape[1],
        "nz_outer": T_out.shape[1],
        "zc_in":    zc_in,
        "zc_out":   zc_out,
        "zc":       zc_all,
        "T":        T_all,
        "xHII":     xHII_all,
        "xHeII":    xHeII_all,
        "xHeIII":   xHeIII_all,
        "density":  dens_all,
        "Ts_H":     Ts_H_all,
        "Ts_He":    Ts_He_all,
        "Tb21":     Tb21_all, 
        "TbHe3":    TbHe3_all,
    }

# =====================================================
# Analytical Age Auxiliary Function
# =====================================================
def age_Gyr(z):
    """Simple analytical approximation for cosmological age vs redshift."""
    return (2.0 / (3.0 * np.sqrt(1.0 - OMEGA_m))) * (1.0 / (100.0 * h * 1.0227e-3)) * np.arcsinh(np.sqrt((1.0 - OMEGA_m) / OMEGA_m) / (1.0 + z)**1.5)

# =====================================================
# 3. Main plotting function (Symmetric Semicircle)
# =====================================================
def plot_double_annulus_hive(dataset, rotation_deg=0, y_offset=0.12):
    from matplotlib.colors import LogNorm, SymLogNorm
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes

    N        = dataset["N"]
    nz_inner = dataset["nz_inner"]
    nz_outer = dataset["nz_outer"]
    zc_in    = dataset["zc_in"]
    zc_out   = dataset["zc_out"]

    # Symmetric logarithmic scaling to handle absorption (-) and emission (+)
    vmin_21 = np.nanpercentile(dataset["Tb21"], 1)
    vmax_21 = np.nanpercentile(dataset["Tb21"], 99)# +100
    norm_21 = SymLogNorm(linthresh=1.0, linscale=0.5, vmin=vmin_21, vmax=vmax_21, base=10)

    vmin_He3 = np.nanpercentile(dataset["TbHe3"], 1)
    vmax_He3 = np.nanpercentile(dataset["TbHe3"], 99)
    norm_He3 = SymLogNorm(linthresh=0.01, linscale=0.5, vmin=vmin_He3, vmax=vmax_He3, base=10)

    norm_HII  = LogNorm(vmin=1.0e-4, vmax=1.0)
    norm_HeII = LogNorm(vmin=1.0e-5, vmax=1.0)

    deg = np.pi / 180
    rot = rotation_deg * deg

    R_core  = 0.08
    R_split = 0.45
    R_outer = 1.0

    sectors = [
        {
            "label":       r"$\delta T^\mathrm{H}_\mathrm{b} \mathrm{[mK]}$",
            "label_angle": 22.5 * deg + rot,
            "t0": 0 * deg + rot, "t1": 45 * deg + rot,
            "cmap": "coolwarm",   "norm": norm_21,   "data": dataset["Tb21"],
            "dy":  y_offset, "label_radius_offset": 0.22,
        },
        {
            "label":       r"$x_\mathrm{H\,II}$",
            "label_angle": 67.5 * deg + rot,
            "t0": 45 * deg + rot, "t1": 90 * deg + rot,
            "cmap": "viridis",    "norm": norm_HII,  "data": dataset["xHII"],
            "dy":  y_offset, "label_radius_offset": 0.22,
        },
        {
            "label":       r"$x_\mathrm{He\,II}$",
            "label_angle": 112.5 * deg + rot,
            "t0": 90 * deg + rot, "t1": 135 * deg + rot,
            "cmap": "magma",     "norm": norm_HeII, "data": dataset["xHeII"],
            "dy":  y_offset, "label_radius_offset": 0.22,
        },
        {
            "label":       r"$\delta T^\mathrm{He}_\mathrm{b} \mathrm{[\mu K]}$",
            "label_angle": 157.5 * deg + rot,
            "t0": 135 * deg + rot, "t1": 180 * deg + rot,
            "cmap": "plasma",  "norm": norm_He3,  "data": dataset["TbHe3"],
            "dy":  y_offset, "label_radius_offset": 0.22,
        },
    ]

    fig, ax = plt.subplots(figsize=(10, 10), facecolor='white')
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_xlim(-1.30, 1.30)
    ax.set_ylim(-1.30, 1.30)
    fig.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.05)

    r_inner_edges = np.linspace(R_core,          R_split, nz_inner + 1)
    r_outer_edges = np.linspace(R_split - 0.002, R_outer, nz_outer + 1)

    for sec in sectors:
        dy    = sec["dy"]
        theta = np.linspace(sec["t0"], sec["t1"], N + 1)

        TH_in, RR_in = np.meshgrid(theta, r_inner_edges, indexing='ij')
        ax.pcolormesh(
            RR_in * np.cos(TH_in),
            RR_in * np.sin(TH_in) + dy,
            sec["data"][:, :nz_inner],
            cmap=sec["cmap"], norm=sec["norm"],
            shading='flat', edgecolor='none', rasterized=True
        )

        TH_out, RR_out = np.meshgrid(theta, r_outer_edges, indexing='ij')
        ax.pcolormesh(
            RR_out * np.cos(TH_out),
            RR_out * np.sin(TH_out) + dy,
            sec["data"][:, nz_inner:],
            cmap=sec["cmap"], norm=sec["norm"],
            shading='flat', edgecolor='none', rasterized=True
        )

    for ang in [0*deg, 45*deg, 90*deg, 135*deg, 180*deg]:
        ax.plot(
            [R_core * np.cos(ang), R_outer * np.cos(ang)],
            [R_core * np.sin(ang) + y_offset, R_outer * np.sin(ang) + y_offset],
            color='white', lw=4, zorder=10, solid_capstyle='butt'
        )

    ax.add_artist(plt.Circle((0, y_offset), R_core, color='white', zorder=30))

    z_milestones = [3.0, 5.0, 10.0, 20.0]
    r_positions = []
    for zm in z_milestones:
        if zm <= zc_in[-1]:
            idx = np.abs(zc_in - zm).argmin()
            r_positions.append(r_inner_edges[idx])
        else:
            idx = np.abs(zc_out - zm).argmin()
            r_positions.append(r_outer_edges[idx])

    for r, zm in zip(r_positions, z_milestones):
        t_gyr   = age_Gyr(zm)
        age_str = f"${t_gyr:.1f}\\,\\mathrm{{Gyr}}$"
        z_str   = f"$z={zm:.0f}$"

        ax.text( r, y_offset - 0.06, z_str,
                fontsize=11, fontweight='bold', color='#222222',
                ha='center', va='top', zorder=51,
                bbox=dict(fc='white', ec='none', pad=1.0))
        ax.text(-r, y_offset - 0.06, age_str,
                fontsize=11, fontweight='bold', color='#444444',
                ha='center', va='top', zorder=51,
                bbox=dict(fc='white', ec='none', pad=1.0))

    for sec in sectors:
        custom_r = R_outer + sec["label_radius_offset"]
        custom_ang = sec["label_angle"]
        current_fontsize = 24 if ("x_\\mathrm{H" in sec["label"] or "x_\\mathrm{He" in sec["label"]) else 19  
        
        ax.text(
            custom_r * np.cos(custom_ang),
            custom_r * np.sin(custom_ang) + sec["dy"],
            sec["label"],
            ha='center', va='center', fontsize=current_fontsize, fontweight='bold', color='#222222'
        )

    # 2x2 Layout Specifications
    cbar_specs = [
        ("magma",     norm_HeII, r"$x_\mathrm{He\,II}$",                      (0.08, 0.26, 0.38, 0.03)),
        ("viridis",   norm_HII,  r"$x_\mathrm{H\,II}$",                       (0.54, 0.26, 0.38, 0.03)),
        ("plasma",  norm_He3,  r"$\delta T^\mathrm{He}_\mathrm{b}\ [\mu\mathrm{K}]$", (0.08, 0.11, 0.38, 0.03)),
        ("coolwarm",  norm_21,   r"$\delta T^\mathrm{H}_\mathrm{b}\ [\mathrm{mK}]$",     (0.54, 0.11, 0.38, 0.03)),
    ]
    
    for cmap, norm, label, bbox in cbar_specs:
        cax = inset_axes(
            ax, width="100%", height="100%", loc="lower left",
            bbox_to_anchor=bbox, bbox_transform=ax.transAxes, borderpad=0
        )
        
        sm = ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cb = fig.colorbar(sm, cax=cax, orientation='horizontal')
        cb.set_label(label, fontsize=12, fontweight='bold', color='#222222', labelpad=6)
        cb.ax.tick_params(labelsize=10, colors='#333333')

    # If running headless or on an SSH server, remember to swap plt.show() 
    # with plt.savefig("output.png", bbox_inches='tight', dpi=300)
    plt.show()

# =====================================================
# EXECUTION BLOCK
# =====================================================
dataset_combined = generate_combined_dataset(
    outer_dir="/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_640",
    inner_dir="/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_below_z_4p75_new/postprocessing/rlc_640",
    SliceFraction=0.1,
    outer_zmin=5.0, outer_zmax=20.0,
    inner_zmin=3.0, inner_zmax=4.75
)

# Explicitly call the plotting function here
plot_double_annulus_hive(dataset_combined, rotation_deg=0, y_offset=0.12)

In [ ]:
# =====================================================
# 3. Main plotting function (Quarter Circle Quadrant Layout)
# =====================================================
def plot_double_annulus_hive(dataset, rotation_deg=0, y_offset=0.0):
    from matplotlib.colors import LogNorm, SymLogNorm
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes

    N        = dataset["N"]
    nz_inner = dataset["nz_inner"]
    nz_outer = dataset["nz_outer"]
    zc_in    = dataset["zc_in"]
    zc_out   = dataset["zc_out"]

    # --- YOUR ORIGINAL EXACT NORMALIZATIONS & QUANTITIES ---
    vmin_21 = np.nanpercentile(dataset["Tb21"], 1)
    vmax_21 = np.nanpercentile(dataset["Tb21"], 99)
    norm_21 = SymLogNorm(linthresh=1.0, linscale=0.5, vmin=vmin_21, vmax=vmax_21, base=10)

    vmin_He3 = np.nanpercentile(dataset["TbHe3"], 1)
    vmax_He3 = np.nanpercentile(dataset["TbHe3"], 99)
    norm_He3 = SymLogNorm(linthresh=0.01, linscale=0.5, vmin=vmin_He3, vmax=vmax_He3, base=10)

    norm_HII  = LogNorm(vmin=1.0e-4, vmax=1.0)
    norm_HeII = LogNorm(vmin=1.0e-5, vmax=1.0)

    deg = np.pi / 180
    rot = rotation_deg * deg

    # Concentric boundaries
    R_core  = 0.08
    R_split = 0.45
    R_outer = 1.0

    # 4 Sectors arranged symmetrically into a single 90-degree quadrant (22.5 deg each)
    # y_offset is set to 0.0 to match your drawing's flush corner origin
    sectors = [
        {
            "label":       r"$\delta T^\mathrm{H}_\mathrm{b} \mathrm{[mK]}$",
            "label_angle": 11.25 * deg + rot,
            "t0": 0 * deg + rot, "t1": 22.5 * deg + rot,
            "cmap": "coolwarm",   "norm": norm_21,   "data": dataset["Tb21"],
            "dy": 0.0, "label_radius_offset": 0.20,
        },
        {
            "label":       r"$x_\mathrm{H\,II}$",
            "label_angle": 33.75 * deg + rot,
            "t0": 22.5 * deg + rot, "t1": 45 * deg + rot,
            "cmap": "viridis",    "norm": norm_HII,  "data": dataset["xHII"],
            "dy": 0.0, "label_radius_offset": 0.20,
        },
        {
            "label":       r"$x_\mathrm{He\,II}$",
            "label_angle": 56.25 * deg + rot,
            "t0": 45 * deg + rot, "t1": 67.5 * deg + rot,
            "cmap": "magma",     "norm": norm_HeII, "data": dataset["xHeII"],
            "dy": 0.0, "label_radius_offset": 0.20,
        },
        {
            "label":       r"$\delta T^\mathrm{He}_\mathrm{b} \mathrm{[\mu K]}$",
            "label_angle": 78.75 * deg + rot,
            "t0": 67.5 * deg + rot, "t1": 90 * deg + rot,
            "cmap": "plasma",    "norm": norm_He3,  "data": dataset["TbHe3"],
            "dy": 0.0, "label_radius_offset": 0.20,
        },
    ]

    fig, ax = plt.subplots(figsize=(9, 9), facecolor='white')
    ax.set_aspect('equal')
    ax.axis('off')
    
    # Zoom axes limits to strictly isolate the upper-right quadrant
    ax.set_xlim(-0.1, 1.35)
    ax.set_ylim(-0.1, 1.35)
    fig.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.05)

    r_inner_edges = np.linspace(R_core,          R_split, nz_inner + 1)
    r_outer_edges = np.linspace(R_split - 0.002, R_outer, nz_outer + 1)

    # Plot data meshes
    for sec in sectors:
        theta = np.linspace(sec["t0"], sec["t1"], N + 1)

        TH_in, RR_in = np.meshgrid(theta, r_inner_edges, indexing='ij')
        ax.pcolormesh(
            RR_in * np.cos(TH_in), RR_in * np.sin(TH_in),
            sec["data"][:, :nz_inner],
            cmap=sec["cmap"], norm=sec["norm"],
            shading='flat', edgecolor='none', rasterized=True
        )

        TH_out, RR_out = np.meshgrid(theta, r_outer_edges, indexing='ij')
        ax.pcolormesh(
            RR_out * np.cos(TH_out), RR_out * np.sin(TH_out),
            sec["data"][:, nz_inner:],
            cmap=sec["cmap"], norm=sec["norm"],
            shading='flat', edgecolor='none', rasterized=True
        )

    # Draw clear border layout lines (0, 22.5, 45, 67.5, and 90 degrees)
    for ang in [0*deg, 22.5*deg, 45*deg, 67.5*deg, 90*deg]:
        ax.plot(
            [R_core * np.cos(ang), R_outer * np.cos(ang)],
            [R_core * np.sin(ang), R_outer * np.sin(ang)],
            color='white', lw=3, zorder=10, solid_capstyle='butt'
        )

    # Center circle clean cut-out anchor
    ax.add_artist(plt.Circle((0, 0), R_core, color='white', zorder=30))

    # Redshift milestones mapped strictly down the horizontal edge (0 deg)
    z_milestones = [3.0, 5.0, 10.0, 20.0]
    r_positions = []
    for zm in z_milestones:
        if zm <= zc_in[-1]:
            idx = np.abs(zc_in - zm).argmin()
            r_positions.append(r_inner_edges[idx])
        else:
            idx = np.abs(zc_out - zm).argmin()
            r_positions.append(r_outer_edges[idx])

    for r, zm in zip(r_positions, z_milestones):
        t_gyr   = age_Gyr(zm)
        label_str = f"$z={zm:.0f}$\n${t_gyr:.1f}\\,\\mathrm{{Gyr}}$"

        ax.text(r, -0.02, label_str,
                fontsize=9, fontweight='bold', color='#222222',
                ha='center', va='top', zorder=51,
                bbox=dict(fc='white', ec='none', pad=1.0))

    # Exterior field text labels
    for sec in sectors:
        custom_r = R_outer + sec["label_radius_offset"]
        custom_ang = sec["label_angle"]
        current_fontsize = 20 if ("x_\\mathrm{H" in sec["label"] or "x_\\mathrm{He" in sec["label"]) else 16

        ax.text(
            custom_r * np.cos(custom_ang),
            custom_r * np.sin(custom_ang),
            sec["label"],
            ha='center', va='center', fontsize=current_fontsize, fontweight='bold', color='#222222'
        )

    # 4 horizontal linear colorbars systematically placed underneath
    cbar_specs = [
        ("coolwarm",  norm_21,      r"$\delta T^\mathrm{H}_\mathrm{b}\ [\mathrm{mK}]$",     (0.05, 0.02, 0.20, 0.025)),
        ("viridis",   norm_HII,     r"$x_\mathrm{H\,II}$",                       (0.28, 0.02, 0.20, 0.025)),
        ("magma",     norm_HeII,    r"$x_\mathrm{He\,II}$",                      (0.51, 0.02, 0.20, 0.025)),
        ("plasma",    norm_He3,     r"$\delta T^\mathrm{He}_\mathrm{b}\ [\mu\mathrm{K}]$", (0.74, 0.02, 0.20, 0.025)),
    ]
    
    for cmap, norm, label, bbox in cbar_specs:
        cax = inset_axes(
            ax, width="100%", height="100%", loc="lower left",
            bbox_to_anchor=bbox, bbox_transform=ax.transAxes, borderpad=0
        )
        
        sm = ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cb = fig.colorbar(sm, cax=cax, orientation='horizontal')
        cb.set_label(label, fontsize=10, fontweight='bold', color='#222222', labelpad=4)
        cb.ax.tick_params(labelsize=8, colors='#333333')

    plt.show()


# =====================================================
# EXECUTION BLOCK
# =====================================================
dataset_combined = generate_combined_dataset(
    outer_dir="/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_640",
    inner_dir="/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_below_z_4p75_new/postprocessing/rlc_640",
    SliceFraction=0.1,
    outer_zmin=5.0, outer_zmax=20.0,
    inner_zmin=3.0, inner_zmax=4.75
)

# Explicitly call the plotting function here
plot_double_annulus_hive(dataset_combined, rotation_deg=0, y_offset=0.12)

In [ ]:
%matplotlib inline
# =====================================================
# 3. Main plotting function (Fixed Redshift Direction)
# =====================================================
def plot_double_annulus_hive(dataset, rotation_deg=0, y_offset=0.0):
    from matplotlib.colors import LogNorm, SymLogNorm
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes

    N        = dataset["N"]
    nz_inner = dataset["nz_inner"]
    nz_outer = dataset["nz_outer"]
    zc_in    = dataset["zc_in"]
    zc_out   = dataset["zc_out"]

    vmin_21 = np.nanpercentile(dataset["Tb21"], 1)
    vmax_21 = np.nanpercentile(dataset["Tb21"], 99)
    norm_21 = SymLogNorm(linthresh=1.0, linscale=0.5, vmin=vmin_21, vmax=vmax_21, base=10)

    vmin_He3 = np.nanpercentile(dataset["TbHe3"], 1)
    vmax_He3 = np.nanpercentile(dataset["TbHe3"], 99)
    norm_He3 = SymLogNorm(linthresh=0.01, linscale=0.5, vmin=vmin_He3, vmax=vmax_He3, base=10)

    norm_HII  = LogNorm(vmin=1.0e-4, vmax=1.0)
    norm_HeII = LogNorm(vmin=1.0e-5, vmax=1.0)

    deg = np.pi / 180
    rot = rotation_deg * deg

    #R_core  = 0.3
    #R_outer = 3.50
    R_core  = 0.5
    R_outer = 8.0
    label_offset = 0.5

    z_min = zc_in.min()
    z_max = zc_out.max()

    def z_to_radius(z_val):
        return R_core + (R_outer - R_core) * (z_val - z_min) / (z_max - z_min)

    z_inner_edges = np.linspace(zc_in.min(), zc_in.max(), nz_inner + 1)
    r_inner_edges = z_to_radius(z_inner_edges)

    z_outer_edges = np.linspace(zc_in.max(), zc_out.max(), nz_outer + 1)
    r_outer_edges = z_to_radius(z_outer_edges)

    sectors = [
        {
            "label":       r"$x_\mathrm{H\,II}$",
            "label_angle": 101.25 * deg + rot,
            "t0": 90 * deg + rot, "t1": 112.5 * deg + rot,
            "cmap": "viridis", "norm": norm_HII, "data": dataset["xHII"],
            "dy": 0.0, "label_radius_offset": label_offset,
        },
        {
            "label":       r"$\delta T^\mathrm{H}_\mathrm{b}$",
            "label_angle": 123.75 * deg + rot,
            "t0": 112.5 * deg + rot, "t1": 135 * deg + rot,
            "cmap": "coolwarm", "norm": norm_21, "data": dataset["Tb21"],
            "dy": 0.0, "label_radius_offset": label_offset,
        },
        {
            "label":       r"$x_\mathrm{He\,II}$",
            "label_angle": 146.25 * deg + rot,
            "t0": 135 * deg + rot, "t1": 157.5 * deg + rot,
            "cmap": "magma", "norm": norm_HeII, "data": dataset["xHeII"],
            "dy": 0.0, "label_radius_offset": label_offset,
        },
        {
            "label":       r"$\delta T^\mathrm{He}_\mathrm{b}$",
            "label_angle": 168.75 * deg + rot,
            "t0": 157.5 * deg + rot, "t1": 180 * deg + rot,
            "cmap": "plasma", "norm": norm_He3, "data": dataset["TbHe3"],
            "dy": 0.0, "label_radius_offset": label_offset,
        },
    ]

    margin = 0.3
    total_r = R_outer + label_offset + margin
    scale   = 1.5

    fig, ax = plt.subplots(figsize=(total_r * scale, total_r * scale), facecolor='white')
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_xlim(-(total_r), margin)
    ax.set_ylim(-margin, total_r)
    fig.subplots_adjust(left=0.0, right=1.0, top=1.0, bottom=0.0)

    for sec in sectors:
        theta = np.linspace(sec["t0"], sec["t1"], N + 1)

        TH_in, RR_in = np.meshgrid(theta, r_inner_edges, indexing='ij')
        ax.pcolormesh(
            RR_in * np.cos(TH_in), RR_in * np.sin(TH_in),
            sec["data"][:, :nz_inner],
            cmap=sec["cmap"], norm=sec["norm"],
            shading='flat', edgecolor='none', rasterized=True
        )

        TH_out, RR_out = np.meshgrid(theta, r_outer_edges, indexing='ij')
        ax.pcolormesh(
            RR_out * np.cos(TH_out), RR_out * np.sin(TH_out),
            sec["data"][:, nz_inner:],
            cmap=sec["cmap"], norm=sec["norm"],
            shading='flat', edgecolor='none', rasterized=True
        )

    for ang in [90*deg, 112.5*deg, 135*deg, 157.5*deg, 180*deg]:
        ax.plot(
            [R_core * np.cos(ang), R_outer * np.cos(ang)],
            [R_core * np.sin(ang), R_outer * np.sin(ang)],
            color='white', lw=3, zorder=10, solid_capstyle='butt'
        )

    ax.add_artist(plt.Circle((0, 0), R_core, color='white', zorder=30))

    z_milestones = [3.0, 5.0, 10.0, 15.0, 20.0]
    for zm in z_milestones:
        r = z_to_radius(zm)
        label_str = f"$z={zm:.0f}$"
        ax.text(-r, -0.08, label_str,
                fontsize=20, fontweight='bold', color='#222222',
                ha='center', va='top', zorder=51,
                bbox=dict(fc='white', ec='none', pad=1.5))

    for sec in sectors:
        custom_r   = R_outer + sec["label_radius_offset"]
        custom_ang = sec["label_angle"]
        current_fontsize = 35 if ("x_\\mathrm{H" in sec["label"] or "x_\\mathrm{He" in sec["label"]) else 22

        ax.text(
            custom_r * np.cos(custom_ang),
            custom_r * np.sin(custom_ang),
            sec["label"],
            ha='center', va='center',
            fontsize=current_fontsize, fontweight='bold', color='#222222'
        )

    cbar_specs = [
        ("viridis",  norm_HII,  r"$x_\mathrm{H\,II}$",                                  (0.05, -0.06, 0.20, 0.02)),
        ("coolwarm", norm_21,   r"$\delta T^\mathrm{H}_\mathrm{b} \ [\mathrm{mK}]$",    (0.28, -0.06, 0.20, 0.02)),
        ("magma",    norm_HeII, r"$x_\mathrm{He\,II}$",                                 (0.51, -0.06, 0.20, 0.02)),
        ("plasma",   norm_He3,  r"$\delta T^\mathrm{He}_\mathrm{b} \ [\mu\mathrm{K}]$",  (0.74, -0.06, 0.20, 0.02)),
    ]
    
    for cmap, norm, label, bbox in cbar_specs:
        cax = inset_axes(
            ax, width="100%", height="100%", loc="upper left",
            bbox_to_anchor=bbox, bbox_transform=ax.transAxes, borderpad=0
        )
        sm = ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cb = fig.colorbar(sm, cax=cax, orientation='horizontal')
        cb.set_label(label, fontsize=20, fontweight='bold', color='#222222', labelpad=6)
        cb.ax.tick_params(labelsize=18, colors='#333333')
    
        if norm == norm_HII:
            cb.set_ticks([1e-4, 1e-3, 1e-2, 1e-1, 1e0])
            cb.set_ticklabels([r"$10^{-4}$", r"$10^{-3}$", r"$10^{-2}$", r"$10^{-1}$", r"$1$"])
        if norm == norm_HeII:
            cb.set_ticks([1e-5, 1e-3, 1e-1, 1e0])
            cb.set_ticklabels([r"$10^{-5}$", r"$10^{-3}$", r"$10^{-1}$", r"$1$"])

    plt.savefig("lightcone.png", dpi=300, bbox_inches='tight', pad_inches=0.1)
    plt.show()
    
# =====================================================
# EXECUTION BLOCK
# =====================================================
dataset_combined = generate_combined_dataset(
    outer_dir="/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_640",
    inner_dir="/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_below_z_4p75_new/postprocessing/rlc_640",
    SliceFraction=0.1,
    outer_zmin=5.0, outer_zmax=20.0,
    inner_zmin=3.0, inner_zmax=4.75
)

# Execution of the revised quadrant tracking plot
plot_double_annulus_hive(dataset_combined, rotation_deg=0, y_offset=0.0)

In [ ]:
def plot_lightcone_rect(dataset):
    from matplotlib.colors import LogNorm, SymLogNorm
    import matplotlib.gridspec as gridspec

    zc    = dataset["zc"]
    Tb21  = dataset["Tb21"]
    TbHe3 = dataset["TbHe3"]
    xHII  = dataset["xHII"]
    xHeII = dataset["xHeII"]
    N     = dataset["N"]

    z_plot = zc

    vmin_21  = np.nanpercentile(Tb21,  1)
    vmax_21  = np.nanpercentile(Tb21,  99)
    norm_21  = SymLogNorm(linthresh=1.0, linscale=0.5, vmin=vmin_21, vmax=vmax_21, base=10)

    vmin_He3 = np.nanpercentile(TbHe3, 1)
    vmax_He3 = np.nanpercentile(TbHe3, 99)
    norm_He3 = SymLogNorm(linthresh=0.01, linscale=0.5, vmin=vmin_He3, vmax=vmax_He3, base=10)

    norm_HII  = LogNorm(vmin=1e-4, vmax=1.0)
    norm_HeII = LogNorm(vmin=1e-5, vmax=1.0)

    nz = len(zc)
    extent = [z_plot.min(), z_plot.max(), 0, N]

    fig = plt.figure(figsize=(24, 16))
    gs = gridspec.GridSpec(
        4, 2,
        width_ratios=[40, 0.6],
        hspace=0.02,
        wspace=0.02
    )

    ax1  = fig.add_subplot(gs[0, 0])
    ax2  = fig.add_subplot(gs[1, 0], sharex=ax1)
    ax3  = fig.add_subplot(gs[2, 0], sharex=ax1)
    ax4  = fig.add_subplot(gs[3, 0], sharex=ax1)
    cax1 = fig.add_subplot(gs[0, 1])
    cax2 = fig.add_subplot(gs[1, 1])
    cax3 = fig.add_subplot(gs[2, 1])
    cax4 = fig.add_subplot(gs[3, 1])

    # Use nearest interpolation — no stretching, missing pixels are fine
    kwargs = dict(origin="lower", aspect="auto",
                  interpolation="nearest", extent=extent)

    im1 = ax1.imshow(xHII,  cmap="viridis",  norm=norm_HII,  **kwargs)
    im2 = ax2.imshow(Tb21,  cmap="coolwarm", norm=norm_21,   **kwargs)
    im3 = ax3.imshow(xHeII, cmap="magma",    norm=norm_HeII, **kwargs)
    im4 = ax4.imshow(TbHe3, cmap="plasma",   norm=norm_He3,  **kwargs)

    cb1 = fig.colorbar(im1, cax=cax1)
    cb1.set_label(r"$x_\mathrm{H\,II}$", fontsize=20)
    cb1.set_ticks([1e-4, 1e-3, 1e-2, 1e-1, 1e0])
    cb1.set_ticklabels([r"$10^{-4}$", r"$10^{-3}$", r"$10^{-2}$", r"$10^{-1}$", r"$1$"])

    cb2 = fig.colorbar(im2, cax=cax2)
    cb2.set_label(r"$\delta T^\mathrm{H}_\mathrm{b}\ [\mathrm{mK}]$", fontsize=20)

    cb3 = fig.colorbar(im3, cax=cax3)
    cb3.set_label(r"$x_\mathrm{He\,II}$", fontsize=20)
    cb3.set_ticks([1e-5, 1e-3, 1e-1, 1e0])
    cb3.set_ticklabels([r"$10^{-5}$", r"$10^{-3}$", r"$10^{-1}$", r"$1$"])

    cb4 = fig.colorbar(im4, cax=cax4)
    cb4.set_label(r"$\delta T^\mathrm{He}_\mathrm{b}\ [\mu\mathrm{K}]$", fontsize=20)

    for cb in [cb1, cb2, cb3, cb4]:
        cb.ax.tick_params(labelsize=14)

    ax1.set_ylabel("Transverse [pixels]", fontsize=18)
    ax2.set_ylabel("Transverse [pixels]", fontsize=18)
    ax3.set_ylabel("Transverse [pixels]", fontsize=18)
    ax4.set_ylabel("Transverse [pixels]", fontsize=18)
    ax4.set_xlabel("$z$", fontsize=22)

    for ax in [ax1, ax2, ax3, ax4]:
        #ax.set_xscale("log")
        ax.tick_params(axis='both', which='major', labelsize=16, length=6, width=2)
        ax.tick_params(axis='both', which='minor', labelsize=12, length=3, width=1)

    # low-z on left, high-z on right
    ax1.set_xlim(z_plot.max(), z_plot.min())

    ax1.tick_params(labelbottom=False)
    ax2.tick_params(labelbottom=False)
    ax3.tick_params(labelbottom=False)

    plt.savefig("lightcone_rect.png", dpi=300, bbox_inches='tight')
    plt.show()

plot_lightcone_rect(dataset_combined)

## Aaron's code

In [ ]:
import numpy as np
import cmasher as cmr
import cmocean
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import StrMethodFormatter, NullFormatter
import matplotlib.patheffects as path_effects
import h5py
%matplotlib inline

"""
labels = {
    'Density': r'${\bf Density}$',
    'Temperature': r'${\bf Temperature}$',
    'HII_Fraction': r'${\bf HII\ Fraction}$',
    'HeII_Fraction': r'${\bf HeII\ Fraction}$',
    'HeIII_Fraction': r'${\bf HeIII\ Fraction}$',
    'LyaLuminosityRec': r'${\bf Ly\alpha\ Emission}$',
    'VelocitySquared': r'${\bf Velocity}$'
}
cmaps = {
    'Density': cmocean.cm.thermal,
    'Temperature': 'inferno',
    'HII_Fraction': cmr.amber,
    'HeII_Fraction': 'twilight_shifted',
    'HeIII_Fraction': cmr.fusion,
    'LyaLuminosityRec': cmr.rainforest,
    'VelocitySquared': cmr.ember
}
"""
from matplotlib.colors import SymLogNorm

labels = {
    'HII_Fraction': r"$x_\mathrm{H\,II}$",
    'Tb21':         r"$\delta T^\mathrm{H}_\mathrm{b}\ [\mathrm{mK}]$",
    'HeII_Fraction':r"$x_\mathrm{He\,II}$",
    'TbHe3':        r"$\delta T^\mathrm{He}_\mathrm{b}\ [\mu\mathrm{K}]$",
}
cmaps = {
    'HII_Fraction': 'viridis',
    'Tb21':         'coolwarm',
    'HeII_Fraction':'magma',
    'TbHe3':        'plasma',
}

HYDROGEN_MASSFRAC = 0.76
HE_ABUND = (1./HYDROGEN_MASSFRAC - 1.) / 4.
norms = {
    'Density': LogNorm(),
    'Temperature': LogNorm(vmin=100., vmax=1e6, clip=True),
    'HII_Fraction': Normalize(vmin=0., vmax=1., clip=True),
    'HeII_Fraction': Normalize(vmin=0., vmax=HE_ABUND, clip=True),
    'HeIII_Fraction': Normalize(vmin=0., vmax=HE_ABUND, clip=True),
    'LyaLuminosityRec': LogNorm(),
    'VelocitySquared': LogNorm()
}
grey = [0.1, 0.1, 0.1]

def log_minorticks(vmin, vmax):
    expA = int(np.floor(np.log10(vmin)))
    expB = int(np.floor(np.log10(vmax)))
    cofA = int(np.ceil(vmin/10**expA))
    cofB = int(np.floor(vmax/10**expB))
    lmt = []
    while cofA*10**expA <= cofB*10**expB:
        if expA < expB:
            lmt = np.hstack( (lmt, np.linspace(cofA, 9, 10-cofA)*10**expA) )
            cofA = 1
            expA += 1
        else:
            lmt = np.hstack( (lmt, np.linspace(cofA, cofB, cofB-cofA+1)*10**expA) )
            expA += 1
    return lmt

def colorbar(image, label, position='right', scale=1., fontsize=11, labelsize=10.5, ticklabels=None, extend='neither', ax=None):
    vmin, vmax = image.get_clim()
    Lvmin, Lvmax = int(np.ceil(np.log10(vmin))), int(np.floor(np.log10(vmax)))
    Lcticks = np.linspace(Lvmin, Lvmax, Lvmax-Lvmin+1)
    minorticks = log_minorticks(vmin, vmax)
    if position == 'right':
        if ax is None:
            cax = image.figure.add_axes([1, 0, .03*scale, 1])
        else:
            pos = ax.get_position()
            cax = image.figure.add_axes([pos.x1 + 0.02*pos.height, pos.y0 + 0.225*pos.height, 0.02*scale, pos.height*0.55])
        cbar = image.figure.colorbar(image, cax=cax, ticks=10**Lcticks, orientation='vertical', extend=extend)
        cbar.ax.yaxis.set_ticks(minorticks, minor=True)
        cax.yaxis.set_minor_formatter(NullFormatter())
    elif position == 'left':
        cax = image.figure.add_axes([-.03*scale, 0, .03*scale, 1])
        cbar = image.figure.colorbar(image, cax=cax, ticks=10**Lcticks, orientation='vertical', extend=extend)
        cbar.ax.yaxis.set_ticks(minorticks, minor=True)
        cax.yaxis.set_minor_formatter(NullFormatter())
        cbar.ax.yaxis.set_ticks_position('left')
        cbar.ax.yaxis.set_label_position('left')
        if ticklabels is not None:
            cbar.ax.set_yticklabels(ticklabels)
    elif position == 'top':
        cax = image.figure.add_axes([0, 1, 1, .04*scale])
        cbar = image.figure.colorbar(image, cax=cax, ticks=10**Lcticks, orientation='horizontal', extend=extend)
        cbar.ax.xaxis.set_ticks(minorticks, minor=True)
        cax.xaxis.set_minor_formatter(NullFormatter())
        cbar.ax.xaxis.set_ticks_position('top')
        cbar.ax.xaxis.set_label_position('top')
    elif position == 'bottom':
        cax = image.figure.add_axes([0, -.04*scale, 1, .04*scale])
        cbar = image.figure.colorbar(image, cax=cax, ticks=10**Lcticks, orientation='horizontal', extend=extend)
        cbar.ax.xaxis.set_ticks(minorticks, minor=True)
        cax.xaxis.set_minor_formatter(NullFormatter())
    else:
        raise ValueError('position not recognized!')
    cax.tick_params(which='both', direction='in', labelsize=labelsize)
    if fontsize is not None:
        cbar.set_label(label, fontsize=fontsize, labelpad=7)
    else:
        cbar.set_label(label)

def plot_rlc(SliceFraction=0.2, RedshiftAxis=True, AgeAxis=True):
    outer_rlc = "/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_320"
    inner_rlc = "/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_below_z_4p75_new/postprocessing/rlc_320"

    vec_fields  = ['IonFlux', 'Velocities']
    band_fields = ['IonEnergies']

    fields = ['Density', 'Temperature', 'HII_Fraction', 'HeII_Fraction',
              'HeIII_Fraction', 'LyaLuminosityRec', 'LyaLuminosityCol', 'VelocitySquared']

    with h5py.File(f'{outer_rlc}/All.hdf5', 'r') as f:
        header        = f['Header']
        H0            = 100. * header.attrs['HubbleParam']
        Omega0        = header.attrs['Omega0']
        OpeningAngle  = header.attrs['OpeningAngle']
        HalfAngle     = OpeningAngle / 2.
        NumPixels     = header.attrs['NumPixels']
        SliceIndex    = max(0, min(NumPixels - 1, int(SliceFraction * NumPixels)))
        Redshifts_out = f['Redshifts'][:]
        Distances_out = f['Distances'][:]
        ds_out        = {field: f[field][SliceIndex,:,:] for field in fields}

    ds_out['LyaLuminosityRec'] = ds_out['LyaLuminosityRec'] + ds_out['LyaLuminosityCol']
    ds_out.pop('LyaLuminosityCol')
    ds_out['VelocitySquared']  = np.sqrt(ds_out['VelocitySquared'])

    with h5py.File(f'{inner_rlc}/All.hdf5', 'r') as f:
        Redshifts_in = f['Redshifts'][:]
        Distances_in = f['Distances'][:]
        ds_in        = {field: f[field][SliceIndex,:,:] for field in fields}

    ds_in['LyaLuminosityRec'] = ds_in['LyaLuminosityRec'] + ds_in['LyaLuminosityCol']
    ds_in.pop('LyaLuminosityCol')
    ds_in['VelocitySquared']  = np.sqrt(ds_in['VelocitySquared'])

    fields.remove('LyaLuminosityCol')
    fields = fields[::-1]
    print(f'Fields: {fields}')

    # Restrict outer to z <= 20
    zmax = 20.
    imax = np.argmin(np.abs(Redshifts_out - zmax))
    Redshifts_out = Redshifts_out[imax:]
    Distances_out = Distances_out[imax:]
    for field in ds_out:
        ds_out[field] = ds_out[field][:, imax:]

    # Concatenate outer (high-z) + inner (low-z)
    Redshifts = np.concatenate([Redshifts_out, Redshifts_in])
    Distances = np.concatenate([Distances_out, Distances_in])
    ds = {field: np.concatenate([ds_out[field], ds_in[field]], axis=1) for field in ds_out}

    # Trim Distances to match data shape + 1
    nz = ds[fields[0]].shape[1]
    Distances = Distances[:nz + 1]
    Redshifts = Redshifts[:nz + 1]

    import pandas as pd
    from scipy.interpolate import interp1d, CubicSpline
    import astropy.constants as const

    # ============================================================
    # CONSTANTS
    # ============================================================
    PLANCK       = 6.62607015e-27
    k_B          = 1.381e-16
    C_LIGHT      = const.c.cgs.value
    LAMBDA_21cm  = 21.106
    Tcmb0        = 2.7255
    OMEGA_b      = 0.0486
    OMEGA_m      = 0.3089
    h            = 0.6774

    # ============================================================
    # COLLISIONAL RATE SPLINES
    # ============================================================
    _T_HH = np.log10([1, 2, 4, 6, 8, 10, 15, 20, 25, 30, 40, 50, 60, 70, 80, 90, 100, 200, 300, 500, 700, 1000, 2000, 3000, 5000, 7000, 10000])
    _k_HH = np.log10([1.38e-13, 1.43e-13, 2.71e-13, 6.60e-13, 1.47e-12, 2.88e-12, 9.10e-12, 1.78e-11, 2.73e-11, 3.67e-11, 5.38e-11, 6.86e-11, 8.14e-11, 9.25e-11, 1.02e-10, 1.11e-10, 1.19e-10, 1.75e-10, 2.09e-10, 2.56e-10, 2.91e-10, 3.31e-10, 4.27e-10, 4.97e-10, 6.03e-10, 6.87e-10, 7.87e-10])
    _T_eH = np.log10([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000, 20000])
    _k_eH = np.log10([2.39e-10, 3.37e-10, 5.30e-10, 7.46e-10, 1.05e-9, 1.63e-9, 2.26e-9, 3.11e-9, 4.59e-9, 5.92e-9, 7.15e-9, 7.71e-9, 8.17e-9, 8.32e-9, 8.37e-9, 8.29e-9, 8.11e-9])
    _T_pH = _T_eH.copy()
    _k_pH = np.log10([0.40e-9, 0.45e-9, 0.430e-9, 0.369e-9, 0.317e-9, 0.3047e-9, 0.3379e-9, 0.4043e-9, 0.5471e-9, 0.7051e-9, 0.9167e-9, 1.070e-9, 1.301e-9, 1.480e-9, 1.695e-9, 1.975e-9, 2.201e-9])
    _spline_HH = CubicSpline(_T_HH, _k_HH, extrapolate=True)
    _spline_eH = CubicSpline(_T_eH, _k_eH, extrapolate=True)
    _spline_pH = CubicSpline(_T_pH, _k_pH, extrapolate=True)

    def get_kappa_2D(T_K):
        logT = np.log10(np.clip(T_K, 1.0, 20000.0))
        k_HH = 10.0 ** _spline_HH(logT)
        k_eH = 10.0 ** _spline_eH(logT)
        k_pH = 10.0 ** _spline_pH(logT)
        return k_HH, k_eH, k_pH

    # ============================================================
    # EXTRACT FIELDS FROM ds
    # ============================================================
    T_all    = ds['Temperature']
    xHII_all = ds['HII_Fraction']
    xHeII_all= ds['HeII_Fraction'] * 12.666667
    dens_all = ds['Density']

    # Cell-center redshifts
    zc_all = 0.5 * (Redshifts[:-1] + Redshifts[1:])
    zc_all = zc_all[:T_all.shape[1]]
    z2d    = np.tile(zc_all, (NumPixels, 1))

    xHI_all    = 1.0 - xHII_all
    xHeIII_all = 1.0 - xHeII_all

    # ============================================================
    # NUMBER DENSITIES
    # ============================================================
    HYDROGEN_MASSFRAC = 0.76
    m_p               = 1.6738e-24

    rho_H_comoving = HYDROGEN_MASSFRAC * dens_all * 3.0991811197281837e-22
    scale_z3       = (1.0 + z2d) ** 3.0
    rho_H          = rho_H_comoving * scale_z3

    num_H     = rho_H / m_p
    num_HII   = xHII_all  * num_H
    num_HI    = xHI_all   * num_H
    num_HeII  = xHeII_all * num_H
    num_HeIII = xHeIII_all* num_H
    num_p     = num_HII
    num_e     = num_HII + num_HeII + 2.0 * num_HeIII

    mean_density = np.mean(dens_all) if np.mean(dens_all) > 0 else 1.0
    delta_all    = dens_all / mean_density

    # ============================================================
    # BACKGROUND FIELDS
    # ============================================================
    Tcmb     = Tcmb0 * (1.0 + z2d)
    e_cgs    = 4.803204712570263e-10
    me_cgs   = 9.1093837e-28
    f_a      = 0.4162

    T_star    = PLANCK * C_LIGHT / (k_B * LAMBDA_21cm)
    A_10      = 2.85e-15
    A_10_He   = 1.95e-12
    LAMBDA_He = 3.46
    T_star_He = PLANCK * C_LIGHT / (k_B * LAMBDA_He)

    # ============================================================
    # J_ALPHA FROM CSV
    # ============================================================
    df_global    = pd.read_csv("/orcd/data/mvogelsb/004/chcho/LUMINA_all.csv")
    df_global    = df_global.sort_values("redshift")
    interp_Ja_H  = interp1d(df_global["redshift"], df_global["J_alpha"],    kind='linear', bounds_error=False, fill_value="extrapolate")
    interp_Ja_He = interp1d(df_global["redshift"], df_global["J_alpha_He"], kind='linear', bounds_error=False, fill_value="extrapolate")

    J_alpha_all    = interp_Ja_H(z2d)  * delta_all
    J_alpha_He_all = interp_Ja_He(z2d) * delta_all

    # ============================================================
    # COUPLING COEFFICIENTS
    # ============================================================
    const_H    = (16.0 * np.pi**2 * T_star    * e_cgs**2 * f_a) / (27.0 * A_10    * Tcmb * me_cgs * C_LIGHT)
    x_a_H_all  = const_H * J_alpha_all

    const_He   = (16.0 * np.pi**2 * T_star_He * e_cgs**2 * f_a) / (9.0  * A_10_He * Tcmb * me_cgs * C_LIGHT)
    x_a_He_all = const_He * J_alpha_He_all

    k_HH, k_eH, k_pH = get_kappa_2D(T_all)
    x_c_H_all  = T_star    * (k_HH * num_HI + k_eH * num_e + k_pH * num_p) / (A_10    * Tcmb)

    a_0           = 5.29177e-9
    bar_sigma_all = (2.291112e-11 / (k_B * T_all)) * (a_0 ** 2)
    SQRT_all      = np.sqrt((8.0 * k_B * T_all) / (np.pi * me_cgs * C_LIGHT**2))
    rate_He_all   = num_e * SQRT_all * C_LIGHT * bar_sigma_all
    x_c_He_all    = (T_star_He * rate_He_all) / (A_10_He * Tcmb)

    # ============================================================
    # SPIN TEMPERATURES
    # ============================================================
    inv_Tk   = 1.0 / np.maximum(T_all, 1e-3)
    inv_Tcmb = 1.0 / Tcmb

    Ts_H_all  = (1.0 + x_a_H_all  + x_c_H_all)  / (inv_Tcmb + x_a_H_all  * inv_Tk + x_c_H_all  * inv_Tk)
    Ts_He_all = (1.0 + x_a_He_all + x_c_He_all) / (inv_Tcmb + x_a_He_all * inv_Tk + x_c_He_all * inv_Tk)

    # ============================================================
    # BRIGHTNESS TEMPERATURES
    # ============================================================
    cosmo_prefactor = ((OMEGA_b * h**2) / 0.023) * np.sqrt((0.15 * (1.0 + z2d)) / (OMEGA_m * h**2 * 10.0))

    coupling_21 = 1.0 - Tcmb / Ts_H_all
    coupling_He = 1.0 - Tcmb / Ts_He_all

    Tb21_all  = 27.0 * xHI_all    * delta_all * cosmo_prefactor * coupling_21
    TbHe3_all = 1.67 * xHeII_all  * delta_all * cosmo_prefactor * coupling_He

    # Add to ds for plotting
    ds['Tb21']  = Tb21_all
    ds['TbHe3'] = TbHe3_all

    ds['Tb21']  = Tb21_all
    ds['TbHe3'] = TbHe3_all

    vmin_21,  vmax_21  = np.nanpercentile(Tb21_all,  [1, 99])
    vmin_He3, vmax_He3 = np.nanpercentile(TbHe3_all, [1, 99])

    norms = {
        'HII_Fraction': LogNorm(vmin=1e-4, vmax=1.0, clip=True),
        'Tb21':         SymLogNorm(linthresh=1.0,  linscale=0.5, vmin=vmin_21,  vmax=vmax_21,  base=10),
        'HeII_Fraction':LogNorm(vmin=1e-5, vmax=1.0, clip=True),
        'TbHe3':        SymLogNorm(linthresh=0.01, linscale=0.5, vmin=vmin_He3, vmax=vmax_He3, base=10),
    }

    fields = ['HII_Fraction', 'Tb21', 'HeII_Fraction', 'TbHe3']

    #for field in ['Density', 'Temperature', 'LyaLuminosityRec', 'VelocitySquared']:
    #    vlims = np.percentile(ds[field], [1, 99.9])
    #    norms[field] = LogNorm(vmin=vlims[0], vmax=vlims[1], clip=True)

    # Plot
    n_fields = len(fields)
    MaxAngle, RefAngle = float(n_fields) * HalfAngle, np.pi/2.
    #MaxAngle, RefAngle = float(n_fields) * HalfAngle, np.pi/2. + np.pi/4.
    Theta_E = np.linspace(RefAngle-MaxAngle, RefAngle+MaxAngle, n_fields + 1)
    Theta_C = (Theta_E[1:] + Theta_E[:-1]) / 2.
    Dmin, Dmax, Davg = Distances[-1], Distances[0], 0.5 * (Distances[-1] + Distances[0])
    Xcorners = [Dmin*np.cos(Theta_E[0]), Dmax*np.cos(Theta_E[0]), Dmax*np.cos(Theta_E[-1]), Dmin*np.cos(Theta_E[-1])]
    Ycorners = [Dmin*np.sin(Theta_E[0]), Dmax*np.sin(Theta_E[0]), Dmax*np.sin(Theta_E[-1]), Dmin*np.sin(Theta_E[-1])]
    if Theta_E[0] < 0. < Theta_E[-1]: Xcorners.append(Dmax)
    if Theta_E[0] < np.pi/2. < Theta_E[-1]: Ycorners.append(Dmax)
    if Theta_E[0] < np.pi < Theta_E[-1]: Xcorners.append(-Dmax)
    if Theta_E[0] < 3.*np.pi/2. < Theta_E[-1]: Ycorners.append(-Dmax)
    if RedshiftAxis: Xcorners.append(Dmax*np.cos(Theta_E[-1]+0.002))
    if AgeAxis: Xcorners.append(Dmax*np.cos(Theta_E[0]-0.002))
    Xlims = [np.min(Xcorners), np.max(Xcorners)]
    Ylims = [np.min(Ycorners), np.max(Ycorners)]
    dXlims = Xlims[1] - Xlims[0]
    dYlims = Ylims[1] - Ylims[0]
    Buffer = 0.0025
    Xlims = [Xlims[0] - Buffer*dXlims, Xlims[1] + Buffer*dXlims]
    Ylims = [Ylims[0] - Buffer*dYlims, Ylims[1] + Buffer*dYlims]

    fig = plt.figure(figsize=(12,12))
    ax  = plt.axes([0,0,1,1])

    for i, field in enumerate(fields):
        Thetas = np.linspace(Theta_E[i], Theta_E[i+1], NumPixels+1)
        Theta2d, R2d = np.meshgrid(Thetas, Distances, indexing='ij')
        X = R2d * np.cos(Theta2d)
        Y = R2d * np.sin(Theta2d)
        print(f'{field}: {np.percentile(ds[field], [1, 99])}')
        image = ax.pcolormesh(X, Y, ds[field], cmap=cmaps[field], norm=norms[field], rasterized=True, shading='auto')
        #image = ax.pcolormesh(X, Y, ds[field], cmap=cmaps[field], norm=norms[field], rasterized=True) #

    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(Xlims)
    ax.set_ylim(Ylims)
    ax.axis('off')

    for i in range(n_fields + 1):
        ax.plot([Dmin*np.cos(Theta_E[i]), Dmax*np.cos(Theta_E[i])],
                [Dmin*np.sin(Theta_E[i]), Dmax*np.sin(Theta_E[i])], c=grey, lw=1)

    theta = np.linspace(Theta_E[0], Theta_E[-1], 200)
    ax.plot(Dmin*np.cos(theta), Dmin*np.sin(theta), c=grey, lw=1)
    ax.plot(Dmax*np.cos(theta), Dmax*np.sin(theta), c=grey, lw=1)

    Dlab = Dmin - 0.02 * (Dmax - Dmin)
    for i, field in enumerate(fields):
        ax.text(Dlab*np.cos(Theta_C[i]), Dlab*np.sin(Theta_C[i]),
                labels[field], ha='center', va='center', c=grey,
                fontsize=10, rotation=np.degrees(Theta_C[i]))

    if RedshiftAxis:
        for i in range(3, 21):
            iz = np.argmin(np.abs(Redshifts - float(i)))
            df = Distances[-1] / Distances[iz]
            cos_1, cos_2, cos_t = np.cos(Theta_E[-1]), np.cos(Theta_E[-1]+0.002*df), np.cos(Theta_E[-1]+0.006*df)
            sin_1, sin_2, sin_t = np.sin(Theta_E[-1]), np.sin(Theta_E[-1]+0.002*df), np.sin(Theta_E[-1]+0.006*df)
            ax.plot([Distances[iz]*cos_1, Distances[iz]*cos_2],
                    [Distances[iz]*sin_1, Distances[iz]*sin_2], c=grey, lw=1.)
            if i not in [12, 14, 16, 17, 19, 20]:
                ax.text(Distances[iz]*cos_t, Distances[iz]*sin_t,
                        r'${\bf '+str(i)+r'}$', ha='center', va='center',
                        c=grey, fontsize=8, rotation=np.degrees(Theta_E[-1]+0.006*df))
        ax.text(Davg*np.cos(Theta_E[-1]+0.014), Davg*np.sin(Theta_E[-1]+0.014),
                r'${\bf Cosmological\ Redshift}$', ha='center', va='center',
                c=grey, fontsize=10, rotation=np.degrees(Theta_E[-1]+0.014))

    if AgeAxis:
        from astropy.cosmology import FlatLambdaCDM, z_at_value
        from astropy import units as u
        cosmo = FlatLambdaCDM(H0=H0, Om0=Omega0, Tcmb0=2.725)
        t_min = 1e3 * cosmo.age(Redshifts[0]).value
        t_max = 1e3 * cosmo.age(Redshifts[-1]).value
        print(f'Dmin: {Dmin/1e6:g} cMpc/h, Dmax: {Dmax/1e6:g} cMpc/h')
        print(f't_min: {t_min:g} Myr, t_max: {t_max:g} Myr')
        Ages = [200., 300., 400., 500., 600., 700., 800., 900., 1000., 1100., 1200., 1500, 2100]
        for age in Ages:
            z  = z_at_value(cosmo.age, age * u.Myr)
            iz = np.argmin(np.abs(Redshifts - z))
            df = Distances[-1] / Distances[iz]
            cos_1, cos_2, cos_t = np.cos(Theta_E[0]), np.cos(Theta_E[0]-0.002*df), np.cos(Theta_E[0]-0.007*df)
            sin_1, sin_2, sin_t = np.sin(Theta_E[0]), np.sin(Theta_E[0]-0.002*df), np.sin(Theta_E[0]-0.007*df)
            ax.plot([Distances[iz]*cos_1, Distances[iz]*cos_2],
                    [Distances[iz]*sin_1, Distances[iz]*sin_2], c=grey, lw=1.)
            if age not in [1100.]:
                ax.text(Distances[iz]*cos_t, Distances[iz]*sin_t,
                        r'${\bf '+str(int(age))+r'}$', ha='center', va='center',
                        c=grey, fontsize=8, rotation=np.degrees(Theta_E[0]-0.007*df))
        ax.text(Davg*np.cos(Theta_E[0]-0.016), Davg*np.sin(Theta_E[0]-0.016),
                r'${\bf Time\ Since\ Big\ Bang\ \,(Myr)}$', ha='center', va='center',
                c=grey, fontsize=10, rotation=np.degrees(Theta_E[0]-0.016))


#    plt.savefig('rlc_fan.png', bbox_inches='tight', dpi=200, pad_inches=0.025)
    plt.show()

plot_rlc(SliceFraction=0.1)

---

In [ ]:
"""
"N": N,
        "nz_inner": T_in.shape[1],
        "nz_outer": T_out.shape[1],
        "zc_in": zc_in,
        "zc_out": zc_out,
        "zc": zc_all,
        "T": T_all,
        "xHII": xHII_all,
        "xHeII": xHeII_all,
        "xHeIII": xHeIII_all,
        "Tb21": np.abs(Tb21_all),
        "TbHe3": np.abs(TbHe3_all),
    }
"""

plt.plot(dataset_combined["zc"], dataset_combined["xHII"][160,:])
#plt.xlim(3.8,3.7)

In [ ]:
plt.hist(dataset_combined["xHII"][160, :], bins=50, color='steelblue', edgecolor='none')
plt.xlabel(r"$x_\mathrm{H\,II}$")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(dataset_combined["xHII"][160, :], bins=50, 
        color='steelblue', edgecolor='none', density=True)
ax.set_xlabel(r"$x_\mathrm{H\,II}$")
ax.set_ylabel("PDF")
ax.set_xlim(0, 1)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

---

---

## All Tests Dummy

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.cm import ScalarMappable
import cmasher as cmr
import h5py

Tcmb0 = 2.725


def plot_hive(rlc_dir, SliceFraction=0.2, zmax=20.0, rotation_deg=0):

    # =====================================================
    # Load data
    # =====================================================
    with h5py.File(f"{rlc_dir}/All.hdf5", "r") as f:
        N     = f["Header"].attrs["NumPixels"]
        i     = int(SliceFraction * N)
        T     = f["Temperature"][i]
        xHII  = f["HII_Fraction"][i]
        xHeII = f["HeII_Fraction"][i] * 12.666667
        z     = f["Redshifts"][:]
        D     = f["Distances"][:]

    # =====================================================
    # Restrict redshift
    # =====================================================
    imax = np.argmin(np.abs(z - zmax))
    z, D = z[imax:], D[imax:]
    T, xHII, xHeII = T[:, imax:], xHII[:, imax:], xHeII[:, imax:]

    zc = 0.5 * (z[:-1] + z[1:])
    nz = T.shape[1]
    zc = zc[:nz]

    z2d = np.tile(zc, (N, 1))

    # =====================================================
    # Physics
    # =====================================================
    xHI    = 1.0 - xHII
    xHeIII = 1.0 - xHeII
    delta  = xHI / np.mean(xHI[xHI > 0])

    Tcmb     = Tcmb0 * (1 + z2d)
    Ts       = np.maximum(T, Tcmb + 1e-5)
    coupling = 1.0 - Tcmb / Ts

    Tb21  = 27.0 * xHI   * delta * np.sqrt((1 + z2d) / 10) * coupling
    TbHe3 = 0.03 * xHeII * delta * np.sqrt((1 + z2d) / 10) * coupling

    # =====================================================
    # Colormaps / norms
    # =====================================================
    norm_21    = LogNorm(vmin=1e-2, vmax=np.percentile(np.abs(Tb21),  99))
    norm_He3   = LogNorm(vmin=1e-5, vmax=np.percentile(np.abs(TbHe3), 99))
    norm_HII   = Normalize(0, 1)
    norm_HeIII = Normalize(0, 1)

    # =====================================================
    # Geometry: 4 equal 90° sectors
    # Spokes at 45°, 135°, 225°, 315° (diagonals)
    # Upper-left  (90°–180°):  HII
    # Upper-right (0°–90°):    21cm
    # Lower-left  (180°–270°): HeIII
    # Lower-right (270°–360°): 3He
    # =====================================================
    deg    = np.pi / 180
    rot    = rotation_deg * deg
    two_pi = 2 * np.pi

    # Spokes on axes (0°, 90°, 180°, 270°) — aligned with sector boundaries
    axis_angles = np.array([0, 90, 180, 270]) * deg + rot

    sectors = [
        {
            "data":        np.abs(Tb21),
            "cmap":        cmr.fusion,
            "norm":        norm_21,
            "t0":          0   * deg + rot,
            "t1":          90  * deg + rot,
            "label":       "21cm $T_b$",
            "label_angle": 45  * deg + rot,
        },
        {
            "data":        xHII,
            "cmap":        cmr.amber,
            "norm":        norm_HII,
            "t0":          90  * deg + rot,
            "t1":          180 * deg + rot,
            "label":       "HII",
            "label_angle": 135 * deg + rot,
        },
        {
            "data":        xHeIII,
            "cmap":        "magma",
            "norm":        norm_HeIII,
            "t0":          180 * deg + rot,
            "t1":          270 * deg + rot,
            "label":       "HeIII",
            "label_angle": 225 * deg + rot,
        },
        {
            "data":        np.abs(TbHe3),
            "cmap":        "coolwarm",
            "norm":        norm_He3,
            "t0":          270 * deg + rot,
            "t1":          360 * deg + rot,
            "label":       "$^3$He $T_b$",
            "label_angle": 315 * deg + rot,
        },
    ]

    R_outer = 1.0
    r_edges = np.linspace(0, R_outer, nz + 1)

    # =====================================================
    # Plot
    # =====================================================
    fig, ax = plt.subplots(figsize=(9, 9), facecolor='white')
    ax.set_aspect('equal')
    ax.axis('off')

    ax.set_xlim(-1.25, 1.25)
    ax.set_ylim(-1.25, 1.25)

    # -- Sectors --
    for sec in sectors:
        data = sec["data"]
        t0   = sec["t0"]
        t1   = sec["t1"]

        theta = np.linspace(t0, t1, N + 1)          # (N+1,)

        TH, RR = np.meshgrid(theta, r_edges, indexing='ij')   # (N+1, nz+1)
        X = RR * np.cos(TH)
        Y = RR * np.sin(TH)

        C = data[:, ::-1]                            # flip: high-z outer

        ax.pcolormesh(X, Y, C,
                      cmap=sec["cmap"],
                      norm=sec["norm"],
                      shading='auto',
                      rasterized=True)

    # -- White spoke dividers --
    for ang in axis_angles:
        ax.plot([0, R_outer * np.cos(ang)],
                [0, R_outer * np.sin(ang)],
                color='white', lw=6, zorder=5,
                solid_capstyle='butt')

    # -- Outer circle --
    th = np.linspace(0, two_pi, 1000)
    ax.plot(R_outer * np.cos(th), R_outer * np.sin(th),
            color='black', lw=1.2, zorder=6)

    # -- Labels outside circle --
    label_r = R_outer + 0.12
    for sec in sectors:
        ang = sec["label_angle"]
        ax.text(label_r * np.cos(ang), label_r * np.sin(ang),
                sec["label"],
                ha='center', va='center',
                fontsize=13, fontweight='bold', color='#222222')

    # -- Redshift ticks on each spoke --
    z_min, z_max_d = zc.min(), zc.max()

    def z_to_r(zval):
        return (zval - z_min) / (z_max_d - z_min) * R_outer

    tick_zvals = np.linspace(z_min, z_max_d, 5)

    for ang in axis_angles:
        for tz in tick_zvals:
            tr       = z_to_r(tz)
            tick_len = 0.025
            perp     = ang + np.pi / 2
            tx, ty   = tr * np.cos(ang), tr * np.sin(ang)
            ax.plot([tx - tick_len * np.cos(perp),
                     tx + tick_len * np.cos(perp)],
                    [ty - tick_len * np.sin(perp),
                     ty + tick_len * np.sin(perp)],
                    color='white', lw=1.5, zorder=7)

    # z labels on right spoke (0°) only
    ang0 = 0 * deg + rot
    for tz in tick_zvals:
        tr = z_to_r(tz)
        ax.text(tr * np.cos(ang0) + 0.04,
                tr * np.sin(ang0) + 0.02,
                f"z={tz:.1f}", fontsize=7, color='#666666', zorder=8)

    # =====================================================
    # Colorbars
    # =====================================================
    cbar_specs = [
        (cmr.fusion, norm_21,    "21cm $T_b$ (mK)"),
        ("coolwarm", norm_He3,   "$^3$He $T_b$ (mK)"),
        (cmr.amber,  norm_HII,   "HII fraction"),
        ("magma",    norm_HeIII, "HeIII fraction"),
    ]
    for k, (cmap, norm, label) in enumerate(cbar_specs):
        cax = fig.add_axes([0.05 + k * 0.23, 0.02, 0.18, 0.018])
        sm  = ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cb  = fig.colorbar(sm, cax=cax, orientation='horizontal')
        cb.set_label(label, fontsize=8, color='#555555')
        cb.ax.tick_params(labelsize=7, colors='#555555')

    plt.tight_layout(rect=[0, 0.08, 1, 1])
    plt.savefig("hive_plot_circle4.png", dpi=300, bbox_inches='tight',
                facecolor='white')
    plt.show()


# =====================================================
# RUN
# =====================================================
plot_hive(
    "/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_640",
    SliceFraction=0.1,
    zmax=20.0,
    rotation_deg=0,
)

In [ ]:
import numpy as np
import cmasher as cmr
import cmocean
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import StrMethodFormatter, NullFormatter
import matplotlib.patheffects as path_effects
import h5py

labels = {
    'Density': r'${\bf Density}$',
    'Temperature': r'${\bf Temperature}$',
    'HII_Fraction': r'${\bf HII\ Fraction}$',
    'HeII_Fraction': r'${\bf HeII\ Fraction}$',
    'HeIII_Fraction': r'${\bf HeIII\ Fraction}$',
    'LyaLuminosityRec': r'${\bf Ly\alpha\ Emission}$',
    'VelocitySquared': r'${\bf Velocity}$'
}
cmaps = {
    'Density': cmocean.cm.thermal,
    'Temperature': 'inferno',
    'HII_Fraction': cmr.amber,
    'HeII_Fraction': 'twilight_shifted',
    'HeIII_Fraction': cmr.fusion,
    'LyaLuminosityRec': cmr.rainforest,
    'VelocitySquared': cmr.ember
}
HYDROGEN_MASSFRAC = 0.76      # Mass fraction of hydrogen
HE_ABUND = (1./HYDROGEN_MASSFRAC - 1.) / 4. # Helium abundance = n_He / n_H
norms = {
    'Density': LogNorm(),
    'Temperature': LogNorm(vmin=100., vmax=1e6, clip=True),
    'HII_Fraction': Normalize(vmin=0., vmax=1., clip=True),
    'HeII_Fraction': Normalize(vmin=0., vmax=HE_ABUND, clip=True),
    'HeIII_Fraction': Normalize(vmin=0., vmax=HE_ABUND, clip=True),
    'LyaLuminosityRec': LogNorm(),
    'VelocitySquared': LogNorm()
}
grey = [0.1, 0.1, 0.1]

def log_minorticks(vmin, vmax):
    expA = int(np.floor(np.log10(vmin)))
    expB = int(np.floor(np.log10(vmax)))
    cofA = int(np.ceil(vmin/10**expA))
    cofB = int(np.floor(vmax/10**expB))
    lmt = []
    while cofA*10**expA <= cofB*10**expB:
        if expA < expB:
            lmt = np.hstack( (lmt, np.linspace(cofA, 9, 10-cofA)*10**expA) )
            cofA = 1
            expA += 1
        else:
            lmt = np.hstack( (lmt, np.linspace(cofA, cofB, cofB-cofA+1)*10**expA) )
            expA += 1
    return lmt

def colorbar(image, label, position='right', scale=1., fontsize=11, labelsize=10.5, ticklabels=None, extend='neither', ax=None):
    vmin, vmax = image.get_clim()
    Lvmin, Lvmax = int(np.ceil(np.log10(vmin))), int(np.floor(np.log10(vmax)))
    Lcticks = np.linspace(Lvmin, Lvmax, Lvmax-Lvmin+1)
    minorticks = log_minorticks(vmin, vmax)
    if position == 'right':
        if ax is None:
            cax = image.figure.add_axes([1, 0, .03*scale, 1])
        else:
            pos = ax.get_position()
            # cax = image.figure.add_axes([pos.x1 + 0.05*pos.height, pos.y0 + 0.05*pos.height, 0.02*scale, pos.height*0.9])
            cax = image.figure.add_axes([pos.x1 + 0.02*pos.height, pos.y0 + 0.225*pos.height, 0.02*scale, pos.height*0.55])
        cbar = image.figure.colorbar(image, cax=cax, ticks=10**Lcticks, orientation='vertical', extend=extend)
        cbar.ax.yaxis.set_ticks(minorticks, minor=True)
        cax.yaxis.set_minor_formatter(NullFormatter())
    elif position == 'left':
        cax = image.figure.add_axes([-.03*scale, 0, .03*scale, 1])
        cbar = image.figure.colorbar(image, cax=cax, ticks=10**Lcticks, orientation='vertical', extend=extend)
        cbar.ax.yaxis.set_ticks(minorticks, minor=True)
        cax.yaxis.set_minor_formatter(NullFormatter())
        cbar.ax.yaxis.set_ticks_position('left')
        cbar.ax.yaxis.set_label_position('left')
        if ticklabels is not None:
            cbar.ax.set_yticklabels(ticklabels)
    elif position == 'top':
        cax = image.figure.add_axes([0, 1, 1, .04*scale])
        cbar = image.figure.colorbar(image, cax=cax, ticks=10**Lcticks, orientation='horizontal', extend=extend)
        cbar.ax.xaxis.set_ticks(minorticks, minor=True)
        cax.xaxis.set_minor_formatter(NullFormatter())
        cbar.ax.xaxis.set_ticks_position('top')
        cbar.ax.xaxis.set_label_position('top')
    elif position == 'bottom':
        cax = image.figure.add_axes([0, -.04*scale, 1, .04*scale])
        cbar = image.figure.colorbar(image, cax=cax, ticks=10**Lcticks, orientation='horizontal', extend=extend)
        cbar.ax.xaxis.set_ticks(minorticks, minor=True)
        cax.xaxis.set_minor_formatter(NullFormatter())
    else:
        raise ValueError('position not recognized!')
    cax.tick_params(which='both', direction='in', labelsize=labelsize)
    if fontsize is not None:
        cbar.set_label(label, fontsize=fontsize, labelpad=7)
    else:
        cbar.set_label(label) #, fontsize=14, labelpad=8)
    # cbar.ax.set_yticklabels([r'$10^{%g}$' % val for val in Lcticks], fontsize=13)
    # cbar.solids.set_edgecolor("face")
    # locs = [0, 1];  ax.set_xticks(locs);  ax.set_yticks(locs)

def plot_rlc(rlc_dir='.', SliceFraction=0.2, RedshiftAxis=True, AgeAxis=True):
    vec_fields = ['IonFlux', 'Velocities']  # Vector fields
    band_fields = ['IonEnergies']  # Band fields
    with h5py.File(f'{rlc_dir}/All.hdf5', 'r') as f:
        # Copy header attributes, datasets, and dataset attributes
        header = f['Header']
        H0 = 100. * header.attrs['HubbleParam']  # Hubble constant [km/s/cMpc]
        Omega0 = header.attrs['Omega0']  # Total matter density
        OpeningAngle = header.attrs['OpeningAngle']  # Opening angle [rad]
        # OpeningAngle_arcmin = OpeningAngle * 180. / np.pi * 60.  # Opening angle [arcmin]
        HalfAngle = OpeningAngle / 2.  # Half angle [arcmin]
        NumPixels = header.attrs['NumPixels']  # Number of pixels
        # Fields
        fields = list(f.keys())
        print(f"All fields: {fields}")
        for key in ['Header', 'Redshifts', 'Distances', 'Segments']:
            if key in fields:
                fields.remove(key)
        for key in vec_fields + band_fields:
            if key in fields:
                fields.remove(key)  # Remove vector and band fields
        # fields = ['Density', 'Temperature', 'HII_Fraction']
        # fields = ['DensityDM', 'Density', 'DensityStars', 'DensityMetals', 'Temperature', 'HII_Fraction', 'HeII_Fraction', 'HeIII_Fraction', 'IonLuminosityStars', 'LuminosityAGN', 'LyaLuminosityRec', 'LyaLuminosityCol', 'StarFormationRate', 'VelocitySquared']
        fields = ['Density', 'Temperature', 'HII_Fraction', 'HeII_Fraction', 'HeIII_Fraction', 'LyaLuminosityRec', 'LyaLuminosityCol', 'VelocitySquared']
        # IonEnergies -> IonEnergy
        # Read fields
        SliceIndex = max(0, min(NumPixels - 1, int(SliceFraction * NumPixels)))
        ds = {field: f[field][SliceIndex,:,:] for field in fields}
        ds['LyaLuminosityRec'] = ds['LyaLuminosityRec'] + ds['LyaLuminosityCol']
        ds.pop('LyaLuminosityCol'); fields.remove('LyaLuminosityCol')
        ds['VelocitySquared'] = np.sqrt(ds['VelocitySquared'])
        fields = fields[::-1]
        print(f'Fields: {fields}')
        Redshifts = f['Redshifts'][:]  # Redshifts
        Distances = f['Distances'][:]  # Distances [cMpc/h]
    # Restrict to redshift <= 20
    zmax = 20.
    imax = np.argmin(np.abs(Redshifts - zmax))
    Redshifts = Redshifts[imax:]
    Distances = Distances[imax:]
    for field in ds:
        ds[field] = ds[field][:,imax:]
    for field in ['Density', 'Temperature', 'LyaLuminosityRec', 'VelocitySquared']:
        vlims = np.percentile(ds[field], [1, 99.9])
        norms[field] = LogNorm(vmin=vlims[0], vmax=vlims[1], clip=True)
    # Plot fields
    n_fields = len(fields)
    MaxAngle, RefAngle = float(n_fields) * HalfAngle, np.pi/2.
    Theta_E = np.linspace(RefAngle-MaxAngle, RefAngle+MaxAngle, n_fields + 1)
    Theta_C = (Theta_E[1:] + Theta_E[:-1]) / 2.
    Dmin, Dmax, Davg = Distances[-1], Distances[0], 0.5 * (Distances[-1] + Distances[0])
    Xcorners = [Dmin*np.cos(Theta_E[0]), Dmax*np.cos(Theta_E[0]), Dmax*np.cos(Theta_E[-1]), Dmin*np.cos(Theta_E[-1])]
    Ycorners = [Dmin*np.sin(Theta_E[0]), Dmax*np.sin(Theta_E[0]), Dmax*np.sin(Theta_E[-1]), Dmin*np.sin(Theta_E[-1])]
    if Theta_E[0] < 0. < Theta_E[-1]: Xcorners.append(Dmax)
    if Theta_E[0] < np.pi/2. < Theta_E[-1]: Ycorners.append(Dmax)
    if Theta_E[0] < np.pi < Theta_E[-1]: Xcorners.append(-Dmax)
    if Theta_E[0] < 3.*np.pi/2. < Theta_E[-1]: Ycorners.append(-Dmax)
    if RedshiftAxis:
        Xcorners.append(Dmax*np.cos(Theta_E[-1]+0.002))
    if AgeAxis:
        Xcorners.append(Dmax*np.cos(Theta_E[0]-0.002))
    Xlims, Ylims = [np.min(Xcorners), np.max(Xcorners)], [np.min(Ycorners), np.max(Ycorners)]
    dXlims = Xlims[1] - Xlims[0]
    dYlims = Ylims[1] - Ylims[0]
    Buffer = 0.0025
    Xlims = [Xlims[0] - Buffer*dXlims, Xlims[1] + Buffer*dXlims]
    Ylims = [Ylims[0] - Buffer*dYlims, Ylims[1] + Buffer*dYlims]
    fig = plt.figure(figsize=(12,12))
    ax = plt.axes([0,0,1,1])
    for i, field in enumerate(fields):
        Thetas = np.linspace(Theta_E[i], Theta_E[i+1], NumPixels+1)
        Theta2d, R2d = np.meshgrid(Thetas, Distances, indexing='ij')
        X = R2d * np.cos(Theta2d)
        Y = R2d * np.sin(Theta2d)
        print(f'{field}: {np.percentile(ds[field], [1, 99])}')
        # print(f'{field}: {100.*np.count_nonzero(ds[field])/ds[field].size:.2f}%  [{np.count_nonzero(ds[field])}]')
        image = ax.pcolormesh(X, Y, ds[field], cmap=cmaps[field], norm=norms[field], rasterized=True)
        # colorbar(image, labels[field], position='right', ax=ax, scale=0.75, fontsize=9, labelsize=7.5)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(Xlims)
    ax.set_ylim(Ylims)
    ax.axis('off')
    # Draw lines around each cone image
    for i in range(n_fields + 1):
        ax.plot([Dmin*np.cos(Theta_E[i]), Dmax*np.cos(Theta_E[i])], [Dmin*np.sin(Theta_E[i]), Dmax*np.sin(Theta_E[i])], c=grey, lw=1)
    # Draw arcs at the outer and inner edges
    theta = np.linspace(Theta_E[0], Theta_E[-1], 200)
    ax.plot(Dmin*np.cos(theta), Dmin*np.sin(theta), c=grey, lw=1)
    ax.plot(Dmax*np.cos(theta), Dmax*np.sin(theta), c=grey, lw=1)
    # Add labels
    Dlab = Dmin - 0.02 * (Dmax - Dmin)
    for i, field in enumerate(fields):
        ax.text(Dlab*np.cos(Theta_C[i]), Dlab*np.sin(Theta_C[i]), labels[field], ha='center', va='center', c=grey, fontsize=10, rotation=np.degrees(Theta_C[i])-90.)
    # Add a redshift axis
    if RedshiftAxis:
        for i in range(5, 21):
            iz = np.argmin(np.abs(Redshifts - float(i)))  # Index of closest redshift
            df = Distances[-1] / Distances[iz]
            cos_1, cos_2, cos_t = np.cos(Theta_E[-1]), np.cos(Theta_E[-1]+0.002*df), np.cos(Theta_E[-1]+0.006*df)
            sin_1, sin_2, sin_t = np.sin(Theta_E[-1]), np.sin(Theta_E[-1]+0.002*df), np.sin(Theta_E[-1]+0.006*df)
            ax.plot([Distances[iz]*cos_1, Distances[iz]*cos_2], [Distances[iz]*sin_1, Distances[iz]*sin_2], c=grey, lw=1.)
            if i not in [12, 14, 16, 17, 19, 20]:
                ax.text(Distances[iz]*cos_t, Distances[iz]*sin_t, r'${\bf '+str(i)+r'}$', ha='center', va='center', c=grey, fontsize=8, rotation=np.degrees(Theta_E[-1]+0.006*df))
        ax.text(Davg*np.cos(Theta_E[-1]+0.014), Davg*np.sin(Theta_E[-1]+0.014), r'${\bf Cosmological\ Redshift}$', ha='center', va='center', c=grey, fontsize=10, rotation=np.degrees(Theta_E[-1]+0.014))
    # Add an age axis
    if AgeAxis:
        from astropy.cosmology import FlatLambdaCDM
        from astropy.cosmology import z_at_value
        from astropy import units as u
        cosmo = FlatLambdaCDM(H0=H0, Om0=Omega0, Tcmb0=2.725)
        t_min = 1e3 * cosmo.age(Redshifts[0]).value  # Minimum time [Myr]
        t_max = 1e3 * cosmo.age(Redshifts[-1]).value  # Maximum time [Myr]
        # z_split = np.array(z_at_value(cosmo.age, t_split * u.Myr))  # Target redshifts from split times
        print(f'Dmin: {Dmin/1e6:g} cMpc/h, Dmax: {Dmax/1e6:g} cMpc/h')
        print(f't_min: {t_min:g} Myr, t_max: {t_max:g} Myr')
        Ages = [200., 300., 400., 500., 600., 700., 800., 900., 1000., 1100., 1200.]
        for age in Ages:
            z = z_at_value(cosmo.age, age * u.Myr)  # Target redshift for this age
            iz = np.argmin(np.abs(Redshifts - z))  # Index of closest redshift
            df = Distances[-1] / Distances[iz]
            cos_1, cos_2, cos_t = np.cos(Theta_E[0]), np.cos(Theta_E[0]-0.002*df), np.cos(Theta_E[0]-0.007*df)
            sin_1, sin_2, sin_t = np.sin(Theta_E[0]), np.sin(Theta_E[0]-0.002*df), np.sin(Theta_E[0]-0.007*df)
            ax.plot([Distances[iz]*cos_1, Distances[iz]*cos_2], [Distances[iz]*sin_1, Distances[iz]*sin_2], c=grey, lw=1.)
            if age not in [1100.]:
                ax.text(Distances[iz]*cos_t, Distances[iz]*sin_t, r'${\bf '+str(int(age))+r'}$', ha='center', va='center', c=grey, fontsize=8, rotation=np.degrees(Theta_E[0]-0.007*df))
        ax.text(Davg*np.cos(Theta_E[0]-0.016), Davg*np.sin(Theta_E[0]-0.016), r'${\bf Time\ Since\ Big\ Bang\ \,(Myr)}$', ha='center', va='center', c=grey, fontsize=10, rotation=np.degrees(Theta_E[0]-0.016))
    ax.text(0.5*Dmin*(np.cos(Theta_E[0])+np.cos(Theta_E[-1])), 0.5*Dmin*(np.sin(Theta_E[0])+np.sin(Theta_E[-1])),
            r'${\bf LUMINA}$', ha='center', va='center', c=grey, fontsize=13)
    #fig.savefig('rlc_fan.pdf', bbox_inches='tight', dpi=200, pad_inches=0.025, transparent=True)
    #plt.close()
    plt.show()

In [ ]:
%matplotlib inline
rlc_dir = '/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_320'

plot_rlc(rlc_dir=rlc_dir)

In [ ]:
%matplotlib inline

import numpy as np
import cmasher as cmr
import cmocean
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import NullFormatter
import h5py

# =========================================================
# Labels / colormaps
# =========================================================

labels = {
    'Density': r'${\bf Density}$',
    'Temperature': r'${\bf Temperature}$',
    'HII_Fraction': r'${\bf HII\ Fraction}$',
    'HeII_Fraction': r'${\bf HeII\ Fraction}$',
    'HeIII_Fraction': r'${\bf HeIII\ Fraction}$',
    'LyaLuminosityRec': r'${\bf Ly\alpha\ Emission}$',
    'VelocitySquared': r'${\bf Velocity}$'
}

cmaps = {
    'Density': cmocean.cm.thermal,
    'Temperature': 'inferno',
    'HII_Fraction': cmr.amber,
    'HeII_Fraction': 'twilight_shifted',
    'HeIII_Fraction': cmr.fusion,
    'LyaLuminosityRec': cmr.rainforest,
    'VelocitySquared': cmr.ember
}

grey = [0.1, 0.1, 0.1]

# =========================================================
# Physical constants
# =========================================================

HYDROGEN_MASSFRAC = 0.76
HE_ABUND = (1. / HYDROGEN_MASSFRAC - 1.) / 4.

norms = {
    'Density': LogNorm(),
    'Temperature': LogNorm(vmin=100., vmax=1e6, clip=True),
    'HII_Fraction': Normalize(vmin=0., vmax=1., clip=True),
    'HeII_Fraction': Normalize(vmin=0., vmax=HE_ABUND, clip=True),
    'HeIII_Fraction': Normalize(vmin=0., vmax=HE_ABUND, clip=True),
    'LyaLuminosityRec': LogNorm(),
    'VelocitySquared': LogNorm()
}

# =========================================================
# Main plotting function
# =========================================================

def plot_rlc_semicircle(
    rlc_dir='.',
    SliceFraction=0.2,
    zmax=20.
):
    vec_fields = ['IonFlux', 'Velocities']
    band_fields = ['IonEnergies']

    # =====================================================
    # Read file
    # =====================================================

    with h5py.File(f'{rlc_dir}/All.hdf5', 'r') as f:

        header = f['Header']

        H0 = 100. * header.attrs['HubbleParam']
        Omega0 = header.attrs['Omega0']
        NumPixels = header.attrs['NumPixels']

        fields = [
            'Density',
            'Temperature',
            'HII_Fraction',
            'HeII_Fraction',
            'HeIII_Fraction',
            'LyaLuminosityRec',
            'LyaLuminosityCol',
            'VelocitySquared'
        ]

        SliceIndex = max(
            0,
            min(NumPixels - 1, int(SliceFraction * NumPixels))
        )

        ds = {
            field: f[field][SliceIndex, :, :]
            for field in fields
        }

        # Combine Lyα channels
        ds['LyaLuminosityRec'] = (
            ds['LyaLuminosityRec']
            + ds['LyaLuminosityCol']
        )

        ds.pop('LyaLuminosityCol')
        fields.remove('LyaLuminosityCol')

        # sqrt(v^2)
        ds['VelocitySquared'] = np.sqrt(ds['VelocitySquared'])

        # reverse for plotting order
        fields = fields[::-1]

        Redshifts = f['Redshifts'][:]
        Distances = f['Distances'][:]

    # =====================================================
    # Restrict redshift range
    # =====================================================

    imax = np.argmin(np.abs(Redshifts - zmax))

    Redshifts = Redshifts[imax:]
    Distances = Distances[imax:]

    for field in ds:
        ds[field] = ds[field][:, imax:]

    # =====================================================
    # Dynamic log limits
    # =====================================================

    for field in [
        'Density',
        'Temperature',
        'LyaLuminosityRec',
        'VelocitySquared'
    ]:
        vlims = np.percentile(ds[field], [1, 99.5])

        norms[field] = LogNorm(
            vmin=max(vlims[0], 1e-30),
            vmax=vlims[1],
            clip=True
        )

    # =====================================================
    # Semi-circle geometry
    # =====================================================

    n_fields = len(fields)

    # exact 180-degree semicircle
    Theta_E = np.linspace(0, np.pi, n_fields + 1)
    Theta_C = 0.5 * (Theta_E[:-1] + Theta_E[1:])

    Dmin = Distances[-1]
    Dmax = Distances[0]
    Davg = 0.5 * (Dmin + Dmax)

    theta = np.linspace(0, np.pi, 500)

    X_outer = Dmax * np.cos(theta)
    Y_outer = Dmax * np.sin(theta)

    X_inner = Dmin * np.cos(theta)
    Y_inner = Dmin * np.sin(theta)

    Xlims = [np.min(X_outer), np.max(X_outer)]
    Ylims = [0, np.max(Y_outer)]

    Buffer = 0.03

    dx = Xlims[1] - Xlims[0]
    dy = Ylims[1] - Ylims[0]

    Xlims = [
        Xlims[0] - Buffer * dx,
        Xlims[1] + Buffer * dx
    ]

    Ylims = [
        Ylims[0] - Buffer * dy,
        Ylims[1] + Buffer * dy
    ]

    # =====================================================
    # Figure
    # =====================================================

    fig, ax = plt.subplots(figsize=(15, 8))

    for i, field in enumerate(fields):

        Thetas = np.linspace(
            Theta_E[i],
            Theta_E[i + 1],
            NumPixels + 1
        )

        Theta2d, R2d = np.meshgrid(
            Thetas,
            Distances,
            indexing='ij'
        )

        X = R2d * np.cos(Theta2d)
        Y = R2d * np.sin(Theta2d)

        print(
            f"{field}: "
            f"{np.percentile(ds[field], [1, 99])}"
        )

        image = ax.pcolormesh(
            X,
            Y,
            ds[field],
            cmap=cmaps[field],
            norm=norms[field],
            rasterized=True,
            shading='auto'
        )

    # =====================================================
    # Outline
    # =====================================================

    for i in range(n_fields + 1):
        ax.plot(
            [
                Dmin * np.cos(Theta_E[i]),
                Dmax * np.cos(Theta_E[i])
            ],
            [
                Dmin * np.sin(Theta_E[i]),
                Dmax * np.sin(Theta_E[i])
            ],
            c=grey,
            lw=1
        )

    ax.plot(
        Dmin * np.cos(theta),
        Dmin * np.sin(theta),
        c=grey,
        lw=1
    )

    ax.plot(
        Dmax * np.cos(theta),
        Dmax * np.sin(theta),
        c=grey,
        lw=1
    )

    # =====================================================
    # Labels
    # =====================================================

    Dlab = Dmin - 0.06 * (Dmax - Dmin)

    for i, field in enumerate(fields):
        ax.text(
            Dlab * np.cos(Theta_C[i]),
            Dlab * np.sin(Theta_C[i]),
            labels[field],
            ha='center',
            va='center',
            fontsize=10,
            color=grey,
            rotation=np.degrees(Theta_C[i]) - 90.
        )

    # =====================================================
    # Central title
    # =====================================================

    ax.text(
        0,
        0.55 * Dmin,
        r'${\bf LUMINA}$',
        ha='center',
        va='center',
        fontsize=14,
        color=grey
    )

    # =====================================================
    # Final cosmetics
    # =====================================================

    ax.set_aspect('equal')
    ax.set_xlim(Xlims)
    ax.set_ylim(Ylims)
    ax.axis('off')

    plt.tight_layout()
    plt.show()


# =========================================================
# Run
# =========================================================

rlc_dir = "/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_320"

plot_rlc_semicircle(
    rlc_dir=rlc_dir,
    SliceFraction=0.2,
    zmax=20.
)

In [ ]:
%matplotlib inline

import numpy as np
import cmasher as cmr
import cmocean
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
import h5py

# =========================================================
# Labels / colormaps
# =========================================================

labels = {
    'Density': r'${\bf Density}$',
    'Temperature': r'${\bf Temperature}$',
    'HII_Fraction': r'${\bf HII\ Fraction}$',
    'HeII_Fraction': r'${\bf HeII\ Fraction}$',
    'HeIII_Fraction': r'${\bf HeIII\ Fraction}$',
    'LyaLuminosityRec': r'${\bf Ly\alpha\ Emission}$',
    'VelocitySquared': r'${\bf Velocity}$'
}

cmaps = {
    'Density': cmocean.cm.thermal,
    'Temperature': 'inferno',
    'HII_Fraction': cmr.amber,
    'HeII_Fraction': 'twilight_shifted',
    'HeIII_Fraction': cmr.fusion,
    'LyaLuminosityRec': cmr.rainforest,
    'VelocitySquared': cmr.ember
}

grey = [0.1, 0.1, 0.1]

# =========================================================
# Physical constants
# =========================================================

HYDROGEN_MASSFRAC = 0.76
HE_ABUND = (1. / HYDROGEN_MASSFRAC - 1.) / 4.

norms = {
    'Density': LogNorm(),
    'Temperature': LogNorm(vmin=100., vmax=1e6, clip=True),
    'HII_Fraction': Normalize(vmin=0., vmax=1., clip=True),
    'HeII_Fraction': Normalize(vmin=0., vmax=HE_ABUND, clip=True),
    'HeIII_Fraction': Normalize(vmin=0., vmax=HE_ABUND, clip=True),
    'LyaLuminosityRec': LogNorm(),
    'VelocitySquared': LogNorm()
}

# =========================================================
# Main function
# =========================================================

def plot_rlc_pizza(
    rlc_dir='.',
    SliceFraction=0.2,
    zmax=20.
):
    # =====================================================
    # Read file
    # =====================================================

    with h5py.File(f'{rlc_dir}/All.hdf5', 'r') as f:

        header = f['Header']
        NumPixels = header.attrs['NumPixels']

        fields = [
            'Density',
            'Temperature',
            'HII_Fraction',
            'HeII_Fraction',
            'HeIII_Fraction',
            'LyaLuminosityRec',
            'LyaLuminosityCol',
            'VelocitySquared'
        ]

        SliceIndex = max(
            0,
            min(NumPixels - 1, int(SliceFraction * NumPixels))
        )

        ds = {
            field: f[field][SliceIndex, :, :]
            for field in fields
        }

        # Combine Lyα channels
        ds['LyaLuminosityRec'] = (
            ds['LyaLuminosityRec']
            + ds['LyaLuminosityCol']
        )

        ds.pop('LyaLuminosityCol')
        fields.remove('LyaLuminosityCol')

        # sqrt(v^2)
        ds['VelocitySquared'] = np.sqrt(ds['VelocitySquared'])

        # reverse order for plotting
        fields = fields[::-1]

        Redshifts = f['Redshifts'][:]
        Distances = f['Distances'][:]

    # =====================================================
    # Restrict redshift range
    # =====================================================

    imax = np.argmin(np.abs(Redshifts - zmax))

    Redshifts = Redshifts[imax:]
    Distances = Distances[imax:]

    for field in ds:
        ds[field] = ds[field][:, imax:]

    # =====================================================
    # Dynamic limits for log fields
    # =====================================================

    for field in [
        'Density',
        'Temperature',
        'LyaLuminosityRec',
        'VelocitySquared'
    ]:
        vlims = np.percentile(ds[field], [1, 99.5])

        norms[field] = LogNorm(
            vmin=max(vlims[0], 1e-30),
            vmax=vlims[1],
            clip=True
        )

    # =====================================================
    # Full semicircle (pizza style)
    # =====================================================

    n_fields = len(fields)

    # exact 180-degree semicircle
    Theta_E = np.linspace(0, np.pi, n_fields + 1)
    Theta_C = 0.5 * (Theta_E[:-1] + Theta_E[1:])

    # FULL filled semicircle
    Dmin = 0.0
    Dmax = Distances[0]
    Davg = 0.5 * Dmax

    theta = np.linspace(0, np.pi, 500)

    X_outer = Dmax * np.cos(theta)
    Y_outer = Dmax * np.sin(theta)

    Xlims = [np.min(X_outer), np.max(X_outer)]
    Ylims = [0, np.max(Y_outer)]

    Buffer = 0.03

    dx = Xlims[1] - Xlims[0]
    dy = Ylims[1] - Ylims[0]

    Xlims = [
        Xlims[0] - Buffer * dx,
        Xlims[1] + Buffer * dx
    ]

    Ylims = [
        Ylims[0] - Buffer * dy,
        Ylims[1] + Buffer * dy
    ]

    # labels inside slices
    Dlab = 0.82 * Dmax

    # =====================================================
    # Figure
    # =====================================================

    fig, ax = plt.subplots(figsize=(15, 8))

    for i, field in enumerate(fields):

        Thetas = np.linspace(
            Theta_E[i],
            Theta_E[i + 1],
            NumPixels + 1
        )

        # IMPORTANT:
        # radial coordinate must start from 0 for pizza shape
        Rvals = np.linspace(
            Dmax,
            Dmin,
            len(Distances)
        )

        Theta2d, R2d = np.meshgrid(
            Thetas,
            Rvals,
            indexing='ij'
        )

        X = R2d * np.cos(Theta2d)
        Y = R2d * np.sin(Theta2d)

        # Stretch original data to full radius
        data = ds[field]

        image = ax.pcolormesh(
            X,
            Y,
            data,
            cmap=cmaps[field],
            norm=norms[field],
            shading='auto',
            rasterized=True
        )

    # =====================================================
    # Outer boundary + radial separators
    # =====================================================

    for i in range(n_fields + 1):
        ax.plot(
            [
                Dmin * np.cos(Theta_E[i]),
                Dmax * np.cos(Theta_E[i])
            ],
            [
                Dmin * np.sin(Theta_E[i]),
                Dmax * np.sin(Theta_E[i])
            ],
            c=grey,
            lw=1
        )

    # Outer arc only
    ax.plot(
        Dmax * np.cos(theta),
        Dmax * np.sin(theta),
        c=grey,
        lw=1
    )

    # =====================================================
    # Labels
    # =====================================================

    for i, field in enumerate(fields):
        ax.text(
            Dlab * np.cos(Theta_C[i]),
            Dlab * np.sin(Theta_C[i]),
            labels[field],
            ha='center',
            va='center',
            fontsize=10,
            color=grey,
            rotation=np.degrees(Theta_C[i]) - 90.
        )

    # =====================================================
    # Center label
    # =====================================================

    ax.text(
        0,
        0.18 * Dmax,
        r'${\bf LUMINA}$',
        ha='center',
        va='center',
        fontsize=14,
        color=grey
    )

    # =====================================================
    # Final cosmetics
    # =====================================================

    ax.set_aspect('equal')
    ax.set_xlim(Xlims)
    ax.set_ylim(Ylims)
    ax.axis('off')

    plt.tight_layout()
    plt.show()


# =========================================================
# Run
# =========================================================

rlc_dir = "/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_320"

plot_rlc_pizza(
    rlc_dir=rlc_dir,
    SliceFraction=0.2,
    zmax=20.
)

In [ ]:
%matplotlib inline

import numpy as np
import cmasher as cmr
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import h5py

# =====================================================
# Labels / colormaps
# =====================================================

labels = {
    "Tb21cm": r"${\bf HI\ 21cm}$",
    "TbHe3": r"${\bf ^3HeII\ 3.46cm}$"
}

cmaps = {
    "Tb21cm": cmr.fusion,
    "TbHe3": cmr.ember
}

grey = [0.1, 0.1, 0.1]

Tcmb0 = 2.725  # K


# =====================================================
# Main function
# =====================================================

def plot_brightness_temperature_pizza(
    rlc_dir='.',
    SliceFraction=0.2,
    zmax=20.
):

    # =================================================
    # Read file
    # =================================================

    with h5py.File(f"{rlc_dir}/All.hdf5", "r") as f:

        NumPixels = f["Header"].attrs["NumPixels"]

        fields = [
            "Density",
            "Temperature",
            "HII_Fraction",
            "HeII_Fraction"
        ]

        SliceIndex = max(
            0,
            min(NumPixels - 1, int(SliceFraction * NumPixels))
        )

        ds = {
            field: f[field][SliceIndex, :, :]
            for field in fields
        }

        Redshifts = f["Redshifts"][:]   # usually edges
        Distances = f["Distances"][:]   # usually edges

    # =================================================
    # Restrict to z <= zmax
    # =================================================

    imax = np.argmin(np.abs(Redshifts - zmax))

    Redshifts = Redshifts[imax:]
    Distances = Distances[imax:]

    for field in ds:
        ds[field] = ds[field][:, imax:]

    # =================================================
    # Convert redshift edges -> cell centers
    # =================================================

    z_centers = 0.5 * (Redshifts[:-1] + Redshifts[1:])

    # Match exact shape
    nz = ds["Density"].shape[1]
    z_centers = z_centers[:nz]

    # keep radial edges for pcolormesh
    Distances = Distances[:nz + 1]

    # =================================================
    # Extract fields
    # =================================================

    density = ds["Density"]
    temperature = ds["Temperature"]
    xHII = ds["HII_Fraction"]
    xHeII = ds["HeII_Fraction"]

    xHI = 1.0 - xHII

    # baryon overdensity
    mean_density = np.mean(density)
    delta_b = density / mean_density

    # 2D redshift grid
    z2d = np.tile(z_centers, (NumPixels, 1))

    # CMB temperature
    Tcmb = Tcmb0 * (1.0 + z2d)

    # Assume spin temperature ~ gas temperature
    Ts = np.maximum(temperature, Tcmb + 1e-5)

    coupling = 1.0 - Tcmb / Ts

    # =================================================
    # HI 21 cm brightness temperature [mK]
    # =================================================

    #\delta T_b^{\mathrm{HI}} \approx 27\,x_{\mathrm{HI}}\,(1+\delta_b)\left(\frac{1+z}{10}\right)^{1/2}\left(1-\frac{T_{\mathrm{CMB}}}{T_S}\right)

    Tb21 = (
        27.0
        * xHI
        * delta_b
        * np.sqrt((1.0 + z2d) / 10.0)
        * coupling
    )

    # =================================================
    # 3HeII brightness temperature [mK]
    # simplified visualization model
    # =================================================

    #\delta T_b^{^3\mathrm{HeII}} \propto x_{\mathrm{HeII}}\,(1+\delta_b)\left(\frac{1+z}{10}\right)^{1/2}\left(1-\frac{T_{\mathrm{CMB}}}{T_S}\right)

    TbHe3 = (
        0.03
        * xHeII
        * delta_b
        * np.sqrt((1.0 + z2d) / 10.0)
        * coupling
    )

    plot_fields = {
        "Tb21cm": np.abs(Tb21),
        "TbHe3": np.abs(TbHe3)
    }

    fields = ["Tb21cm", "TbHe3"]

    # =================================================
    # Dynamic normalization
    # =================================================

    norms = {}

    for field in fields:
        data = plot_fields[field]

        positive = data[data > 0]

        vmin = max(np.percentile(positive, 5), 1e-6)
        vmax = np.percentile(positive, 99)

        norms[field] = LogNorm(
            vmin=vmin,
            vmax=vmax,
            clip=True
        )

    # =================================================
    # 120-degree pizza geometry
    # =================================================

    n_fields = len(fields)

    opening_angle = np.radians(120.0)
    theta_center = np.pi / 2.0

    Theta_E = np.linspace(
        theta_center - opening_angle / 2.0,
        theta_center + opening_angle / 2.0,
        n_fields + 1
    )

    Theta_C = 0.5 * (Theta_E[:-1] + Theta_E[1:])

    Dmin = 0.0
    Dmax = Distances[0]

    theta = np.linspace(
        Theta_E[0],
        Theta_E[-1],
        500
    )

    Dlab = 0.78 * Dmax

    # =================================================
    # Figure
    # =================================================

    fig, ax = plt.subplots(figsize=(12, 8))

    for i, field in enumerate(fields):

        Thetas = np.linspace(
            Theta_E[i],
            Theta_E[i + 1],
            NumPixels + 1
        )

        # pcolormesh requires edges
        Rvals = np.linspace(
            Dmax,
            Dmin,
            len(z_centers) + 1
        )

        Theta2d, R2d = np.meshgrid(
            Thetas,
            Rvals,
            indexing="ij"
        )

        X = R2d * np.cos(Theta2d)
        Y = R2d * np.sin(Theta2d)

        data = plot_fields[field]

        ax.pcolormesh(
            X,
            Y,
            data,
            cmap=cmaps[field],
            norm=norms[field],
            shading="auto",
            rasterized=True
        )

    # =================================================
    # Boundaries
    # =================================================

    for i in range(n_fields + 1):
        ax.plot(
            [0, Dmax * np.cos(Theta_E[i])],
            [0, Dmax * np.sin(Theta_E[i])],
            c=grey,
            lw=1
        )

    ax.plot(
        Dmax * np.cos(theta),
        Dmax * np.sin(theta),
        c=grey,
        lw=1
    )

    # =================================================
    # Labels
    # =================================================

    for i, field in enumerate(fields):
        ax.text(
            Dlab * np.cos(Theta_C[i]),
            Dlab * np.sin(Theta_C[i]),
            labels[field],
            ha="center",
            va="center",
            fontsize=13,
            color=grey,
            rotation=np.degrees(Theta_C[i]) - 90
        )

    ax.text(
        0,
        0.28 * Dmax,
        r"${\bf Brightness\ Temperature}$",
        ha="center",
        va="center",
        fontsize=14,
        color=grey
    )

    # =================================================
    # Final cosmetics
    # =================================================

    ax.set_aspect("equal")
    ax.axis("off")

    plt.tight_layout()
    plt.show()


# =====================================================
# Run
# =====================================================

rlc_dir = "/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_320"

plot_brightness_temperature_pizza(
    rlc_dir=rlc_dir,
    SliceFraction=0.2,
    zmax=20.
)

In [ ]:
%matplotlib inline

import numpy as np
import cmasher as cmr
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
import h5py

# =====================================================
# Labels / Colormaps
# =====================================================

labels = {
    "HII_Fraction": r"${\bf HII\ Fraction}$",
    "Tb21cm": r"${\bf HI\ 21cm}$",
    "StarFormationRate": r"${\bf Star\ Formation\ Rate}$",
    "Temperature": r"${\bf Temperature}$",
    "TbHe3": r"${\bf ^3HeII\ 3.46cm}$",
    "HeIII_Fraction": r"${\bf HeIII\ Fraction}$"
}

cmaps = {
    "HII_Fraction": cmr.amber,
    "Tb21cm": cmr.fusion,
    "StarFormationRate": "inferno",
    "Temperature": "magma",
    "TbHe3": cmr.ember,
    "HeIII_Fraction": cmr.flamingo
}

grey = [0.1, 0.1, 0.1]
Tcmb0 = 2.725  # K


# =====================================================
# Main function
# =====================================================

def plot_brightness_temperature_pizza(
    rlc_dir=".",
    SliceFraction=0.2,
    zmax=20.0
):

    # =================================================
    # Read file
    # =================================================

    with h5py.File(f"{rlc_dir}/All.hdf5", "r") as f:

        NumPixels = f["Header"].attrs["NumPixels"]

        base_fields = [
            "StarFormationRate",
            "Temperature",
            "HII_Fraction",
            "HeII_Fraction",
        ]

        SliceIndex = max(
            0,
            min(NumPixels - 1, int(SliceFraction * NumPixels))
        )

        ds = {
            field: f[field][SliceIndex, :, :]
            for field in base_fields
        }

        Redshifts = f["Redshifts"][:]   # usually N+1
        Distances = f["Distances"][:]   # usually N+1

    # =================================================
    # Restrict to z <= zmax
    # =================================================

    imax = np.argmin(np.abs(Redshifts - zmax))

    Redshifts = Redshifts[imax:]
    Distances = Distances[imax:]

    for field in ds:
        ds[field] = ds[field][:, imax:]

    # =================================================
    # Convert redshift edges -> cell centers
    # =================================================

    z_centers = 0.5 * (Redshifts[:-1] + Redshifts[1:])

    nz = ds["Temperature"].shape[1]

    z_centers = z_centers[:nz]

    # keep radial edges for pcolormesh
    Distances = Distances[:nz + 1]

    # =================================================
    # Extract fields
    # =================================================

    sfr = ds["StarFormationRate"]
    temperature = ds["Temperature"]
    xHII = ds["HII_Fraction"]
    xHeII = ds["HeII_Fraction"]

    xHI = 1.0 - xHII
    xHeIII = 1.0 - xHeII

    # Use SFR as large-scale tracer proxy for overdensity
    mean_sfr = np.mean(sfr[sfr > 0]) if np.any(sfr > 0) else 1.0
    delta_b = np.where(sfr > 0, sfr / mean_sfr, 1.0)

    # 2D redshift grid
    z2d = np.tile(z_centers, (NumPixels, 1))

    # CMB temperature
    Tcmb = Tcmb0 * (1.0 + z2d)

    # assume Ts ~ Tgas
    Ts = np.maximum(temperature, Tcmb + 1e-5)

    coupling = 1.0 - Tcmb / Ts

    # =================================================
    # HI 21 cm brightness temperature [mK]
    # =================================================

    Tb21 = (
        27.0
        * xHI
        * delta_b
        * np.sqrt((1.0 + z2d) / 10.0)
        * coupling
    )

    # =================================================
    # 3HeII brightness temperature [mK]
    # simplified visualization model
    # =================================================

    TbHe3 = (
        0.03
        * xHeII
        * delta_b
        * np.sqrt((1.0 + z2d) / 10.0)
        * coupling
    )

    # =================================================
    # Fields to plot
    # =================================================

    plot_fields = {
        "HII_Fraction": xHII,
        "Tb21cm": np.abs(Tb21),
        "StarFormationRate": sfr,
        "Temperature": temperature,
        "TbHe3": np.abs(TbHe3),
        "HeIII_Fraction": xHeIII
    }

    fields = [
        "HII_Fraction",
        "Tb21cm",
        "StarFormationRate",
        "Temperature",
        "TbHe3",
        "HeIII_Fraction"
    ]

    # =================================================
    # Robust norms
    # =================================================

    norms = {
        "HII_Fraction": Normalize(vmin=0.0, vmax=1.0),
        "HeIII_Fraction": Normalize(vmin=0.0, vmax=1.0)
    }

    for field in [
        "Tb21cm",
        "StarFormationRate",
        "Temperature",
        "TbHe3"
    ]:

        data = plot_fields[field]

        positive = data[
            np.isfinite(data) & (data > 0)
        ]

        if len(positive) < 10:
            vmin = 1e-6
            vmax = 1.0
        else:
            vmin = np.percentile(positive, 5)
            vmax = np.percentile(positive, 99)

            vmin = max(vmin, 1e-6)

            if vmax <= vmin:
                vmax = vmin * 10.0

        norms[field] = LogNorm(
            vmin=vmin,
            vmax=vmax,
            clip=True
        )

        print(
            field,
            "vmin =", vmin,
            "vmax =", vmax,
            "min =", np.min(data),
            "max =", np.max(data)
        )

    # =================================================
    # 180-degree pizza geometry
    # =================================================

    n_fields = len(fields)

    opening_angle = np.radians(180.0)
    theta_center = np.pi / 2.0

    Theta_E = np.linspace(
        theta_center - opening_angle / 2.0,
        theta_center + opening_angle / 2.0,
        n_fields + 1
    )

    Theta_C = 0.5 * (Theta_E[:-1] + Theta_E[1:])

    Dmin = 0.0
    Dmax = Distances[0]

    theta = np.linspace(
        Theta_E[0],
        Theta_E[-1],
        500
    )

    Dlab = 0.82 * Dmax

    # =================================================
    # Figure
    # =================================================

    fig, ax = plt.subplots(figsize=(16, 9))

    for i, field in enumerate(fields):

        Thetas = np.linspace(
            Theta_E[i],
            Theta_E[i + 1],
            NumPixels + 1
        )

        Rvals = np.linspace(
            Dmax,
            Dmin,
            len(z_centers) + 1
        )

        Theta2d, R2d = np.meshgrid(
            Thetas,
            Rvals,
            indexing="ij"
        )

        X = R2d * np.cos(Theta2d)
        Y = R2d * np.sin(Theta2d)

        data = plot_fields[field]

        ax.pcolormesh(
            X,
            Y,
            data,
            cmap=cmaps[field],
            norm=norms[field],
            shading="auto",
            rasterized=True
        )

    # =================================================
    # Boundaries
    # =================================================

    for i in range(n_fields + 1):
        ax.plot(
            [0, Dmax * np.cos(Theta_E[i])],
            [0, Dmax * np.sin(Theta_E[i])],
            c=grey,
            lw=1
        )

    ax.plot(
        Dmax * np.cos(theta),
        Dmax * np.sin(theta),
        c=grey,
        lw=1
    )

    # =================================================
    # Labels
    # =================================================

    for i, field in enumerate(fields):
        ax.text(
            Dlab * np.cos(Theta_C[i]),
            Dlab * np.sin(Theta_C[i]),
            labels[field],
            ha="center",
            va="center",
            fontsize=11,
            color=grey,
            rotation=np.degrees(Theta_C[i]) - 90
        )

    ax.text(
        0,
        0.20 * Dmax,
        r"${\bf Brightness\ Temperature\ Lightcone}$",
        ha="center",
        va="center",
        fontsize=14,
        color=grey
    )

    # =================================================
    # Final cosmetics
    # =================================================

    ax.set_aspect("equal")
    ax.axis("off")

    plt.tight_layout()
    plt.show()


# =====================================================
# Run
# =====================================================

rlc_dir = "/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_320"

plot_brightness_temperature_pizza(
    rlc_dir=rlc_dir,
    SliceFraction=0.2,
    zmax=20.0
)

In [ ]:
%matplotlib inline

import numpy as np
import cmasher as cmr
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
import h5py

# =====================================================
# Labels / Colormaps
# =====================================================

labels = {
    "HII_Fraction": r"${\bf HII\ Fraction}$",
    "Tb21cm": r"${\bf HI\ 21cm}$",
    "StarFormationRate": r"${\bf Star\ Formation\ Rate}$",
    "Temperature": r"${\bf Temperature}$",
    "TbHe3": r"${\bf ^3HeII\ 3.46cm}$",
    "HeIII_Fraction": r"${\bf HeIII\ Fraction}$"
}

cmaps = {
    "HII_Fraction": cmr.amber,
    "Tb21cm": cmr.fusion,
    "StarFormationRate": "inferno",
    "Temperature": "magma",
    "TbHe3": cmr.ember,
    "HeIII_Fraction": cmr.flamingo
}

grey = [0.1, 0.1, 0.1]
Tcmb0 = 2.725


# =====================================================
# Main function
# =====================================================

def plot_spiral_lightcone(
    rlc_dir=".",
    SliceFraction=0.2,
    zmax=20.0
):

    # =================================================
    # Read file
    # =================================================

    with h5py.File(f"{rlc_dir}/All.hdf5", "r") as f:

        NumPixels = f["Header"].attrs["NumPixels"]

        base_fields = [
            "StarFormationRate",
            "Temperature",
            "HII_Fraction",
            "HeII_Fraction"
        ]

        SliceIndex = max(
            0,
            min(NumPixels - 1, int(SliceFraction * NumPixels))
        )

        ds = {
            field: f[field][SliceIndex, :, :]
            for field in base_fields
        }

        Redshifts = f["Redshifts"][:]
        Distances = f["Distances"][:]

    # =================================================
    # Restrict z range
    # =================================================

    imax = np.argmin(np.abs(Redshifts - zmax))

    Redshifts = Redshifts[imax:]
    Distances = Distances[imax:]

    for field in ds:
        ds[field] = ds[field][:, imax:]

    # =================================================
    # Redshift centers
    # =================================================

    z_centers = 0.5 * (Redshifts[:-1] + Redshifts[1:])

    nz = ds["Temperature"].shape[1]

    z_centers = z_centers[:nz]
    Distances = Distances[:nz + 1]

    # =================================================
    # Extract fields
    # =================================================

    sfr = ds["StarFormationRate"]
    temperature = ds["Temperature"]
    xHII = ds["HII_Fraction"]
    xHeII = ds["HeII_Fraction"]

    xHI = 1.0 - xHII
    xHeIII = 1.0 - xHeII

    # use SFR proxy for overdensity
    mean_sfr = np.mean(sfr[sfr > 0]) if np.any(sfr > 0) else 1.0
    delta_b = np.where(sfr > 0, sfr / mean_sfr, 1.0)

    z2d = np.tile(z_centers, (NumPixels, 1))
    Tcmb = Tcmb0 * (1.0 + z2d)

    Ts = np.maximum(temperature, Tcmb + 1e-5)
    coupling = 1.0 - Tcmb / Ts

    # =================================================
    # Brightness temperatures
    # =================================================

    Tb21 = (
        27.0
        * xHI
        * delta_b
        * np.sqrt((1.0 + z2d) / 10.0)
        * coupling
    )

    TbHe3 = (
        0.03
        * xHeII
        * delta_b
        * np.sqrt((1.0 + z2d) / 10.0)
        * coupling
    )

    plot_fields = {
        "HII_Fraction": xHII,
        "Tb21cm": np.abs(Tb21),
        "StarFormationRate": sfr,
        "Temperature": temperature,
        "TbHe3": np.abs(TbHe3),
        "HeIII_Fraction": xHeIII
    }

    fields = [
        "HII_Fraction",
        "Tb21cm",
        "StarFormationRate",
        "Temperature",
        "TbHe3",
        "HeIII_Fraction"
    ]

    # =================================================
    # Norms
    # =================================================

    norms = {
        "HII_Fraction": Normalize(vmin=0.0, vmax=1.0),
        "HeIII_Fraction": Normalize(vmin=0.0, vmax=1.0)
    }

    for field in [
        "Tb21cm",
        "StarFormationRate",
        "Temperature",
        "TbHe3"
    ]:
        data = plot_fields[field]
        positive = data[np.isfinite(data) & (data > 0)]

        if len(positive) < 10:
            vmin = 1e-6
            vmax = 1.0
        else:
            vmin = max(np.percentile(positive, 5), 1e-6)
            vmax = np.percentile(positive, 99)

            if vmax <= vmin:
                vmax = vmin * 10.0

        norms[field] = LogNorm(
            vmin=vmin,
            vmax=vmax,
            clip=True
        )

    # =================================================
    # Spiral geometry
    #
    # Instead of wedges:
    # each field occupies one spiral turn
    # =================================================

    n_fields = len(fields)

    # total turns = number of fields
    turns_per_field = 1.0
    total_turns = n_fields * turns_per_field

    theta_start = 0.0
    theta_end = 2.0 * np.pi * total_turns

    Dmax = Distances[0]

    fig, ax = plt.subplots(figsize=(11, 11))

    for i, field in enumerate(fields):

        # angular extent for one field
        theta0 = 2.0 * np.pi * i
        theta1 = 2.0 * np.pi * (i + 1)

        theta_edges = np.linspace(
            theta0,
            theta1,
            NumPixels + 1
        )

        # radial edges
        r_edges = np.linspace(
            0.0,
            Dmax,
            len(z_centers) + 1
        )

        Theta2d, R2d = np.meshgrid(
            theta_edges,
            r_edges,
            indexing="ij"
        )

        # spiral offset:
        # radius increases with angle
        spiral_pitch = 0.12 * Dmax
        R_spiral = R2d + spiral_pitch * (
            (Theta2d - theta0) / (2.0 * np.pi)
            + i
        )

        X = R_spiral * np.cos(Theta2d)
        Y = R_spiral * np.sin(Theta2d)

        data = plot_fields[field]

        ax.pcolormesh(
            X,
            Y,
            data,
            cmap=cmaps[field],
            norm=norms[field],
            shading="auto",
            rasterized=True
        )

        # label position
        theta_c = 0.5 * (theta0 + theta1)
        r_lab = Dmax + spiral_pitch * (i + 0.6)

        ax.text(
            r_lab * np.cos(theta_c),
            r_lab * np.sin(theta_c),
            labels[field],
            fontsize=11,
            color=grey,
            ha="center",
            va="center",
            rotation=np.degrees(theta_c) - 90
        )

    # =================================================
    # Center label
    # =================================================

    ax.text(
        0,
        0,
        r"${\bf Spiral\ Lightcone}$",
        fontsize=14,
        color=grey,
        ha="center",
        va="center"
    )

    # =================================================
    # Final cosmetics
    # =================================================

    ax.set_aspect("equal")
    ax.axis("off")

    plt.tight_layout()
    plt.show()


# =====================================================
# Run
# =====================================================

rlc_dir = "/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_320"

plot_spiral_lightcone(
    rlc_dir=rlc_dir,
    SliceFraction=0.2,
    zmax=20.0
)

In [ ]:
%matplotlib inline

import numpy as np
import cmasher as cmr
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
import h5py

# =====================================================
# Labels / Colormaps
# =====================================================

labels = {
    "HII_Fraction": r"${\bf HII\ Fraction}$",
    "Tb21cm": r"${\bf HI\ 21cm}$",
    "StarFormationRate": r"${\bf Star\ Formation\ Rate}$",
    "Temperature": r"${\bf Temperature}$",
    "TbHe3": r"${\bf ^3HeII\ 3.46cm}$",
    "HeIII_Fraction": r"${\bf HeIII\ Fraction}$"
}

cmaps = {
    "HII_Fraction": cmr.amber,
    "Tb21cm":cmr.fusion, #cmr.bubblegum,
    "StarFormationRate": "inferno",
    "Temperature": "magma",
    "TbHe3": "coolwarm", #cmr.gem,
    "HeIII_Fraction": "BuPu" #cmr.flamingo
}

grey = [0.1, 0.1, 0.1]
Tcmb0 = 2.725  # K


# =====================================================
# Main function
# =====================================================

def plot_full_circle_lightcone(
    rlc_dir=".",
    SliceFraction=0.2,
    zmax=20.0
):

    # =================================================
    # Read file
    # =================================================

    with h5py.File(f"{rlc_dir}/All.hdf5", "r") as f:

        NumPixels = f["Header"].attrs["NumPixels"]

        base_fields = [
            "StarFormationRate",
            "Temperature",
            "HII_Fraction",
            "HeII_Fraction"
        ]

        SliceIndex = max(
            0,
            min(NumPixels - 1, int(SliceFraction * NumPixels))
        )

        ds = {
            field: f[field][SliceIndex, :, :]
            for field in base_fields
        }

        Redshifts = f["Redshifts"][:]   # usually N+1
        Distances = f["Distances"][:]   # usually N+1

    # =================================================
    # Restrict to z <= zmax
    # =================================================

    imax = np.argmin(np.abs(Redshifts - zmax))

    Redshifts = Redshifts[imax:]
    Distances = Distances[imax:]

    for field in ds:
        ds[field] = ds[field][:, imax:]

    # =================================================
    # Convert redshift edges -> centers
    # =================================================

    z_centers = 0.5 * (Redshifts[:-1] + Redshifts[1:])

    nz = ds["Temperature"].shape[1]

    z_centers = z_centers[:nz]

    # keep radial edges for pcolormesh
    Distances = Distances[:nz + 1]

    # =================================================
    # Extract fields
    # =================================================

    sfr = ds["StarFormationRate"]
    temperature = ds["Temperature"]
    xHII = ds["HII_Fraction"]
    xHeII = ds["HeII_Fraction"]

    xHI = 1.0 - xHII
    xHeIII = 1.0 - xHeII

    # use SFR as overdensity proxy
    mean_sfr = np.mean(sfr[sfr > 0]) if np.any(sfr > 0) else 1.0
    delta_b = np.where(sfr > 0, sfr / mean_sfr, 1.0)

    # redshift 2D
    z2d = np.tile(z_centers, (NumPixels, 1))

    # CMB temperature
    Tcmb = Tcmb0 * (1.0 + z2d)

    # assume Ts ~ Tgas
    Ts = np.maximum(temperature, Tcmb + 1e-5)

    coupling = 1.0 - Tcmb / Ts

    # =================================================
    # Brightness temperatures
    # =================================================

    Tb21 = (
        27.0
        * xHI
        * delta_b
        * np.sqrt((1.0 + z2d) / 10.0)
        * coupling
    )

    TbHe3 = (
        0.03
        * xHeII
        * delta_b
        * np.sqrt((1.0 + z2d) / 10.0)
        * coupling
    )

    # =================================================
    # Fields to plot
    # =================================================

    plot_fields = {
        "HII_Fraction": xHII,
        "Tb21cm": np.abs(Tb21),
        "StarFormationRate": sfr,
        "Temperature": temperature,
        "TbHe3": np.abs(TbHe3),
        "HeIII_Fraction": xHeIII
    }

    fields = [
        "HII_Fraction",
        "Tb21cm",
        "StarFormationRate",
        "Temperature",
        "TbHe3",
        "HeIII_Fraction"
    ]

    # =================================================
    # Robust norms
    # =================================================

    norms = {
        "HII_Fraction": Normalize(vmin=0.0, vmax=1.0),
        "HeIII_Fraction": Normalize(vmin=0.0, vmax=1.0)
    }

    for field in [
        "Tb21cm",
        "StarFormationRate",
        "Temperature",
        "TbHe3"
    ]:
        data = plot_fields[field]

        positive = data[
            np.isfinite(data) & (data > 0)
        ]

        if len(positive) < 10:
            vmin = 1e-6
            vmax = 1.0
        else:
            vmin = np.percentile(positive, 5)
            vmax = np.percentile(positive, 99)

            vmin = max(vmin, 1e-6)

            if vmax <= vmin:
                vmax = vmin * 10.0

        norms[field] = LogNorm(
            vmin=vmin,
            vmax=vmax,
            clip=True
        )

        print(
            field,
            "vmin =", vmin,
            "vmax =", vmax
        )

    # =================================================
    # Full circle geometry (360 degrees)
    # =================================================

    n_fields = len(fields)

    Theta_E = np.linspace(
        0.0,
        2.0 * np.pi,
        n_fields + 1
    )

    Theta_C = 0.5 * (Theta_E[:-1] + Theta_E[1:])

    Dmin = 0.0
    Dmax = Distances[0]

    theta = np.linspace(
        0.0,
        2.0 * np.pi,
        1000
    )

    Dlab = 0.82 * Dmax

    # =================================================
    # Figure
    # =================================================

    fig, ax = plt.subplots(figsize=(12, 12))

    for i, field in enumerate(fields):

        Thetas = np.linspace(
            Theta_E[i],
            Theta_E[i + 1],
            NumPixels + 1
        )

        # radial edges
        Rvals = np.linspace(
            Dmax,
            Dmin,
            len(z_centers) + 1
        )

        Theta2d, R2d = np.meshgrid(
            Thetas,
            Rvals,
            indexing="ij"
        )

        X = R2d * np.cos(Theta2d)
        Y = R2d * np.sin(Theta2d)

        data = plot_fields[field]

        ax.pcolormesh(
            X,
            Y,
            data,
            cmap=cmaps[field],
            norm=norms[field],
            shading="auto",
            rasterized=True
        )

    # =================================================
    # Boundaries
    # =================================================

    for i in range(n_fields + 1):
        ax.plot(
            [0, Dmax * np.cos(Theta_E[i])],
            [0, Dmax * np.sin(Theta_E[i])],
            c=grey,
            lw=1
        )

    ax.plot(
        Dmax * np.cos(theta),
        Dmax * np.sin(theta),
        c=grey,
        lw=1
    )

    # =================================================
    # Labels
    # =================================================

    for i, field in enumerate(fields):

        angle_deg = np.degrees(Theta_C[i])

        # nicer text rotation
        rotation = angle_deg - 90
        if 90 < angle_deg < 270:
            rotation += 180

        ax.text(
            Dlab * np.cos(Theta_C[i]),
            Dlab * np.sin(Theta_C[i]),
            labels[field],
            ha="center",
            va="center",
            fontsize=11,
            color=grey,
            rotation=rotation
        )

    ax.text(
        0,
        0,
        r"${\bf Full\ Circle\ Lightcone}$",
        ha="center",
        va="center",
        fontsize=14,
        color=grey
    )

    # =================================================
    # Final cosmetics
    # =================================================

    ax.set_aspect("equal")
    ax.axis("off")

    plt.tight_layout()
    plt.show()


# =====================================================
# Run
# =====================================================

rlc_dir = "/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_320"

plot_full_circle_lightcone(
    rlc_dir=rlc_dir,
    SliceFraction=0.2,
    zmax=20.0
)

In [ ]:
%matplotlib inline

import numpy as np
import cmasher as cmr
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
import h5py

# =====================================================
# Labels / Colormaps
# =====================================================

labels = {
    "HII_Fraction": r"${\bf HII\ Fraction}$",
    "Tb21cm": r"${\bf HI\ 21cm}$",
    "Temperature": r"${\bf Temperature}$",
    "HeIII_Fraction": r"${\bf HeIII\ Fraction}$",
    "TbHe3": r"${\bf ^3HeII\ 3.46cm}$"
}

cmaps = {
    "HII_Fraction": cmr.amber,
    "Tb21cm": cmr.fusion,
    "Temperature": "magma",
    "HeIII_Fraction": "BuPu",
    "TbHe3": "coolwarm"
}

grey = [0.1, 0.1, 0.1]
Tcmb0 = 2.725  # K


# =====================================================
# Main function
# =====================================================

def plot_300deg_lightcone(
    rlc_dir=".",
    SliceFraction=0.2,
    zmax=20.0
):

    # =================================================
    # Read file
    # =================================================

    with h5py.File(f"{rlc_dir}/All.hdf5", "r") as f:

        NumPixels = f["Header"].attrs["NumPixels"]

        base_fields = [
            "Temperature",
            "HII_Fraction",
            "HeII_Fraction"
        ]

        SliceIndex = max(
            0,
            min(NumPixels - 1, int(SliceFraction * NumPixels))
        )

        ds = {
            field: f[field][SliceIndex, :, :]
            for field in base_fields
        }

        Redshifts = f["Redshifts"][:]   # usually N+1
        Distances = f["Distances"][:]   # usually N+1

    # =================================================
    # Restrict to z <= zmax
    # =================================================

    imax = np.argmin(np.abs(Redshifts - zmax))

    Redshifts = Redshifts[imax:]
    Distances = Distances[imax:]

    for field in ds:
        ds[field] = ds[field][:, imax:]

    # =================================================
    # Convert redshift edges -> centers
    # =================================================

    z_centers = 0.5 * (Redshifts[:-1] + Redshifts[1:])

    nz = ds["Temperature"].shape[1]

    z_centers = z_centers[:nz]
    Distances = Distances[:nz + 1]  # keep edges for pcolormesh

    # =================================================
    # Extract fields
    # =================================================

    temperature = ds["Temperature"]
    xHII = ds["HII_Fraction"]
    xHeII = ds["HeII_Fraction"]

    xHI = 1.0 - xHII
    xHeIII = 1.0 - xHeII

    # simple overdensity proxy
    mean_xHI = np.mean(xHI[xHI > 0]) if np.any(xHI > 0) else 1.0
    delta_b = np.where(xHI > 0, xHI / mean_xHI, 1.0)

    # redshift 2D
    z2d = np.tile(z_centers, (NumPixels, 1))

    # CMB temperature
    Tcmb = Tcmb0 * (1.0 + z2d)

    # assume spin temperature ~ gas temperature
    Ts = np.maximum(temperature, Tcmb + 1e-5)

    coupling = 1.0 - Tcmb / Ts

    # =================================================
    # Brightness temperatures
    # =================================================

    # HI 21 cm brightness temperature [mK]
    Tb21 = (
        27.0
        * xHI
        * delta_b
        * np.sqrt((1.0 + z2d) / 10.0)
        * coupling
    )

    # 3HeII brightness temperature [mK]
    TbHe3 = (
        0.03
        * xHeII
        * delta_b
        * np.sqrt((1.0 + z2d) / 10.0)
        * coupling
    )

    # =================================================
    # Fields to plot
    # =================================================

    plot_fields = {
        "HII_Fraction": xHII,
        "Tb21cm": np.abs(Tb21),
        "Temperature": temperature,
        "HeIII_Fraction": xHeIII,
        "TbHe3": np.abs(TbHe3)
    }

    fields = [
        "HII_Fraction",
        "Tb21cm",
        "Temperature",
        "HeIII_Fraction",
        "TbHe3"
    ]

    # =================================================
    # Norms
    # =================================================

    norms = {
        "HII_Fraction": Normalize(vmin=0.0, vmax=1.0),
        "HeIII_Fraction": Normalize(vmin=0.0, vmax=1.0)
    }

    for field in [
        "Tb21cm",
        "Temperature",
        "TbHe3"
    ]:
        data = plot_fields[field]

        positive = data[
            np.isfinite(data) & (data > 0)
        ]

        if len(positive) < 10:
            vmin = 1e-6
            vmax = 1.0
        else:
            vmin = np.percentile(positive, 5)
            vmax = np.percentile(positive, 99)

            vmin = max(vmin, 1e-6)

            if vmax <= vmin:
                vmax = vmin * 10.0

        norms[field] = LogNorm(
            vmin=vmin,
            vmax=vmax,
            clip=True
        )

        print(field, "vmin =", vmin, "vmax =", vmax)

    # =================================================
    # 300-degree geometry
    #
    # blank opening on RIGHT side
    # =================================================

    n_fields = len(fields)

    opening_angle = np.radians(300.0)

    # center filled region on LEFT side
    # so blank 60-degree opening is on RIGHT
    theta_center = np.pi

    Theta_E = np.linspace(
        theta_center - opening_angle / 2.0,
        theta_center + opening_angle / 2.0,
        n_fields + 1
    )

    Theta_C = 0.5 * (Theta_E[:-1] + Theta_E[1:])

    Dmax = Distances[0]

    theta = np.linspace(
        Theta_E[0],
        Theta_E[-1],
        1000
    )

    Dlab = 0.84 * Dmax

    # =================================================
    # Figure
    # =================================================

    fig, ax = plt.subplots(figsize=(12, 12))

    for i, field in enumerate(fields):

        Thetas = np.linspace(
            Theta_E[i],
            Theta_E[i + 1],
            NumPixels + 1
        )

        Rvals = np.linspace(
            Dmax,
            0.0,
            len(z_centers) + 1
        )

        Theta2d, R2d = np.meshgrid(
            Thetas,
            Rvals,
            indexing="ij"
        )

        X = R2d * np.cos(Theta2d)
        Y = R2d * np.sin(Theta2d)

        data = plot_fields[field]

        ax.pcolormesh(
            X,
            Y,
            data,
            cmap=cmaps[field],
            norm=norms[field],
            shading="auto",
            rasterized=True
        )

    # =================================================
    # Boundaries
    # =================================================

    for i in range(n_fields + 1):
        ax.plot(
            [0, Dmax * np.cos(Theta_E[i])],
            [0, Dmax * np.sin(Theta_E[i])],
            c=grey,
            lw=1
        )

    ax.plot(
        Dmax * np.cos(theta),
        Dmax * np.sin(theta),
        c=grey,
        lw=1
    )

    # =================================================
    # Labels
    # =================================================

    for i, field in enumerate(fields):

        angle_deg = np.degrees(Theta_C[i]) % 360

        rotation = angle_deg - 90
        if 90 < angle_deg < 270:
            rotation += 180

        ax.text(
            Dlab * np.cos(Theta_C[i]),
            Dlab * np.sin(Theta_C[i]),
            labels[field],
            ha="center",
            va="center",
            fontsize=12,
            color=grey,
            rotation=rotation
        )

    ax.text(
        0,
        0,
        r"${\bf 300^\circ\ Lightcone}$",
        ha="center",
        va="center",
        fontsize=14,
        color=grey
    )

    # =================================================
    # Final cosmetics
    # =================================================

    ax.set_aspect("equal")
    ax.axis("off")

    plt.tight_layout()
    plt.show()


# =====================================================
# Run
# =====================================================

rlc_dir = "/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_320"

plot_300deg_lightcone(
    rlc_dir=rlc_dir,
    SliceFraction=0.2,
    zmax=20.0
)

In [ ]:
%matplotlib inline

import numpy as np
import cmasher as cmr
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import h5py

# =====================================================
# Labels / Colormaps
# =====================================================

labels = {
    "Tb21cm": r"${\bf HI\ 21cm}$",
    "Temperature": r"${\bf Temperature}$",
    "TbHe3": r"${\bf ^3HeII\ 3.46cm}$"
}

cmaps = {
    "Tb21cm": cmr.fusion,
    "Temperature": "magma",
    "TbHe3": "coolwarm"
}

grey = [0.1, 0.1, 0.1]
Tcmb0 = 2.725  # K


# =====================================================
# Main function
# =====================================================

def plot_hive_lightcone(rlc_dir=".", SliceFraction=0.2, zmax=20.0):

    # =================================================
    # Read file
    # =================================================

    with h5py.File(f"{rlc_dir}/All.hdf5", "r") as f:

        NumPixels = f["Header"].attrs["NumPixels"]

        base_fields = [
            "Temperature",
            "HII_Fraction",
            "HeII_Fraction"
        ]

        SliceIndex = int(SliceFraction * NumPixels)

        ds = {
            field: f[field][SliceIndex, :, :]
            for field in base_fields
        }

        Redshifts = f["Redshifts"][:]
        Distances = f["Distances"][:]

    # =================================================
    # Restrict to z <= zmax
    # =================================================

    imax = np.argmin(np.abs(Redshifts - zmax))

    Redshifts = Redshifts[imax:]
    Distances = Distances[imax:]

    for field in ds:
        ds[field] = ds[field][:, imax:]

    # =================================================
    # Centers
    # =================================================

    z_centers = 0.5 * (Redshifts[:-1] + Redshifts[1:])
    nz = ds["Temperature"].shape[1]

    z_centers = z_centers[:nz]
    Distances = Distances[:nz + 1]

    # =================================================
    # Extract fields
    # =================================================

    temperature = ds["Temperature"]
    xHII = ds["HII_Fraction"]
    xHeII = ds["HeII_Fraction"]

    xHI = 1.0 - xHII

    # overdensity proxy
    mean_xHI = np.mean(xHI[xHI > 0]) if np.any(xHI > 0) else 1.0
    delta_b = np.where(xHI > 0, xHI / mean_xHI, 1.0)

    z2d = np.tile(z_centers, (NumPixels, 1))

    Tcmb = Tcmb0 * (1.0 + z2d)
    Ts = np.maximum(temperature, Tcmb + 1e-5)

    coupling = 1.0 - Tcmb / Ts

    # =================================================
    # Brightness temperatures
    # =================================================

    Tb21 = (
        27.0
        * xHI
        * delta_b
        * np.sqrt((1.0 + z2d) / 10.0)
        * coupling
    )

    TbHe3 = (
        0.03
        * xHeII
        * delta_b
        * np.sqrt((1.0 + z2d) / 10.0)
        * coupling
    )

    # =================================================
    # Fields to plot
    # =================================================

    plot_fields = {
        "Tb21cm": np.abs(Tb21),
        "Temperature": temperature,
        "TbHe3": np.abs(TbHe3)
    }

    fields = ["Tb21cm", "Temperature", "TbHe3"]

    # =================================================
    # Log norms
    # =================================================

    norms = {}

    for field in fields:
        data = plot_fields[field]
        positive = data[np.isfinite(data) & (data > 0)]

        if len(positive) < 10:
            vmin, vmax = 1e-6, 1.0
        else:
            vmin = max(np.percentile(positive, 5), 1e-6)
            vmax = np.percentile(positive, 99)

        norms[field] = LogNorm(vmin=vmin, vmax=vmax)

        print(field, "vmin =", vmin, "vmax =", vmax)

    # =================================================
    # Hive geometry (3 sectors)
    # =================================================

    n_fields = len(fields)

    Theta_E = np.linspace(0, 2*np.pi, n_fields + 1)
    Theta_C = 0.5 * (Theta_E[:-1] + Theta_E[1:])

    Dmax = Distances[0]
    Dlab = 0.75 * Dmax

    # =================================================
    # Plot
    # =================================================

    fig, ax = plt.subplots(figsize=(10, 10))

    for i, field in enumerate(fields):

        Thetas = np.linspace(
            Theta_E[i],
            Theta_E[i+1],
            NumPixels + 1
        )

        Rvals = Distances[::-1]  # physical radial axis

        Theta2d, R2d = np.meshgrid(
            Thetas,
            Rvals,
            indexing="ij"
        )

        X = R2d * np.cos(Theta2d)
        Y = R2d * np.sin(Theta2d)

        ax.pcolormesh(
            X,
            Y,
            plot_fields[field],
            cmap=cmaps[field],
            norm=norms[field],
            shading="auto",
            rasterized=True
        )

    # =================================================
    # Boundaries
    # =================================================

    theta = np.linspace(0, 2*np.pi, 1000)

    for t in Theta_E:
        ax.plot([0, Dmax*np.cos(t)], [0, Dmax*np.sin(t)], c=grey, lw=1)

    ax.plot(Dmax*np.cos(theta), Dmax*np.sin(theta), c=grey, lw=1)

    # =================================================
    # Labels
    # =================================================

    for i, field in enumerate(fields):

        angle_deg = np.degrees(Theta_C[i]) % 360
        rotation = angle_deg - 90

        if 90 < angle_deg < 270:
            rotation += 180

        ax.text(
            Dlab * np.cos(Theta_C[i]),
            Dlab * np.sin(Theta_C[i]),
            labels[field],
            ha="center",
            va="center",
            fontsize=12,
            color=grey,
            rotation=rotation
        )

    ax.text(
        0, 0,
        r"${\bf Hive\ Lightcone}$",
        ha="center",
        va="center",
        fontsize=14,
        color=grey
    )

    # =================================================
    # Final
    # =================================================

    ax.set_aspect("equal")
    ax.axis("off")

    plt.tight_layout()
    plt.show()


# =====================================================
# Run
# =====================================================

rlc_dir = "/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_320"

plot_hive_lightcone(
    rlc_dir=rlc_dir,
    SliceFraction=.2,
    zmax=20.0
)

In [ ]:
%matplotlib inline

import numpy as np
import cmasher as cmr
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
import h5py

# =====================================================
# Labels / Colormaps
# =====================================================

labels = {
    "Tb21cm": r"${\bf HI\ 21cm}$",
    "TbHe3": r"${\bf ^3HeII\ 3.46cm}$",
    "HII_Fraction": r"${\bf HII}$",
    "HeIII_Fraction": r"${\bf HeIII}$"
}

cmaps = {
    "Tb21cm": cmr.fusion,
    "TbHe3": "coolwarm",
    "HII_Fraction": cmr.amber,
    "HeIII_Fraction": "BuPu"
}

grey = [0.1, 0.1, 0.1]
Tcmb0 = 2.725


# =====================================================
# Main
# =====================================================

def plot_custom_hive(rlc_dir=".", SliceFraction=0.2, zmax=20.0):

    # -------------------------------
    # Load data
    # -------------------------------
    with h5py.File(f"{rlc_dir}/All.hdf5", "r") as f:

        NumPixels = f["Header"].attrs["NumPixels"]

        fields = ["Temperature", "HII_Fraction", "HeII_Fraction"]

        SliceIndex = int(SliceFraction * NumPixels)

        ds = {k: f[k][SliceIndex, :, :] for k in fields}

        Redshifts = f["Redshifts"][:]
        Distances = f["Distances"][:]

    # -------------------------------
    # Restrict z
    # -------------------------------
    imax = np.argmin(np.abs(Redshifts - zmax))

    Redshifts = Redshifts[imax:]
    Distances = Distances[imax:]

    for k in ds:
        ds[k] = ds[k][:, imax:]

    # -------------------------------
    # Centers
    # -------------------------------
    z_centers = 0.5 * (Redshifts[:-1] + Redshifts[1:])
    nz = ds["Temperature"].shape[1]

    z_centers = z_centers[:nz]
    Distances = Distances[:nz + 1]

    # -------------------------------
    # Physics
    # -------------------------------
    temperature = ds["Temperature"]
    xHII = ds["HII_Fraction"]
    xHeII = ds["HeII_Fraction"]

    xHI = 1.0 - xHII
    xHeIII = 1.0 - xHeII

    mean_xHI = np.mean(xHI[xHI > 0]) if np.any(xHI > 0) else 1.0
    delta_b = np.where(xHI > 0, xHI / mean_xHI, 1.0)

    z2d = np.tile(z_centers, (NumPixels, 1))

    Tcmb = Tcmb0 * (1 + z2d)
    Ts = np.maximum(temperature, Tcmb + 1e-5)

    coupling = 1 - Tcmb / Ts

    Tb21 = 27.0 * xHI * delta_b * np.sqrt((1+z2d)/10.0) * coupling
    TbHe3 = 0.03 * xHeII * delta_b * np.sqrt((1+z2d)/10.0) * coupling

    plot_fields = {
        "Tb21cm": np.abs(Tb21),
        "TbHe3": np.abs(TbHe3),
        "HII_Fraction": xHII,
        "HeIII_Fraction": xHeIII
    }

    # -------------------------------
    # Norms
    # -------------------------------
    norms = {}

    for f in ["Tb21cm", "TbHe3"]:
        data = plot_fields[f]
        pos = data[np.isfinite(data) & (data > 0)]

        vmin = max(np.percentile(pos, 5), 1e-6)
        vmax = np.percentile(pos, 99)

        norms[f] = LogNorm(vmin=vmin, vmax=vmax)

    norms["HII_Fraction"] = Normalize(0, 1)
    norms["HeIII_Fraction"] = Normalize(0, 1)

    # -------------------------------
    # Custom geometry (key part)
    # -------------------------------
    # Angles (radians)
    # Left: 21cm
    # Right: 3He
    # Top split: HII / HeIII

    Theta_E = [
        -np.pi/2,        # bottom
        0,               # right-bottom
        np.pi/2,         # right-top
        3*np.pi/4,       # split
        np.pi,           # left-top
        3*np.pi/2        # back to bottom
    ]

    sector_defs = [
        ("Tb21cm", Theta_E[4], Theta_E[5]),        # left-bottom
        ("TbHe3", Theta_E[0], Theta_E[1]),         # right-bottom
        ("HII_Fraction", Theta_E[2], Theta_E[3]),  # top-left half
        ("HeIII_Fraction", Theta_E[3], Theta_E[4]) # top-right half
    ]

    Dmax = Distances[0]

    # -------------------------------
    # Plot
    # -------------------------------
    fig, ax = plt.subplots(figsize=(10,10))

    for field, t0, t1 in sector_defs:

        Thetas = np.linspace(t0, t1, NumPixels + 1)
        Rvals = Distances[::-1]

        Theta2d, R2d = np.meshgrid(Thetas, Rvals, indexing="ij")

        X = R2d * np.cos(Theta2d)
        Y = R2d * np.sin(Theta2d)

        ax.pcolormesh(
            X, Y,
            plot_fields[field],
            cmap=cmaps[field],
            norm=norms[field],
            shading="auto",
            rasterized=True
        )

        # label
        tc = 0.5*(t0+t1)
        ax.text(
            0.75*Dmax*np.cos(tc),
            0.75*Dmax*np.sin(tc),
            labels[field],
            ha="center", va="center",
            fontsize=12,
            color=grey
        )

    # -------------------------------
    # Boundaries
    # -------------------------------
    theta = np.linspace(0, 2*np.pi, 500)
    ax.plot(Dmax*np.cos(theta), Dmax*np.sin(theta), color=grey, lw=1)

    for t in Theta_E:
        ax.plot([0, Dmax*np.cos(t)], [0, Dmax*np.sin(t)], color=grey, lw=1)

    # -------------------------------
    # Final
    # -------------------------------
    ax.set_aspect("equal")
    ax.axis("off")

    plt.tight_layout()
    plt.show()


# =====================================================
# Run
# =====================================================

rlc_dir = "/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_320"

plot_custom_hive(
    rlc_dir=rlc_dir,
    SliceFraction=0.2,
    zmax=20.0
)

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
import cmasher as cmr
import h5py

Tcmb0 = 2.725

def plot_true_hive(rlc_dir, SliceFraction=0.2, zmax=20):

    # =============================
    # Load data
    # =============================
    with h5py.File(f"{rlc_dir}/All.hdf5", "r") as f:
        N = f["Header"].attrs["NumPixels"]
        i = int(SliceFraction * N)

        T = f["Temperature"][i]
        xHII = f["HII_Fraction"][i]
        xHeII = f["HeII_Fraction"][i]

        z = f["Redshifts"][:]
        D = f["Distances"][:]

    # =============================
    # Trim z
    # =============================
    imax = np.argmin(np.abs(z - zmax))

    z = z[imax:]
    D = D[imax:]

    T = T[:, imax:]
    xHII = xHII[:, imax:]
    xHeII = xHeII[:, imax:]

    zc = 0.5 * (z[:-1] + z[1:])
    nz = T.shape[1]

    zc = zc[:nz]
    D = D[:nz+1]

    z2d = np.tile(zc, (N,1))

    xHI = 1 - xHII
    xHeIII = 1 - xHeII

    delta = xHI / np.mean(xHI[xHI > 0])

    Tcmb = Tcmb0 * (1 + z2d)
    Ts = np.maximum(T, Tcmb + 1e-5)

    coupling = 1 - Tcmb / Ts

    Tb21 = 27 * xHI * delta * np.sqrt((1+z2d)/10) * coupling
    TbHe3 = 0.03 * xHeII * delta * np.sqrt((1+z2d)/10) * coupling

    fields = {
        "21cm": np.abs(Tb21),
        "3He": np.abs(TbHe3),
        "HII": xHII,
        "HeIII": xHeIII
    }

    # =============================
    # Colormaps & norms
    # =============================
    cmaps = {
        "21cm": cmr.fusion,
        "3He": "coolwarm",
        "HII": cmr.amber,
        "HeIII": "BuPu"
    }

    norms = {
        "21cm": LogNorm(vmin=1e-2, vmax=np.percentile(fields["21cm"],99)),
        "3He": LogNorm(vmin=1e-5, vmax=np.percentile(fields["3He"],99)),
        "HII": Normalize(0,1),
        "HeIII": Normalize(0,1)
    }

    # =============================
    # Geometry (THIS is the core)
    # =============================

    gap = np.radians(5)   # small gap between sectors
    sector = 2*np.pi / 3  # 120°

    # centers of 3 sectors (Y shape)
    centers = [
        -2*np.pi/3,   # lower-left
        0,            # lower-right
        2*np.pi/3     # top
    ]

    # define sectors with gaps
    sectors = []

    # lower-left (3He)
    c = centers[0]
    sectors.append(("3He",
        c - sector/2 + gap/2,
        c + sector/2 - gap/2))

    # lower-right (21cm)
    c = centers[1]
    sectors.append(("21cm",
        c - sector/2 + gap/2,
        c + sector/2 - gap/2))

    # top split into two (60° each)
    c = centers[2]

    t0 = c - sector/2 + gap/2
    t1 = c + sector/2 - gap/2

    mid = 0.5*(t0 + t1)

    sectors.append(("HeIII", t0, mid))
    sectors.append(("HII", mid, t1))

    # =============================
    # Radial (annulus)
    # =============================

    R_outer = D[0]
    R_inner = 0.1 * R_outer

    Rvals = np.linspace(R_outer, R_inner, len(zc)+1)

    # =============================
    # Plot
    # =============================

    fig, ax = plt.subplots(figsize=(10,10))

    def draw(data, t0, t1, cmap, norm):
        theta = np.linspace(t0, t1, N+1)
        TH, RR = np.meshgrid(theta, Rvals, indexing='ij')

        X = RR * np.cos(TH)
        Y = RR * np.sin(TH)

        ax.pcolormesh(X, Y, data, cmap=cmap, norm=norm, shading='auto')

    for name, t0, t1 in sectors:
        draw(fields[name], t0, t1, cmaps[name], norms[name])

    # =============================
    # Boundaries
    # =============================

    theta = np.linspace(0, 2*np.pi, 500)

    ax.plot(R_outer*np.cos(theta), R_outer*np.sin(theta), 'k', lw=1)
    ax.plot(R_inner*np.cos(theta), R_inner*np.sin(theta), 'k', lw=1)

    for _, t0, t1 in sectors:
        ax.plot([0, R_outer*np.cos(t0)], [0, R_outer*np.sin(t0)], color='k', lw=2)
        ax.plot([0, R_outer*np.cos(t1)], [0, R_outer*np.sin(t1)], color='k', lw=2)

    ax.set_aspect('equal')
    ax.axis('off')

    plt.tight_layout()
    plt.show()


# RUN
plot_true_hive("/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_320")

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
import cmasher as cmr
import h5py

Tcmb0 = 2.725

def plot_final_hive(rlc_dir, SliceFraction=0.2, zmax=20):

    # =============================
    # Load
    # =============================
    with h5py.File(f"{rlc_dir}/All.hdf5", "r") as f:
        N = f["Header"].attrs["NumPixels"]
        i = int(SliceFraction * N)

        T = f["Temperature"][i]
        xHII = f["HII_Fraction"][i]
        xHeII = f["HeII_Fraction"][i]

        z = f["Redshifts"][:]
        D = f["Distances"][:]

    # =============================
    # Trim
    # =============================
    imax = np.argmin(np.abs(z - zmax))

    z = z[imax:]
    D = D[imax:]

    T = T[:, imax:]
    xHII = xHII[:, imax:]
    xHeII = xHeII[:, imax:]

    zc = 0.5 * (z[:-1] + z[1:])
    nz = T.shape[1]

    zc = zc[:nz]
    D = D[:nz+1]

    z2d = np.tile(zc, (N,1))

    # =============================
    # Physics
    # =============================
    xHI = 1 - xHII
    xHeIII = 1 - xHeII

    delta = xHI / np.mean(xHI[xHI > 0])

    Tcmb = Tcmb0 * (1 + z2d)
    Ts = np.maximum(T, Tcmb + 1e-5)

    coupling = 1 - Tcmb / Ts

    Tb21 = 27 * xHI * delta * np.sqrt((1+z2d)/10) * coupling
    TbHe3 = 0.03 * xHeII * delta * np.sqrt((1+z2d)/10) * coupling

    fields = {
        "21cm": np.abs(Tb21),
        "3He": np.abs(TbHe3),
        "HII": xHII,
        "HeIII": xHeIII
    }

    # =============================
    # Style
    # =============================
    cmaps = {
        "21cm": cmr.fusion,
        "3He": "coolwarm",
        "HII": cmr.amber,
        "HeIII": "BuPu"
    }

    norms = {
        "21cm": LogNorm(vmin=1e-2, vmax=np.percentile(fields["21cm"],99)),
        "3He": LogNorm(vmin=1e-5, vmax=np.percentile(fields["3He"],99)),
        "HII": Normalize(0,1),
        "HeIII": Normalize(0,1)
    }

    # =============================
    # Geometry
    # =============================

    deg = np.pi/180
    gap = 6 * deg

    w120 = 120 * deg
    w60  = 60 * deg

    # Start from bottom (270°) and go CCW
    base = 270 * deg

    # Lower-left (3He)
    t0_3He = base
    t1_3He = t0_3He + w120 - gap

    # Lower-right (21cm)
    t0_21 = t1_3He + gap
    t1_21 = t0_21 + w120 - gap

    # Top sector spans full 120° (no internal gap)
    t0_top = t1_21 + gap
    t1_top = t0_top + w120 - gap
    
    # Split cleanly into two equal halves
    t_mid = 0.5 * (t0_top + t1_top)
    
    t0_HeIII = t0_top
    t1_HeIII = t_mid
    
    t0_HII = t_mid
    t1_HII = t1_top

    rotation_deg = -117   # example
    rot = np.radians(rotation_deg)

    sectors = [
        ("3He", t0_3He + rot, t1_3He + rot),
        ("21cm", t0_21 + rot, t1_21 + rot),
        ("HeIII", t0_HeIII + rot, t1_HeIII + rot),
        ("HII", t0_HII + rot, t1_HII + rot)
    ]

    # =============================
    # Radial
    # =============================
    R_outer = D[0]
    R_inner = 0.1 * R_outer

    Rvals = np.linspace(R_outer, R_inner, len(zc)+1)

    # =============================
    # Plot
    # =============================
    fig, ax = plt.subplots(figsize=(10,10))

    def draw(data, t0, t1, cmap, norm):
        theta = np.linspace(t0, t1, N+1)
        TH, RR = np.meshgrid(theta, Rvals, indexing='ij')

        X = RR*np.cos(TH)
        Y = RR*np.sin(TH)

        ax.pcolormesh(X, Y, data, cmap=cmap, norm=norm, shading='auto')

    for name, t0, t1 in sectors:
        draw(fields[name], t0, t1, cmaps[name], norms[name])

    # =============================
    # Boundaries
    # =============================
    theta = np.linspace(0, 2*np.pi, 500)

    ax.plot(R_outer*np.cos(theta), R_outer*np.sin(theta), color='k', lw=1)
    ax.plot(R_inner*np.cos(theta), R_inner*np.sin(theta), color='k', lw=1)

    # Thick white separators (Y-like)
    for _, t0, t1 in sectors:
        ax.plot([0, R_outer*np.cos(t0)], [0, R_outer*np.sin(t0)],
                color='white', lw=6)
        ax.plot([0, R_outer*np.cos(t1)], [0, R_outer*np.sin(t1)],
                color='white', lw=6)

    ax.set_aspect('equal')
    ax.axis('off')

    plt.tight_layout()
    plt.show()


# RUN
plot_final_hive("/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_320")

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
import cmasher as cmr
import h5py

Tcmb0 = 2.725


def plot_hive(rlc_dir, SliceFraction=0.2, zmax=20.0, rotation_deg=0):

    # =====================================================
    # Load data
    # =====================================================
    with h5py.File(f"{rlc_dir}/All.hdf5", "r") as f:
        N = f["Header"].attrs["NumPixels"]
        i = int(SliceFraction * N)

        T = f["Temperature"][i]
        xHII = f["HII_Fraction"][i]
        xHeII = f["HeII_Fraction"][i]

        z = f["Redshifts"][:]
        D = f["Distances"][:]

    # =====================================================
    # Restrict redshift
    # =====================================================
    imax = np.argmin(np.abs(z - zmax))

    z = z[imax:]
    D = D[imax:]

    T = T[:, imax:]
    xHII = xHII[:, imax:]
    xHeII = xHeII[:, imax:]

    zc = 0.5 * (z[:-1] + z[1:])
    nz = T.shape[1]

    zc = zc[:nz]
    D = D[:nz + 1]

    z2d = np.tile(zc, (N, 1))

    # =====================================================
    # Physics
    # =====================================================
    xHI = 1 - xHII
    xHeIII = 1 - xHeII

    delta = xHI / np.mean(xHI[xHI > 0])

    Tcmb = Tcmb0 * (1 + z2d)
    Ts = np.maximum(T, Tcmb + 1e-5)

    coupling = 1 - Tcmb / Ts

    Tb21 = 27 * xHI * delta * np.sqrt((1 + z2d) / 10) * coupling
    TbHe3 = 0.03 * xHeII * delta * np.sqrt((1 + z2d) / 10) * coupling

    fields = {
        "21cm": np.abs(Tb21),
        "3He": np.abs(TbHe3),
        "HII": xHII,
        "HeIII": xHeIII
    }

    # =====================================================
    # Colormaps / norms
    # =====================================================
    cmaps = {
        "21cm": cmr.fusion,
        "3He": "coolwarm",
        "HII": cmr.amber,
        "HeIII": "BuPu"
    }

    norms = {
        "21cm": LogNorm(vmin=1e-2, vmax=np.percentile(fields["21cm"], 99)),
        "3He": LogNorm(vmin=1e-5, vmax=np.percentile(fields["3He"], 99)),
        "HII": Normalize(0, 1),
        "HeIII": Normalize(0, 1)
    }

    # =====================================================
    # Geometry (CORRECT)
    # =====================================================
    deg = np.pi / 180
    gap = 6 * deg
    rot = rotation_deg * deg

    # each sector EXACTLY 120° after removing gaps
    w = (2 * np.pi - 3 * gap) / 3

    base = 270 * deg + rot

    # 3 main sectors
    t0_A = base
    t1_A = t0_A + w

    t0_B = t1_A + gap
    t1_B = t0_B + w

    t0_C = t1_B + gap
    t1_C = t0_C + w

    # split top sector (NO GAP HERE)
    mid_C = 0.5 * (t0_C + t1_C)

    sectors = [
        ("3He", t0_A, t1_A),      # lower-left
        ("21cm", t0_B, t1_B),     # lower-right
        ("HeIII", t0_C, mid_C),   # upper-left
        ("HII", mid_C, t1_C)      # upper-right
    ]

    # =====================================================
    # Radial (annulus)
    # =====================================================
    R_outer = D[0]
    R_inner = 0.1 * R_outer

    Rvals = np.linspace(R_outer, R_inner, len(zc) + 1)

    # =====================================================
    # Plot
    # =====================================================
    fig, ax = plt.subplots(figsize=(10, 10))

    def draw(data, t0, t1, cmap, norm):
        theta = np.linspace(t0, t1, N + 1)
        TH, RR = np.meshgrid(theta, Rvals, indexing='ij')

        X = RR * np.cos(TH)
        Y = RR * np.sin(TH)

        ax.pcolormesh(
            X, Y, data,
            cmap=cmap,
            norm=norm,
            shading='auto',
            edgecolors='none'
        )

    for name, t0, t1 in sectors:
        draw(fields[name], t0, t1, cmaps[name], norms[name])

    # =====================================================
    # Boundaries (ONLY 3 gaps)
    # =====================================================
    boundaries = [t0_A, t1_A, t1_B, t1_C]

    def draw_gap(t_center, width):
        theta = np.linspace(t_center - width/2, t_center + width/2, 200)
        r_outer = R_outer
        r_inner = R_inner
    
        TH, RR = np.meshgrid(theta, [r_outer, r_inner], indexing='ij')
    
        X = RR * np.cos(TH)
        Y = RR * np.sin(TH)
    
        ax.fill(
            np.concatenate([X[:,0], X[::-1,1]]),
            np.concatenate([Y[:,0], Y[::-1,1]]),
            color='white',
            zorder=10
        )

    gap_centers = [
        t1_A + gap/2,
        t1_B + gap/2,
        t1_C + gap/2
    ]
    
    for gc in gap_centers:
        draw_gap(gc, gap)

    # outer & inner circle
    theta = np.linspace(0, 2*np.pi, 500)
    ax.plot(R_outer*np.cos(theta), R_outer*np.sin(theta), color='black', lw=1)
    ax.plot(R_inner*np.cos(theta), R_inner*np.sin(theta), color='black', lw=1)

    ax.set_aspect('equal')
    ax.axis('off')

    plt.tight_layout()
    plt.show()


# =====================================================
# RUN
# =====================================================
plot_hive(
    "/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_320",
    SliceFraction=0.1,
    zmax=20.0,
    rotation_deg=-117   # <-- change this to rotate
)

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
import cmasher as cmr
import h5py

Tcmb0 = 2.725


def plot_hive(rlc_dir, SliceFraction=0.2, zmax=20.0, rotation_deg=0):

    # =====================================================
    # Load data
    # =====================================================
    with h5py.File(f"{rlc_dir}/All.hdf5", "r") as f:
        N = f["Header"].attrs["NumPixels"]
        i = int(SliceFraction * N)

        T = f["Temperature"][i]
        xHII = f["HII_Fraction"][i]
        xHeII = f["HeII_Fraction"][i]

        z = f["Redshifts"][:]
        D = f["Distances"][:]

    # =====================================================
    # Restrict redshift
    # =====================================================
    imax = np.argmin(np.abs(z - zmax))

    z = z[imax:]
    D = D[imax:]

    T = T[:, imax:]
    xHII = xHII[:, imax:]
    xHeII = xHeII[:, imax:]

    zc = 0.5 * (z[:-1] + z[1:])
    nz = T.shape[1]

    zc = zc[:nz]
    D = D[:nz + 1]

    z2d = np.tile(zc, (N, 1))

    # =====================================================
    # Physics
    # =====================================================
    xHI = 1 - xHII
    xHeIII = 1 - xHeII

    delta = xHI / np.mean(xHI[xHI > 0])

    Tcmb = Tcmb0 * (1 + z2d)
    Ts = np.maximum(T, Tcmb + 1e-5)

    coupling = 1 - Tcmb / Ts

    Tb21 = 27 * xHI * delta * np.sqrt((1 + z2d) / 10) * coupling
    TbHe3 = 0.03 * xHeII * delta * np.sqrt((1 + z2d) / 10) * coupling

    fields = {
        "21cm": np.abs(Tb21),
        "3He": np.abs(TbHe3),
        "HII": xHII,
        "HeIII": xHeIII
    }

    # =====================================================
    # Colormaps / norms
    # =====================================================
    cmaps = {
        "21cm": cmr.fusion,
        "3He": "coolwarm",
        "HII": cmr.amber,
        "HeIII": "BuPu"
    }

    norms = {
        "21cm": LogNorm(vmin=1e-2, vmax=np.percentile(fields["21cm"], 99)),
        "3He": LogNorm(vmin=1e-5, vmax=np.percentile(fields["3He"], 99)),
        "HII": Normalize(0, 1),
        "HeIII": Normalize(0, 1)
    }

    # =====================================================
    # Geometry (CORRECT + CONSISTENT)
    # =====================================================
    deg = np.pi / 180
    gap = 6 * deg
    rot = rotation_deg * deg

    total = 2 * np.pi
    sector_width = (total - 3 * gap) / 3

    theta = 270 * deg + rot

    main_sectors = []
    gap_sectors = []

    for _ in range(3):
        t0 = theta
        t1 = t0 + sector_width
        main_sectors.append((t0, t1))

        g0 = t1
        g1 = g0 + gap
        gap_sectors.append((g0, g1))

        theta = g1

    # split top sector (NO GAP)
    t0_C, t1_C = main_sectors[2]
    mid_C = 0.5 * (t0_C + t1_C)

    sectors = [
        ("3He", main_sectors[0][0], main_sectors[0][1]),
        ("21cm", main_sectors[1][0], main_sectors[1][1]),
        ("HeIII", t0_C, mid_C),
        ("HII", mid_C, t1_C)
    ]

    # =====================================================
    # Radial (annulus)
    # =====================================================
    R_outer = D[0]
    R_inner = 0.1 * R_outer

    Rvals = np.linspace(R_outer, R_inner, len(zc) + 1)

    # =====================================================
    # Plot
    # =====================================================
    fig, ax = plt.subplots(figsize=(10, 10))

    def draw(data, t0, t1, cmap, norm):
        theta = np.linspace(t0, t1, N + 1)
        TH, RR = np.meshgrid(theta, Rvals, indexing='ij')

        X = RR * np.cos(TH)
        Y = RR * np.sin(TH)

        ax.pcolormesh(
            X, Y, data,
            cmap=cmap,
            norm=norm,
            shading='auto',
            edgecolors='none'
        )

    for name, t0, t1 in sectors:
        draw(fields[name], t0, t1, cmaps[name], norms[name])

    # =====================================================
    # Draw TRUE equal-width gaps (not lines)
    # =====================================================
    def draw_gap(t0, t1):
        theta = np.linspace(t0, t1, 200)
        TH, RR = np.meshgrid(theta, [R_outer, R_inner], indexing='ij')

        X = RR * np.cos(TH)
        Y = RR * np.sin(TH)

        ax.fill(
            np.concatenate([X[:, 0], X[::-1, 1]]),
            np.concatenate([Y[:, 0], Y[::-1, 1]]),
            color='white',
            zorder=10
        )

    for g0, g1 in gap_sectors:
        draw_gap(g0, g1)

    # =====================================================
    # Circle outlines
    # =====================================================
    theta = np.linspace(0, 2*np.pi, 500)
    ax.plot(R_outer*np.cos(theta), R_outer*np.sin(theta), color='black', lw=1)
    ax.plot(R_inner*np.cos(theta), R_inner*np.sin(theta), color='black', lw=1)

    ax.set_aspect('equal')
    ax.axis('off')

    plt.tight_layout()
    plt.show()


# =====================================================
# RUN
# =====================================================
plot_hive(
    "/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_320",
    SliceFraction=0.1,
    zmax=20.0,
    rotation_deg=-117
)

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
import cmasher as cmr
import h5py

Tcmb0 = 2.725


def plot_hive(rlc_dir, SliceFraction=0.2, zmax=20.0, rotation_deg=0):

    # ===================== Load =====================
    with h5py.File(f"{rlc_dir}/All.hdf5", "r") as f:
        N = f["Header"].attrs["NumPixels"]
        i = int(SliceFraction * N)

        T = f["Temperature"][i]
        xHII = f["HII_Fraction"][i]
        xHeII = f["HeII_Fraction"][i]

        z = f["Redshifts"][:]
        D = f["Distances"][:]

    imax = np.argmin(np.abs(z - zmax))

    z = z[imax:]
    D = D[imax:]

    T = T[:, imax:]
    xHII = xHII[:, imax:]
    xHeII = xHeII[:, imax:]

    zc = 0.5 * (z[:-1] + z[1:])
    nz = T.shape[1]

    zc = zc[:nz]
    D = D[:nz + 1]

    z2d = np.tile(zc, (N, 1))

    # ===================== Physics =====================
    xHI = 1 - xHII
    xHeIII = 1 - xHeII

    delta = xHI / np.mean(xHI[xHI > 0])

    Tcmb = Tcmb0 * (1 + z2d)
    Ts = np.maximum(T, Tcmb + 1e-5)

    coupling = 1 - Tcmb / Ts

    Tb21 = 27 * xHI * delta * np.sqrt((1 + z2d) / 10) * coupling
    TbHe3 = 0.03 * xHeII * delta * np.sqrt((1 + z2d) / 10) * coupling

    fields = {
        "21cm": np.abs(Tb21),
        "3He": np.abs(TbHe3),
        "HII": xHII,
        "HeIII": xHeIII
    }

    cmaps = {
        "21cm": cmr.fusion,
        "3He": "coolwarm",
        "HII": cmr.amber,
        "HeIII": "BuPu"
    }

    norms = {
        "21cm": LogNorm(vmin=1e-2, vmax=np.percentile(fields["21cm"], 99)),
        "3He": LogNorm(vmin=1e-5, vmax=np.percentile(fields["3He"], 99)),
        "HII": Normalize(0, 1),
        "HeIII": Normalize(0, 1)
    }

    # ===================== Geometry =====================
    deg = np.pi / 180
    rot = rotation_deg * deg

    base = 270 * deg + rot
    w = 2 * np.pi / 3

    t0_A, t1_A = base, base + w
    t0_B, t1_B = t1_A, t1_A + w
    t0_C, t1_C = t1_B, t1_B + w

    mid_C = 0.5 * (t0_C + t1_C)

    sectors = [
        ("3He", t0_A, t1_A),
        ("21cm", t0_B, t1_B),
        ("HeIII", t0_C, mid_C),
        ("HII", mid_C, t1_C)
    ]

    # ===================== Radial =====================
    R_outer = D[0]
    R_inner = 0.1 * R_outer
    Rvals = np.linspace(R_outer, R_inner, len(zc) + 1)

    fig, ax = plt.subplots(figsize=(8, 8))

    # ===================== Draw sectors =====================
    def draw(data, t0, t1, cmap, norm):
        theta = np.linspace(t0, t1, N + 1)
        TH, RR = np.meshgrid(theta, Rvals, indexing='ij')

        X = RR * np.cos(TH)
        Y = RR * np.sin(TH)

        ax.pcolormesh(X, Y, data, cmap=cmap, norm=norm, shading='auto')

    for name, t0, t1 in sectors:
        draw(fields[name], t0, t1, cmaps[name], norms[name])

    # ===================== Draw constant-width gaps =====================
    def draw_gap(angle, width):
        dx, dy = np.cos(angle), np.sin(angle)
        px, py = -dy, dx

        L = R_outer * 2

        corners = [
            (+px*width/2 - dx*L, +py*width/2 - dy*L),
            (-px*width/2 - dx*L, -py*width/2 - dy*L),
            (-px*width/2 + dx*L, -py*width/2 + dy*L),
            (+px*width/2 + dx*L, +py*width/2 + dy*L),
        ]

        x = [c[0] for c in corners]
        y = [c[1] for c in corners]

        ax.fill(x, y, color='white', zorder=10)

    gap_width = 0.06 * R_outer

    gap_angles = [t1_A, t1_B, t1_C]

    for ang in gap_angles:
        draw_gap(ang, gap_width)

    # ===================== Circles =====================
    theta = np.linspace(0, 2*np.pi, 500)
    ax.plot(R_outer*np.cos(theta), R_outer*np.sin(theta), color='black', lw=1)
    ax.plot(R_inner*np.cos(theta), R_inner*np.sin(theta), color='black', lw=1)

    ax.set_aspect('equal')
    ax.axis('off')

    plt.tight_layout()
    plt.show()


# ===================== RUN =====================
plot_hive(
    "/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_320",
    SliceFraction=0.1,
    zmax=20.0,
    rotation_deg=-117
)

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
import cmasher as cmr
import h5py

Tcmb0 = 2.725


def plot_hive(rlc_dir, SliceFraction=0.2, zmax=20.0, rotation_deg=0, gap_width_frac=0.008):
    # =====================================================
    # Load data
    # =====================================================
    with h5py.File(f"{rlc_dir}/All.hdf5", "r") as f:
        N = f["Header"].attrs["NumPixels"]
        i = int(SliceFraction * N)
        T = f["Temperature"][i]
        xHII = f["HII_Fraction"][i]
        xHeII = f["HeII_Fraction"][i]
        z = f["Redshifts"][:]
        D = f["Distances"][:]

    # =====================================================
    # Restrict redshift
    # =====================================================
    imax = np.argmin(np.abs(z - zmax))
    z, D = z[imax:], D[imax:]
    T, xHII, xHeII = T[:, imax:], xHII[:, imax:], xHeII[:, imax:]

    zc = 0.5 * (z[:-1] + z[1:])
    nz = T.shape[1]
    zc = zc[:nz]
    D = D[:nz + 1]

    z2d = np.tile(zc, (N, 1))

    # =====================================================
    # Physics
    # =====================================================
    xHI = 1 - xHII
    xHeIII = 1 - xHeII
    delta = xHI / np.mean(xHI[xHI > 0])

    Tcmb = Tcmb0 * (1 + z2d)
    Ts = np.maximum(T, Tcmb + 1e-5)
    coupling = 1 - Tcmb / Ts

    Tb21  = 27   * xHI   * delta * np.sqrt((1 + z2d) / 10) * coupling
    TbHe3 = 0.03 * xHeII * delta * np.sqrt((1 + z2d) / 10) * coupling

    fields = {
        "21cm":  np.abs(Tb21),
        "3He":   np.abs(TbHe3),
        "HII":   xHII,
        "HeIII": xHeIII,
    }

    cmaps = {
        "21cm":  cmr.fusion,
        "3He":   "coolwarm",
        "HII":   cmr.amber,
        "HeIII": "BuPu",
    }

    norms = {
        "21cm":  LogNorm(vmin=1e-2, vmax=np.percentile(fields["21cm"], 99)),
        "3He":   LogNorm(vmin=1e-5, vmax=np.percentile(fields["3He"],  99)),
        "HII":   Normalize(0, 1),
        "HeIII": Normalize(0, 1),
    }

    # =====================================================
    # Geometry
    # =====================================================
    deg = np.pi / 180
    rot = rotation_deg * deg

    R_outer = D[0]
    R_inner = 0.04 * R_outer

    # Gap is a fixed physical width, converted to an angle at R_outer
    # so the gap visually appears as a straight band of constant width
    gap_phys  = gap_width_frac * R_outer          # physical half-gap
    gap_angle = np.arcsin(gap_phys / R_outer)     # angle subtended at outer edge

    total_gap   = 3 * 2 * gap_angle               # 3 gaps, each ±gap_angle
    total_arc   = 2 * np.pi - total_gap
    sector_width = total_arc / 3                  # each of the 3 main sectors

    # Build sector boundaries with fixed-width angular gaps
    theta_start = 270 * deg + rot
    sectors_raw = []     # (name, t0, t1) before splitting HeIII/HII
    gap_centers = []

    theta = theta_start
    names_order = ["3He", "21cm", "combo"]
    for name in names_order:
        t0 = theta + gap_angle
        t1 = t0 + sector_width
        sectors_raw.append((name, t0, t1))
        gap_centers.append(theta + gap_angle / 2)
        theta = t1 + gap_angle

    # Split "combo" into HeIII and HII
    _, t0_C, t1_C = sectors_raw[2]
    mid_C = 0.5 * (t0_C + t1_C)

    sectors = [
        ("3He",   sectors_raw[0][1], sectors_raw[0][2]),
        ("21cm",  sectors_raw[1][1], sectors_raw[1][2]),
        ("HeIII", t0_C,  mid_C),
        ("HII",   mid_C, t1_C),
    ]

    # Gap angle boundaries (the "forbidden" angular zones)
    gap_zones = []
    theta = theta_start
    for _ in range(3):
        gap_zones.append((theta, theta + 2 * gap_angle))
        theta += 2 * gap_angle + sector_width

    # =====================================================
    # Radial grid
    # =====================================================
    Rvals = np.linspace(R_outer, R_inner, len(zc) + 1)

    # =====================================================
    # Plot
    # =====================================================
    fig, ax = plt.subplots(figsize=(10, 10), facecolor='white')

    def draw_sector(data, t0, t1, cmap, norm):
        theta_arr = np.linspace(t0, t1, N + 1)
        TH, RR = np.meshgrid(theta_arr, Rvals, indexing='ij')
        ax.pcolormesh(
            RR * np.cos(TH), RR * np.sin(TH), data,
            cmap=cmap, norm=norm,
            shading='auto', edgecolors='none', rasterized=True,
        )

    for name, t0, t1 in sectors:
        draw_sector(fields[name], t0, t1, cmaps[name], norms[name])

    # =====================================================
    # Draw gaps as constant-physical-width straight bands
    # Each gap is centered on a radial line; we draw a white
    # rectangle of fixed width ±gap_phys perpendicular to that line.
    # =====================================================
    def draw_straight_gap(gap_t0, gap_t1):
        """
        Draw a white wedge whose angular edges are at gap_t0 and gap_t1,
        BUT clip it so the actual painted region has constant physical width
        equal to gap_phys on both sides of the gap centre.
        We achieve this by drawing a white rotated rectangle (parallelogram)
        that has fixed half-width gap_phys, oriented along the gap bisector.
        """
        t_mid = 0.5 * (gap_t0 + gap_t1)

        # Direction along the gap (radial), and perpendicular
        r_hat = np.array([np.cos(t_mid), np.sin(t_mid)])
        p_hat = np.array([-np.sin(t_mid), np.cos(t_mid)])   # perpendicular

        # Four corners of the constant-width rectangle
        corners = np.array([
            R_inner * r_hat + gap_phys * p_hat,
            R_outer * r_hat + gap_phys * p_hat,
            R_outer * r_hat - gap_phys * p_hat,
            R_inner * r_hat - gap_phys * p_hat,
        ])

        ax.fill(corners[:, 0], corners[:, 1],
                color='white', zorder=10, linewidth=0)

    for g0, g1 in gap_zones:
        draw_straight_gap(g0, g1)

    # =====================================================
    # Circle outlines
    # =====================================================
    #theta_arr = np.linspace(0, 2 * np.pi, 1000)
    #ax.plot(R_outer * np.cos(theta_arr), R_outer * np.sin(theta_arr), color='black', lw=1.2)
    #ax.plot(R_inner * np.cos(theta_arr), R_inner * np.sin(theta_arr), color='black', lw=1.2)

    ax.set_aspect('equal')
    ax.axis('off')
    plt.tight_layout()
    plt.show()


# =====================================================
# RUN
# =====================================================
plot_hive(
    "/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_320",
    SliceFraction=0.1,
    zmax=20.0,
    rotation_deg=-120,
    gap_width_frac=0.04,   # tweak this to taste
)

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
import cmasher as cmr
import h5py
from matplotlib.cm import ScalarMappable

Tcmb0 = 2.725


def plot_hive(rlc_dir, SliceFraction=0.2, zmax=20.0, rotation_deg=0,
              node_size=8, alpha=0.85):

    # =====================================================
    # Load data
    # =====================================================
    with h5py.File(f"{rlc_dir}/All.hdf5", "r") as f:
        N = f["Header"].attrs["NumPixels"]
        i = int(SliceFraction * N)
        T      = f["Temperature"][i]
        xHII   = f["HII_Fraction"][i]
        xHeII  = f["HeII_Fraction"][i]
        z      = f["Redshifts"][:]
        D      = f["Distances"][:]

    # =====================================================
    # Restrict redshift
    # =====================================================
    imax = np.argmin(np.abs(z - zmax))
    z, D = z[imax:], D[imax:]
    T, xHII, xHeII = T[:, imax:], xHII[:, imax:], xHeII[:, imax:]

    zc  = 0.5 * (z[:-1] + z[1:])
    nz  = T.shape[1]
    zc  = zc[:nz]

    z2d = np.tile(zc, (N, 1))

    # =====================================================
    # Physics
    # =====================================================
    xHI    = 1.0 - xHII
    xHeIII = 1.0 - xHeII
    delta  = xHI / np.mean(xHI[xHI > 0])

    Tcmb = Tcmb0 * (1 + z2d)
    Ts   = np.maximum(T, Tcmb + 1e-5)
    coupling = 1.0 - Tcmb / Ts

    Tb21  = 27.0  * xHI   * delta * np.sqrt((1 + z2d) / 10) * coupling
    TbHe3 = 0.03  * xHeII * delta * np.sqrt((1 + z2d) / 10) * coupling

    # =====================================================
    # Hive geometry: 3 axes at 120° apart
    # =====================================================
    deg = np.pi / 180
    rot = rotation_deg * deg

    # Axis directions (unit vectors)
    axis_angles = np.array([90, 210, 330]) * deg + rot   # top, lower-left, lower-right
    axis_dirs   = np.stack([np.cos(axis_angles),
                            np.sin(axis_angles)], axis=1)  # (3, 2)

    # Radial mapping: redshift → radius
    z_min, z_max_data = zc.min(), zc.max()
    R_inner = 0.15
    R_outer = 1.0

    def z_to_r(z_arr):
        """Map redshift to radius (high-z = outer, low-z = inner)."""
        return R_inner + (R_outer - R_inner) * (z_arr - z_min) / (z_max_data - z_min)

    r_vals = z_to_r(zc)   # shape (nz,)

    # =====================================================
    # Colormaps / norms (one per axis / field)
    # =====================================================
    axes_config = [
        # (label,    field_2d,        cmap,         norm)
        ("21cm Tb",  np.abs(Tb21),    cmr.fusion,
         LogNorm(vmin=1e-2, vmax=np.percentile(np.abs(Tb21),  99))),
        ("He3 Tb",   np.abs(TbHe3),   "coolwarm",
         LogNorm(vmin=1e-5, vmax=np.percentile(np.abs(TbHe3), 99))),
        ("HII frac", xHII,            cmr.amber,
         Normalize(0, 1)),
    ]

    # =====================================================
    # Plot
    # =====================================================
    fig, ax = plt.subplots(figsize=(9, 9), facecolor='white')

    # -- Draw axes (spokes) --
    ax_line_kw = dict(color='#cccccc', lw=1.5, zorder=1)
    for d in axis_dirs:
        ax.plot([0, R_outer * d[0]], [0, R_outer * d[1]], **ax_line_kw)

    # -- Draw redshift rings (thin guidelines) --
    ring_zvals = np.linspace(z_min, z_max_data, 5)
    theta_ring = np.linspace(0, 2 * np.pi, 300)
    for rz in ring_zvals:
        rr = z_to_r(rz)
        ax.plot(rr * np.cos(theta_ring), rr * np.sin(theta_ring),
                color='#e0e0e0', lw=0.5, zorder=0)

    # -- Draw nodes per axis --
    # For each axis we take a random subset of pixels to keep the plot readable
    rng     = np.random.default_rng(42)
    n_show  = min(N, 300)          # number of pixel lines to show per axis
    px_idx  = rng.choice(N, n_show, replace=False)

    for axis_idx, (label, field_2d, cmap, norm) in enumerate(axes_config):
        d = axis_dirs[axis_idx]

        # field_2d shape: (N, nz); take subset of pixels
        vals = field_2d[px_idx, :]         # (n_show, nz)

        # For each pixel, scatter nodes along this axis
        for pix in range(n_show):
            v = vals[pix]                  # (nz,) brightness temperature
            x = r_vals * d[0]
            y = r_vals * d[1]

            sm = ScalarMappable(norm=norm, cmap=cmap)
            colors = sm.to_rgba(v)

            ax.scatter(x, y, c=colors, s=node_size,
                       alpha=alpha, linewidths=0, zorder=2)

    # -- Center hub --
    ax.scatter([0], [0], s=60, color='#888888', zorder=5, linewidths=0)

    # -- Axis labels --
    label_offset = 1.12
    axis_labels = ["21cm", "³He", "HII"]
    for label, d in zip(axis_labels, axis_dirs):
        ax.text(label_offset * d[0], label_offset * d[1], label,
                ha='center', va='center',
                fontsize=13, fontweight='bold', color='#333333')

    # -- Redshift tick labels on first axis --
    d0 = axis_dirs[0]
    for rz in ring_zvals:
        rr = z_to_r(rz)
        ax.text(rr * d0[0] + 0.04, rr * d0[1] + 0.04,
                f"z={rz:.1f}", fontsize=7, color='#999999', ha='left')

    # -- Colorbars (compact, one per axis) --
    cbar_axes = [
        fig.add_axes([0.13, 0.07, 0.22, 0.018]),
        fig.add_axes([0.39, 0.07, 0.22, 0.018]),
        fig.add_axes([0.65, 0.07, 0.22, 0.018]),
    ]
    cbar_labels = ["21cm $T_b$ (mK)", "³He $T_b$ (mK)", "HII fraction"]
    for (label, field_2d, cmap, norm), cax, clabel in zip(axes_config, cbar_axes, cbar_labels):
        sm = ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cb = fig.colorbar(sm, cax=cax, orientation='horizontal')
        cb.set_label(clabel, fontsize=8, color='#555555')
        cb.ax.tick_params(labelsize=7, colors='#555555')

    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-1.3, 1.3)
    ax.set_aspect('equal')
    ax.axis('off')

    plt.tight_layout(rect=[0, 0.1, 1, 1])
    plt.savefig("hive_plot.png", dpi=150, bbox_inches='tight',
                facecolor='white')
    plt.show()


# =====================================================
# RUN
# =====================================================
plot_hive(
    "/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_320",
    SliceFraction=0.1,
    zmax=20.0,
    rotation_deg=0,
    node_size=6,
    alpha=0.7,
)

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.cm import ScalarMappable
import cmasher as cmr
import h5py

Tcmb0 = 2.725


def plot_hive(rlc_dir, SliceFraction=0.2, zmax=20.0, rotation_deg=0):

    # =====================================================
    # Load data
    # =====================================================
    with h5py.File(f"{rlc_dir}/All.hdf5", "r") as f:
        N     = f["Header"].attrs["NumPixels"]
        i     = int(SliceFraction * N)
        T     = f["Temperature"][i]
        xHII  = f["HII_Fraction"][i]
        xHeII = f["HeII_Fraction"][i]*12.666667
        z     = f["Redshifts"][:]
        D     = f["Distances"][:]

    # =====================================================
    # Restrict redshift
    # =====================================================
    imax = np.argmin(np.abs(z - zmax))
    z, D = z[imax:], D[imax:]
    T, xHII, xHeII = T[:, imax:], xHII[:, imax:], xHeII[:, imax:]

    zc  = 0.5 * (z[:-1] + z[1:])
    nz  = T.shape[1]
    zc  = zc[:nz]

    z2d = np.tile(zc, (N, 1))

    # =====================================================
    # Physics
    # =====================================================
    xHI    = 1.0 - xHII
    xHeIII = 1.0 - xHeII
    delta  = xHI / np.mean(xHI[xHI > 0])

    Tcmb     = Tcmb0 * (1 + z2d)
    Ts       = np.maximum(T, Tcmb + 1e-5)
    coupling = 1.0 - Tcmb / Ts

    Tb21  = 27.0 * xHI   * delta * np.sqrt((1 + z2d) / 10) * coupling
    TbHe3 = 0.03 * xHeII * delta * np.sqrt((1 + z2d) / 10) * coupling

    # =====================================================
    # Colormaps / norms
    # =====================================================
    norm_21    = LogNorm(vmin=1e-2, vmax=np.percentile(np.abs(Tb21),  99))
    norm_He3   = LogNorm(vmin=1e-5, vmax=np.percentile(np.abs(TbHe3), 99))
    norm_HII   = Normalize(0, 1)
    norm_HeIII = Normalize(0, 1)

    # =====================================================
    # Geometry: 3 axes at 120° apart
    # Sector 0 (120°): 21cm       — axis[0] to axis[1]
    # Sector 1 (120°): 3He        — axis[1] to axis[2]
    # Sector 2 ( 60°): HII        — axis[2] to midpoint
    # Sector 3 ( 60°): HeIII      — midpoint to axis[0]+2pi
    # =====================================================
    deg    = np.pi / 180
    rot    = rotation_deg * deg
    two_pi = 2 * np.pi

    axis_angles = np.array([90, 210, 330]) * deg + rot

    # Midpoint of the third 120° sector
    mid_ang = axis_angles[2] + (two_pi / 6)

    sectors = [
        {
            "data":        np.abs(Tb21),
            "cmap":        cmr.fusion,
            "norm":        norm_21,
            "t0":          axis_angles[0],
            "t1":          axis_angles[1],
            "label":       "21cm",
            "label_angle": axis_angles[0],
        },
        {
            "data":        np.abs(TbHe3),
            "cmap":        "coolwarm",
            "norm":        norm_He3,
            "t0":          axis_angles[1],
            "t1":          axis_angles[2],
            "label":       "$^3$He",
            "label_angle": axis_angles[1],
        },
        {
            "data":        xHII,
            "cmap":        cmr.amber,
            "norm":        norm_HII,
            "t0":          axis_angles[2],
            "t1":          mid_ang,
            "label":       "HI",
            "label_angle": axis_angles[2] + (two_pi / 12),
        },
        {
            "data":        xHeIII,
            "cmap":        "magma",
            "norm":        norm_HeIII,
            "t0":          mid_ang,
            "t1":          axis_angles[0] + two_pi,
            "label":       "HeIII",
            "label_angle": mid_ang + (two_pi / 12),
        },
    ]

    R_outer = 1.0
    r_edges = np.linspace(0, R_outer, nz + 1)

    # =====================================================
    # Plot
    # =====================================================
    fig, ax = plt.subplots(figsize=(9, 9), facecolor='white')
    ax.set_aspect('equal')
    ax.axis('off')

    # -- Sectors --
    for sec in sectors:
        data  = sec["data"]     # (N, nz)
        t0    = sec["t0"]
        t1    = sec["t1"]

        # Use ALL N pixels — no subsampling
        theta = np.linspace(t0, t1, N + 1)      # (N+1,)
        r     = r_edges                          # (nz+1,)

        TH, RR = np.meshgrid(theta, r, indexing='ij')
        X = RR * np.cos(TH)
        Y = RR * np.sin(TH)

        C = data[:, ::-1]                        # (N, nz), flip: high-z outer

        ax.pcolormesh(X, Y, C,
                      cmap=sec["cmap"],
                      norm=sec["norm"],
                      shading='auto',
                      rasterized=True)

    # -- Main spoke gaps (thick white lines) --
    for ang in axis_angles:
        ax.plot([0, R_outer * np.cos(ang)],
                [0, R_outer * np.sin(ang)],
                color='white', lw=6, zorder=5,
                solid_capstyle='butt')

    # -- Thin divider between HII and HeIII --
    #ax.plot([0, R_outer * np.cos(mid_ang)],
    #        [0, R_outer * np.sin(mid_ang)],
    #        color='white', lw=2, zorder=5,
    #        solid_capstyle='butt')

    # -- Outer circle --
    th = np.linspace(0, two_pi, 1000)
    ax.plot(R_outer * np.cos(th), R_outer * np.sin(th),
            color='black', lw=1.2, zorder=6)

    # -- Axis / sector labels --
    label_r = R_outer + 0.10
    for sec in sectors:
        ang = sec["label_angle"]
        ax.text(label_r * np.cos(ang), label_r * np.sin(ang),
                sec["label"],
                ha='center', va='center',
                fontsize=13, fontweight='bold', color='#222222')

    # -- Redshift ticks on each main spoke --
    z_min, z_max_d = zc.min(), zc.max()

    def z_to_r(zval):
        return (zval - z_min) / (z_max_d - z_min) * R_outer

    tick_zvals = np.linspace(z_min, z_max_d, 5)

    for ang in axis_angles:
        for tz in tick_zvals:
            tr       = z_to_r(tz)
            tick_len = 0.025
            perp     = ang + np.pi / 2
            tx, ty   = tr * np.cos(ang), tr * np.sin(ang)
            ax.plot([tx - tick_len * np.cos(perp),
                     tx + tick_len * np.cos(perp)],
                    [ty - tick_len * np.sin(perp),
                     ty + tick_len * np.sin(perp)],
                    color='white', lw=1.5, zorder=7)

    # z labels on first axis only
    ang0 = axis_angles[0]
    for tz in tick_zvals:
        tr = z_to_r(tz)
        ax.text(tr * np.cos(ang0) + 0.04,
                tr * np.sin(ang0) + 0.02,
                f"z={tz:.1f}", fontsize=7, color='#666666', zorder=8)

    # =====================================================
    # Colorbars (4 across the bottom)
    # =====================================================
    cbar_specs = [
        (cmr.fusion,  norm_21,    "21cm $T_b$ (mK)"),
        ("coolwarm",  norm_He3,   "$^3$He $T_b$ (mK)"),
        (cmr.amber,   norm_HII,   "HII fraction"),
        ("magma",      norm_HeIII, "HeIII fraction"),
    ]
    for k, (cmap, norm, label) in enumerate(cbar_specs):
        cax = fig.add_axes([0.05 + k * 0.23, 0.06, 0.18, 0.018])
        sm  = ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cb  = fig.colorbar(sm, cax=cax, orientation='horizontal')
        cb.set_label(label, fontsize=8, color='#555555')
        cb.ax.tick_params(labelsize=7, colors='#555555')

    plt.tight_layout(rect=[0, 0.1, 1, 1])
    plt.savefig("hive_plot.png", dpi=150, bbox_inches='tight',
                facecolor='white')
    plt.show()


# =====================================================
# RUN
# =====================================================
plot_hive(
    "/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_640",
    SliceFraction=0.1,
    zmax=20.0,
    rotation_deg=120,
)

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.cm import ScalarMappable
import cmasher as cmr
import h5py

Tcmb0 = 2.725


def plot_hive(rlc_dir, SliceFraction=0.2, zmax=20.0, rotation_deg=0):

    # =====================================================
    # Load data
    # =====================================================
    with h5py.File(f"{rlc_dir}/All.hdf5", "r") as f:
        N     = f["Header"].attrs["NumPixels"]
        i     = int(SliceFraction * N)
        T     = f["Temperature"][i]
        xHII  = f["HII_Fraction"][i]
        xHeII = f["HeII_Fraction"][i] * 12.666667
        z     = f["Redshifts"][:]
        D     = f["Distances"][:]

    # =====================================================
    # Restrict redshift
    # =====================================================
    imax = np.argmin(np.abs(z - zmax))
    z, D = z[imax:], D[imax:]
    T, xHII, xHeII = T[:, imax:], xHII[:, imax:], xHeII[:, imax:]

    zc = 0.5 * (z[:-1] + z[1:])
    nz = T.shape[1]
    zc = zc[:nz]

    z2d = np.tile(zc, (N, 1))

    # =====================================================
    # Physics
    # =====================================================
    xHI    = 1.0 - xHII
    xHeIII = 1.0 - xHeII
    delta  = xHI / np.mean(xHI[xHI > 0])

    Tcmb     = Tcmb0 * (1 + z2d)
    Ts       = np.maximum(T, Tcmb + 1e-5)
    coupling = 1.0 - Tcmb / Ts

    Tb21  = 27.0 * xHI   * delta * np.sqrt((1 + z2d) / 10) * coupling
    TbHe3 = 0.03 * xHeII * delta * np.sqrt((1 + z2d) / 10) * coupling

    # =====================================================
    # Colormaps / norms
    # =====================================================
    norm_21    = LogNorm(vmin=1e-2, vmax=np.percentile(np.abs(Tb21),  99))
    norm_He3   = LogNorm(vmin=1e-5, vmax=np.percentile(np.abs(TbHe3), 99))
    norm_HII   = Normalize(0, 1)
    norm_HeIII = Normalize(0, 1)

    # =====================================================
    # Geometry: 4 equal 90° sectors
    # Spokes at 45°, 135°, 225°, 315° (diagonals)
    # Upper-left  (90°–180°):  HII
    # Upper-right (0°–90°):    21cm
    # Lower-left  (180°–270°): HeIII
    # Lower-right (270°–360°): 3He
    # =====================================================
    deg    = np.pi / 180
    rot    = rotation_deg * deg
    two_pi = 2 * np.pi

    # Spokes on axes (0°, 90°, 180°, 270°) — aligned with sector boundaries
    axis_angles = np.array([0, 90, 180, 270]) * deg + rot

    sectors = [
        {
            "data":        np.abs(Tb21),
            "cmap":        cmr.fusion,
            "norm":        norm_21,
            "t0":          0   * deg + rot,
            "t1":          90  * deg + rot,
            "label":       "21cm $T_b$",
            "label_angle": 45  * deg + rot,
        },
        {
            "data":        xHII,
            "cmap":        cmr.amber,
            "norm":        norm_HII,
            "t0":          90  * deg + rot,
            "t1":          180 * deg + rot,
            "label":       "HII",
            "label_angle": 135 * deg + rot,
        },
        {
            "data":        xHeIII,
            "cmap":        "magma",
            "norm":        norm_HeIII,
            "t0":          180 * deg + rot,
            "t1":          270 * deg + rot,
            "label":       "HeIII",
            "label_angle": 225 * deg + rot,
        },
        {
            "data":        np.abs(TbHe3),
            "cmap":        "coolwarm",
            "norm":        norm_He3,
            "t0":          270 * deg + rot,
            "t1":          360 * deg + rot,
            "label":       "$^3$He $T_b$",
            "label_angle": 315 * deg + rot,
        },
    ]

    R_outer = 1.0
    r_edges = np.linspace(0, R_outer, nz + 1)

    # =====================================================
    # Plot
    # =====================================================
    fig, ax = plt.subplots(figsize=(9, 9), facecolor='white')
    ax.set_aspect('equal')
    ax.axis('off')

    ax.set_xlim(-1.25, 1.25)
    ax.set_ylim(-1.25, 1.25)

    # -- Sectors --
    for sec in sectors:
        data = sec["data"]
        t0   = sec["t0"]
        t1   = sec["t1"]

        theta = np.linspace(t0, t1, N + 1)          # (N+1,)

        TH, RR = np.meshgrid(theta, r_edges, indexing='ij')   # (N+1, nz+1)
        X = RR * np.cos(TH)
        Y = RR * np.sin(TH)

        C = data[:, ::-1]                            # flip: high-z outer

        ax.pcolormesh(X, Y, C,
                      cmap=sec["cmap"],
                      norm=sec["norm"],
                      shading='auto',
                      rasterized=True)

    # -- White spoke dividers --
    for ang in axis_angles:
        ax.plot([0, R_outer * np.cos(ang)],
                [0, R_outer * np.sin(ang)],
                color='white', lw=6, zorder=5,
                solid_capstyle='butt')

    # -- Outer circle --
    th = np.linspace(0, two_pi, 1000)
    ax.plot(R_outer * np.cos(th), R_outer * np.sin(th),
            color='black', lw=1.2, zorder=6)

    # -- Labels outside circle --
    label_r = R_outer + 0.12
    for sec in sectors:
        ang = sec["label_angle"]
        ax.text(label_r * np.cos(ang), label_r * np.sin(ang),
                sec["label"],
                ha='center', va='center',
                fontsize=13, fontweight='bold', color='#222222')

    # -- Redshift ticks on each spoke --
    z_min, z_max_d = zc.min(), zc.max()

    def z_to_r(zval):
        return (zval - z_min) / (z_max_d - z_min) * R_outer

    tick_zvals = np.linspace(z_min, z_max_d, 5)

    for ang in axis_angles:
        for tz in tick_zvals:
            tr       = z_to_r(tz)
            tick_len = 0.025
            perp     = ang + np.pi / 2
            tx, ty   = tr * np.cos(ang), tr * np.sin(ang)
            ax.plot([tx - tick_len * np.cos(perp),
                     tx + tick_len * np.cos(perp)],
                    [ty - tick_len * np.sin(perp),
                     ty + tick_len * np.sin(perp)],
                    color='white', lw=1.5, zorder=7)

    # z labels on right spoke (0°) only
    ang0 = 0 * deg + rot
    for tz in tick_zvals:
        tr = z_to_r(tz)
        ax.text(tr * np.cos(ang0) + 0.04,
                tr * np.sin(ang0) + 0.02,
                f"z={tz:.1f}", fontsize=7, color='#666666', zorder=8)

    # =====================================================
    # Colorbars
    # =====================================================
    cbar_specs = [
        (cmr.fusion, norm_21,    "21cm $T_b$ (mK)"),
        ("coolwarm", norm_He3,   "$^3$He $T_b$ (mK)"),
        (cmr.amber,  norm_HII,   "HII fraction"),
        ("magma",    norm_HeIII, "HeIII fraction"),
    ]
    for k, (cmap, norm, label) in enumerate(cbar_specs):
        cax = fig.add_axes([0.05 + k * 0.23, 0.02, 0.18, 0.018])
        sm  = ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cb  = fig.colorbar(sm, cax=cax, orientation='horizontal')
        cb.set_label(label, fontsize=8, color='#555555')
        cb.ax.tick_params(labelsize=7, colors='#555555')

    plt.tight_layout(rect=[0, 0.08, 1, 1])
    plt.savefig("hive_plot_circle4.png", dpi=300, bbox_inches='tight',
                facecolor='white')
    plt.show()


# =====================================================
# RUN
# =====================================================
plot_hive(
    "/orcd/data/mvogelsb/005/mkilab001/mvogelsblab003/Lab/Thesan-XL/Flagship_new/postprocessing/rlc_640",
    SliceFraction=0.1,
    zmax=20.0,
    rotation_deg=0,
)